## 1. Imports and Environment Setup

In [1]:
import os
import re
import time
import logging
import warnings
import numpy as np
import pandas as pd
import polars as pl
import matplotlib
import seaborn as sns
import scipy.stats as stats
import statsmodels.api as sm
import matplotlib.pyplot as plt

from pathlib import Path
from tqdm.auto import tqdm
from scipy.stats import chi2
from scipy.stats import shapiro
from datetime import datetime
from scipy.optimize import minimize
from scipy.stats import ttest_1samp
from joblib import Parallel, delayed
from linearmodels.panel import PanelOLS
from statsmodels.regression.linear_model import OLS
from statsmodels.stats.stattools import durbin_watson
from sklearn.preprocessing import PolynomialFeatures
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Environment settings
matplotlib.use("Agg")
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')


class _DummyResults:
    """Lightweight stub used by LP and DJ to expose a .params attribute."""
    pass

/Users/ivanzamatin/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## 2. Visualization Configuration

In [2]:
# ── Blue-gradient palette (dark → light) for sequential model comparisons ──
BLUE_PALETTE = [
    "#08306b",  # 0  navy        (darkest)
    "#08519c",  # 1  dark blue
    "#2171b5",  # 2  medium-dark
    "#4292c6",  # 3  medium
    "#6baed6",  # 4  medium-light
    "#9ecae1",  # 5  light blue
    "#c6dbef",  # 6  very light  (lightest)
]

# Accent colours derived from the same blue family
BLUE_PRIMARY   = "#08519c"   # main line / fill
BLUE_SECONDARY = "#6baed6"   # CI band / secondary element
BLUE_LIGHT     = "#c6dbef"   # background fill
ACCENT_DARK    = "#08306b"   # emphasis / reference line
ZERO_LINE_CLR  = "#4a4a4a"   # neutral grey for zero / reference

# ── Seaborn + Matplotlib global theme ──────────────────────────────────────
# Typography bumped for thesis print readability
sns.set_theme(
    style="ticks",
    context="paper",
    palette=BLUE_PALETTE,
    font_scale=1.35,                          # up from 1.15
    rc={
        # Figure
        "figure.figsize":      (10, 6),
        "figure.dpi":          150,
        # Font
        "font.family":         "serif",
        "font.serif":          ["Times New Roman", "DejaVu Serif"],
        "font.size":           14,            # base size bumped
        # Axes
        "axes.titlesize":      17,            # up from 13
        "axes.titleweight":    "bold",
        "axes.titlepad":       14,
        "axes.labelsize":      15,            # up from 11
        "axes.labelweight":    "semibold",
        "axes.labelpad":       8,
        "axes.linewidth":      1.0,
        "axes.edgecolor":      "#333333",
        "axes.spines.top":     False,
        "axes.spines.right":   False,
        "axes.grid":           True,
        "axes.grid.which":     "major",
        # Grid
        "grid.color":          "#e0e0e0",
        "grid.linewidth":      0.6,
        "grid.linestyle":      "--",
        # Ticks
        "xtick.labelsize":     13,            # up from 10
        "ytick.labelsize":     13,            # up from 10
        "xtick.direction":     "in",
        "ytick.direction":     "in",
        "xtick.major.width":   0.8,
        "ytick.major.width":   0.8,
        "xtick.major.size":    4.0,
        "ytick.major.size":    4.0,
        # Legend
        "legend.fontsize":     13,            # up from 10
        "legend.title_fontsize": 13,
        "legend.frameon":      False,
        "legend.borderpad":    0.5,
        # Patches
        "patch.edgecolor":     "none",
        # Savefig
        "savefig.dpi":         300,
        "savefig.bbox":        "tight",
    },
)

## 3. Data Download from HuggingFace

In [ ]:
# Download the data from HuggingFace
HF_TOKEN = ""


def download_step_by_step(output_csv="rfsd_final_panel.csv"):
    target_cols = [
        'inn', 'okved_section', 'oktmo',
        'line_2110', 'line_1150', 'line_2120', 'line_1120', 'line_4122',
        'outlier'
    ]

    storage_options = {"token": HF_TOKEN}
    years = range(2011, 2024)
    temp_files = []

    print("--- [Status] Starting sequential download by year ---")

    for year in years:
        temp_filename = f"rfsd_{year}.parquet"

        if os.path.exists(temp_filename):
            print(f"--- [Skip] Year {year} already downloaded. ---")
            temp_files.append(temp_filename)
            continue

        url = f"hf://datasets/irlspbru/RFSD/RFSD/year={year}/*.parquet"

        try:
            print(f"--- [Process] Downloading year {year}... ---")
            df_year = pl.read_parquet(url, storage_options=storage_options).select(target_cols)

            # Filter: all columns except line_1120 must be non-null
            cols_to_check = [c for c in target_cols if c != 'line_1120']

            df_year = (
                df_year
                .with_columns(pl.lit(year).alias("year"))
                .filter(pl.col('outlier') != True)
                .drop_nulls(subset=cols_to_check)
            )

            df_year.write_parquet(temp_filename)
            temp_files.append(temp_filename)
            time.sleep(5)

        except Exception as e:
            print(f"[Error] Failed to download year {year}: {e}")
            time.sleep(30)

    if temp_files:
        print("--- [Status] Merging all years into a single panel ---")
        full_panel = pl.scan_parquet(temp_files)

        final_df = (
            full_panel
            .sort(['inn', 'year'])
            .unique(subset=['inn', 'year'], keep='first')
            .collect()
        )

        final_df.write_csv(output_csv)
        print(f"--- [Success] Dataset ready! Total rows: {len(final_df)} ---")
    else:
        print("[Error] Failed to download any year.")


if __name__ == "__main__":
    download_step_by_step()

--- [Status] Starting sequential download by year ---
--- [Skip] Year 2011 already downloaded. ---
--- [Skip] Year 2012 already downloaded. ---
--- [Skip] Year 2013 already downloaded. ---
--- [Skip] Year 2014 already downloaded. ---
--- [Skip] Year 2015 already downloaded. ---
--- [Skip] Year 2016 already downloaded. ---
--- [Skip] Year 2017 already downloaded. ---
--- [Skip] Year 2018 already downloaded. ---
--- [Skip] Year 2019 already downloaded. ---
--- [Skip] Year 2020 already downloaded. ---
--- [Skip] Year 2021 already downloaded. ---
--- [Skip] Year 2022 already downloaded. ---
--- [Skip] Year 2023 already downloaded. ---
--- [Status] Merging all years into a single panel ---
--- [Success] Dataset ready! Total rows: 1533195 ---


## 4. Load Dataset

In [4]:
df = pd.read_csv('rfsd_final_panel.csv')

In [5]:
df.head()

,inn,okved_section,oktmo,line_2110,line_1150,line_2120,line_1120,line_4122,outlier,year
0,7842120005,F,4.091100e+10,7261.0,5317.0,-6031.0,NaN,-889.0,0.0,2019
1,3525174718,E,1.970100e+10,5654.0,5684.0,-4430.0,NaN,-3957.0,0.0,2014
2,5233000595,G,2.265215e+10,145885.0,14870.0,-109421.0,NaN,-16888.0,0.0,2020
3,7714378023,B,4.534800e+10,329304.0,1714.0,-285903.0,NaN,-8705.0,0.0,2017
4,8904029181,F,7.195600e+10,178.0,32171.0,-2821.0,NaN,-1036.0,0.0,2023


## 5. Column Renaming and OKVED Sector Mapping

In [ ]:
df['line_1120'] = df['line_1120'].fillna(0)

rename_dict = {
    'inn': 'inn',
    'oktmo': 'oktmo',
    'year': 'year',
    'okved_section': 'sector',
    'line_4122': 'CFo_labor',
    'line_2110': 'PL_revenue',
    'line_1150': 'B_fixed_assets',
    'line_2120': 'PL_cost_of_sales',
    'line_1120': 'B_research_development'
}

df = df.rename(columns=rename_dict)

okved_sections_mapping = {
'A': 'Agriculture, forestry, hunting, fishing and fish farming',
'B': 'Mining',
'C': 'Manufacturing',
'D': 'Provision of electric energy, gas and steam; air conditioning',
'E': 'Water supply; sanitation, waste collection and disposal, pollution control activities',
'F': 'Construction',
'G': 'Wholesale and retail trade; repair of motor vehicles and motorcycles',
'H': 'Transportation and storage',
'I': 'Activities of hotels and catering establishments',
'J': 'Information and communication activities',
'K': 'Financial and insurance activities',
'L': 'Real estate operations',
'M': 'Professional, scientific and technical activities',
'N': 'Administrative activities and related additional services',
'O': 'Public administration and military support security; social security',
'P': 'Education',
'Q': 'Activities in the field of health and social services',
'R': 'Activities in the field of culture, sports, leisure and entertainment',
'S': 'Provision of other types of services',
'T': 'Activities of households as employers; undifferentiated activities of private households for the production of goods and services for their own consumption',
'U': 'Activities of extraterritorial organizations and bodies' }

df['sector'] = df['sector'].map(okved_sections_mapping)

# ── Restrict to the 8 production sectors used in the cross-sectoral analysis ──
# (Manufacturing, Mining, Agriculture, Construction, Electric energy & gas,
#  Water supply & sanitation, Information & communication, Transportation.)
PRODUCTION_SECTORS = [
    'Manufacturing',
    'Mining',
    'Agriculture, forestry, hunting, fishing and fish farming',
    'Construction',
    'Provision of electric energy, gas and steam; air conditioning',
    'Water supply; sanitation, waste collection and disposal, pollution control activities',
    'Information and communication activities',
    'Transportation and storage',
]
n_before = len(df)
df = df[df['sector'].isin(PRODUCTION_SECTORS)].copy()
print(f"Restricted to {len(PRODUCTION_SECTORS)} production sectors: "
      f"{n_before:,} -> {len(df):,} rows ({len(df)/n_before:.1%} retained)")

df['CFo_labor'] = df['CFo_labor'] * -1
df['PL_cost_of_sales'] = df['PL_cost_of_sales'] * -1

## 6. CPI Deflation (Monthly & Annual)

In [7]:
# Annual CPI indices (December-to-December)
annual_cpi = {
    2011: 106.10, 2012: 106.57, 2013: 106.47, 2014: 111.35,
    2015: 112.91, 2016: 105.39, 2017: 102.51, 2018: 104.26,
    2019: 103.04, 2020: 104.91, 2021: 108.39, 2022: 111.94,
    2023: 107.42
}

# Cumulative price level relative to end of 2010
cumulative_price_level = {}
cumulative = 100.0  # Base: end of 2010 = 100

for year in sorted(annual_cpi.keys()):
    cumulative = cumulative * annual_cpi[year] / 100
    cumulative_price_level[year] = cumulative

print("Cumulative price level (end of year):")
print(cumulative_price_level)

# Monthly CPI chain indices for average-year deflators
monthly_cpi = {
    '2011': [102.37, 100.78, 100.62, 100.43, 100.48, 100.23, 99.99, 99.76, 99.96, 100.48, 100.42, 100.44],
    '2012': [100.50, 100.37, 100.58, 100.31, 100.52, 100.89, 101.23, 100.10, 100.55, 100.46, 100.34, 100.54],
    '2013': [100.97, 100.56, 100.34, 100.51, 100.66, 100.42, 100.82, 100.14, 100.21, 100.57, 100.56, 100.51],
    '2014': [100.59, 100.70, 101.02, 100.90, 100.90, 100.62, 100.49, 100.24, 100.65, 100.82, 101.28, 102.62],
    '2015': [103.85, 102.22, 101.21, 100.46, 100.35, 100.19, 100.80, 100.35, 100.57, 100.74, 100.75, 100.77],
    '2016': [100.96, 100.63, 100.46, 100.44, 100.41, 100.36, 100.54, 100.01, 100.17, 100.43, 100.44, 100.40],
    '2017': [100.62, 100.22, 100.13, 100.33, 100.37, 100.61, 100.07, 99.46, 99.85, 100.20, 100.22, 100.42],
    '2018': [100.31, 100.21, 100.29, 100.38, 100.38, 100.49, 100.27, 100.01, 100.16, 100.35, 100.50, 100.84],
    '2019': [101.01, 100.44, 100.32, 100.29, 100.34, 100.04, 100.20, 99.76, 99.84, 100.13, 100.28, 100.36],
    '2020': [100.40, 100.33, 100.55, 100.83, 100.27, 100.22, 100.35, 99.96, 99.93, 100.43, 100.71, 100.83],
    '2021': [100.67, 100.78, 100.66, 100.58, 100.74, 100.69, 100.31, 100.17, 100.60, 101.11, 100.96, 100.82],
    '2022': [100.99, 101.17, 107.61, 101.56, 100.12, 99.65, 99.61, 99.48, 100.05, 100.18, 100.37, 100.78],
    '2023': [100.84, 100.46, 100.37, 100.38, 100.31, 100.37, 100.63, 100.28, 100.87, 100.83, 101.11, 100.73]
}

# Compute average annual price level
df_monthly = pd.DataFrame(monthly_cpi)
df_monthly_long = df_monthly.stack().reset_index()
df_monthly_long.columns = ['Month', 'Year', 'CPI_Chain']
df_monthly_long['Year'] = df_monthly_long['Year'].astype(int)
df_monthly_long['Factor'] = df_monthly_long['CPI_Chain'] / 100
df_monthly_long['Price_Level'] = df_monthly_long['Factor'].cumprod()

avg_price_by_year = df_monthly_long.groupby('Year')['Price_Level'].mean()

# Base: average price level of 2011
avg_price_2011 = avg_price_by_year[2011]

# P&L deflator (average annual)
deflator_pnl_dict = (avg_price_by_year / avg_price_2011).to_dict()

# Balance sheet deflator (end of year)
deflator_bs_dict = {year: level / avg_price_2011 for year, level in cumulative_price_level.items()}

print(f"\nP&L deflators: {len(deflator_pnl_dict)} years")
print(f"Balance sheet deflators: {len(deflator_bs_dict)} years")
print(f"Example deflator_bs_dict: {deflator_bs_dict}")

# Apply deflators
cols_pnl_cf = ['PL_revenue', 'PL_cost_of_sales', 'CFo_labor']
cols_bs = ['B_fixed_assets', 'B_research_development']

df_real = df.copy()
df_real['year'] = df_real['year'].astype(int)

for col in cols_pnl_cf:
    if col in df_real.columns:
        df_real[col] = df_real[col] / df_real['year'].map(deflator_pnl_dict)

for col in cols_bs:
    if col in df_real.columns:
        df_real[col] = df_real[col] / df_real['year'].map(deflator_bs_dict)

print("\n=== DEFLATION COMPLETE ===")
print(df_real.head())

Cumulative price level (end of year):
{2011: 106.1, 2012: 113.07077, 2013: 120.386448819, 2014: 134.0503107599565, 2015: 151.35620587906686, 2016: 159.51430537594857, 2017: 163.5181144408849, 2018: 170.4839861160666, 2019: 175.66669929399504, 2020: 184.2919342293302, 2021: 199.75402751117102, 2022: 223.60465839600482, 2023: 240.1961240489884}

P&L deflators: 13 years
Balance sheet deflators: 13 years
Example deflator_bs_dict: {2011: 65.13531417079344, 2012: 69.41470431181457, 2013: 73.90583568078897, 2014: 82.29414803055852, 2015: 92.91832254130361, 2016: 97.92662012627987, 2017: 100.38457829144951, 2018: 104.66096132666527, 2019: 107.8426545509959, 2020: 113.1377288894498, 2021: 122.62998434327466, 2022: 137.27200447386164, 2023: 147.45758720582216}

=== DEFLATION COMPLETE ===
          inn                                             sector  \
0  7842120005                                       Construction   
1  3525174718  Water supply; sanitation, waste collection and...   
2  5233

## 7. EDA Helper Functions

In [8]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
EXPORT_DIR = Path(f"eda_export_{timestamp}")
EXPORT_DIR.mkdir(exist_ok=True)
print(f"Export directory: {EXPORT_DIR.resolve()}")

# Reverse map: LaTeX -> plain text
LATEX_UNESCAPE = {
    r"\&": "&", r"\%": "%", r"\$": "$", r"\#": "#",
    r"\_": "_", r"\{": "{", r"\}": "}",
    r"\textasciitilde{}": "~", r"\^{}": "^",
    r"\textbackslash{}": "\\",
    r"\\%": "%", r"\\&": "&",
}

_MATHTEXT_SPECIAL = re.compile(r"(?<!\\)([\$])")


def clean_label(text: str) -> str:
    """Remove LaTeX commands from strings when text.usetex=False."""
    if not isinstance(text, str):
        return text

    for latex, plain in LATEX_UNESCAPE.items():
        text = text.replace(latex, plain)

    text = re.sub(r"\\(?![a-zA-Z{])", "", text)
    text = _MATHTEXT_SPECIAL.sub(r"\\\1", text)

    return text


def _escape_axes(fig):
    """Clean labels on all axes and legends."""
    for ax in fig.get_axes():
        for getter, setter in [
            (ax.get_title, ax.set_title),
            (ax.get_xlabel, ax.set_xlabel),
            (ax.get_ylabel, ax.set_ylabel),
        ]:
            v = getter()
            if v:
                setter(clean_label(v))

        for tick in ax.get_xticklabels() + ax.get_yticklabels():
            t = tick.get_text()
            if t:
                tick.set_text(clean_label(t))

        leg = ax.get_legend()
        if leg:
            for t in leg.get_texts():
                t.set_text(clean_label(t.get_text()))

_fig_counter = 0


def save_all_figures(dpi: int = 150, fmt: str = "png"):
    global _fig_counter

    fig_nums = plt.get_fignums()
    if not fig_nums:
        print("No figures to save")
        return

    print(f"\nFigures found: {len(fig_nums)} | format: {fmt.upper()} | dpi: {dpi}\n")

    for num in fig_nums:
        _fig_counter += 1
        fig = plt.figure(num)

        for ax in fig.get_axes():
            for collection in ax.collections:
                collection.set_rasterized(True)
            for line in ax.lines:
                line.set_rasterized(True)

        _escape_axes(fig)

        path = EXPORT_DIR / f"figure_{_fig_counter:02d}.{fmt}"
        fig.savefig(path, bbox_inches="tight", dpi=dpi, format=fmt,
                    facecolor=fig.get_facecolor())
        plt.close(fig)
        print(f"  Saved: {path.name}")

    print(f"\nAll figures saved to: {EXPORT_DIR.resolve()}\n")

Export directory: /Users/ivanzamatin/Desktop/diploma/fixes_diplom/eda_export_20260417_192246


## 8. Descriptive Statistics (EDA)

In [9]:
def improved_eda(df, show_heatmap=True):
    print(f"Starting analysis. Sample size: {df.shape[0]} rows, {df.shape[1]} columns.")

    working_df = df.copy()

    target_cols = ['PL_revenue', 'B_fixed_assets', 'PL_cost_of_sales',
                   'B_research_development', 'CFo_labor']
    working_df['has_rd'] = (working_df['B_research_development'] > 0).astype(int)

    # --- Sector Analysis ---
    print("\nSector analysis:")

    top_sectors = working_df['sector'].value_counts().head(10).index
    sector_data = working_df[working_df['sector'].isin(top_sectors)]

    # Figure 1: R&D share by sector
    rd_intensity = (sector_data.groupby('sector')['has_rd']
                    .mean()
                    .sort_values(ascending=False))

    plt.figure(figsize=(10, 6))
    sns.barplot(x=rd_intensity.values, y=rd_intensity.index,
                hue=rd_intensity.index, palette="Blues_r", legend=False)
    plt.title('Share of companies with R&D by sector')
    plt.xlabel('Share of innovators (1.0 = 100%)')
    plt.ylabel(None)
    plt.tight_layout()
    plt.show()

    # Figure 2: Revenue distribution by sector
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=sector_data, x='PL_revenue', y='sector',
                hue='sector', palette="Blues_r", legend=False)
    plt.xscale('log')
    plt.title('Revenue Distribution by Sector (Log Scale)')
    plt.xlabel('Revenue (log10)')
    plt.ylabel(None)
    plt.tight_layout()
    plt.show()

    # --- Descriptive Statistics Table ---
    print("\nDescriptive statistics:")
    desc = working_df[target_cols].describe().T
    desc['skewness'] = working_df[target_cols].skew()
    print(desc.round(2))

    # --- Distribution Plots (one figure per variable) ---
    for i, col in enumerate(target_cols):
        log_data = np.log(working_df[col] + 1)

        plt.figure(figsize=(8, 5))
        sns.histplot(log_data, kde=True,
                     color=BLUE_PALETTE[i % len(BLUE_PALETTE)], bins=30)
        plt.title(f'Distribution of {col} (ln)')
        plt.xlabel('ln(Value + 1)')
        plt.ylabel('Frequency')
        plt.axvline(log_data.median(), color='black', ls='--',
                    alpha=0.7, label='Median')
        plt.legend()
        plt.tight_layout()
        plt.show()

    # --- R&D Analysis ---
    # Figure: R&D pie chart
    counts = working_df['has_rd'].value_counts()
    plt.figure(figsize=(7, 7))
    plt.pie(
        counts,
        labels=['Regular', 'Innovators'],
        autopct='%1.1f%%',
        startangle=140,
        colors=[BLUE_PALETTE[0], BLUE_PALETTE[2]],
        explode=(0, 0.1),
        textprops={'fontsize': 10}
    )
    plt.title('Share of companies with R&D investments')
    plt.tight_layout()
    plt.show()

    # Figure: Revenue comparison (Innovators vs Regular)
    plt.figure(figsize=(8, 5))
    sns.barplot(
        data=working_df,
        x='has_rd',
        y='PL_revenue',
        hue='has_rd',
        palette=[BLUE_PALETTE[0], BLUE_PALETTE[2]],
        legend=False
    )
    plt.yscale('log')
    plt.title('Revenue: Innovators vs Regular Firms (Log)')
    plt.xticks([0, 1], ['No R&D', 'Has R&D'])
    plt.xlabel(None)
    plt.ylabel('Revenue (log)')
    plt.tight_layout()
    plt.show()

    # --- Spearman Correlation ---
    corr_spearman = working_df[target_cols].corr(method='spearman')
    mask = np.triu(np.ones_like(corr_spearman, dtype=bool))

    plt.figure(figsize=(10, 8))
    sns.heatmap(
        corr_spearman,
        mask=mask,
        annot=True,
        fmt=".2f",
        cmap='RdBu_r',
        center=0,
        linewidths=1,
        cbar_kws={"shrink": .7}
    )
    plt.title('Spearman Correlation (robust to outliers)')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

    # --- Revenue Trends by Year ---
    plt.figure(figsize=(12, 6))
    sns.violinplot(
        data=working_df,
        x='year',
        y=np.log(working_df['PL_revenue'] + 1),
        hue='year',
        palette=BLUE_PALETTE,
        inner="quartile",
        legend=False
    )
    plt.title('Revenue Distribution Dynamics by Year')
    plt.ylabel('ln(Revenue + 1)')
    plt.xlabel(None)
    plt.tight_layout()
    plt.show()

    save_all_figures()
    return working_df


df_final = improved_eda(df_real)

Starting analysis. Sample size: 1533195 rows, 10 columns.

Sector analysis:

Descriptive statistics:
                            count        mean          std        min  \
PL_revenue              1533195.0  1199052.13  46038787.90 -706016.30   
B_fixed_assets          1533195.0     4519.57    274704.19   -2852.13   
PL_cost_of_sales        1533195.0   966090.78  43520289.54      -0.00   
B_research_development  1533195.0       12.74       942.81    -248.56   
CFo_labor               1533195.0    90908.28   1468800.91      -0.00   

                             25%       50%        75%           max  skewness  
PL_revenue              15412.87  80973.51  397225.40  3.781368e+10    518.53  
B_fixed_assets              6.28     63.79     469.39  1.039155e+08    249.81  
PL_cost_of_sales        10470.41  61073.01  315865.67  3.784850e+10    602.32  
B_research_development      0.00      0.00       0.00  5.576758e+05    337.30  
CFo_labor                1614.27   8548.54   37473.17  5.098

INFO: Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
INFO: Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
INFO: Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
INFO: Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.



Figures found: 11 | format: PNG | dpi: 150

  Saved: figure_01.png
  Saved: figure_02.png
  Saved: figure_03.png
  Saved: figure_04.png
  Saved: figure_05.png
  Saved: figure_06.png
  Saved: figure_07.png
  Saved: figure_08.png
  Saved: figure_09.png
  Saved: figure_10.png
  Saved: figure_11.png

All figures saved to: /Users/ivanzamatin/Desktop/diploma/fixes_diplom/eda_export_20260417_192246



### 8.1 Copy for Further Analysis

In [10]:
df2 = df_real.copy()

### 8.2 Lorenz Curve (Revenue Concentration)

In [11]:
# Lorenz Curve for Revenue Concentration
print("\nCalculating revenue concentration (Lorenz Curve):")

df2['is_near_zero'] = (df2['PL_revenue'] <= 100).astype(int)


def plot_lorenz_curve(data, column):
    s = data[column].dropna()
    values = np.sort(s[s >= 0])

    if len(values) == 0:
        return print("No data available")

    n = len(values)
    cum_dist = np.cumsum(values) / np.sum(values)
    index = np.linspace(0, 1, n)

    # Downsample for large datasets
    if n > 10000:
        step = n // 5000
        index, cum_dist = index[::step], cum_dist[::step]

    plt.plot(index, cum_dist, label='Lorenz curve', color=BLUE_PALETTE[0])
    plt.plot([0, 1], [0, 1], ls='--', color='gray', alpha=0.5, label='Perfect equality')
    plt.fill_between(index, index, cum_dist, alpha=0.1, color=BLUE_PALETTE[0])

    plt.title(f'Concentration: {column}')
    plt.xlabel('Share of companies')
    plt.ylabel('Share of revenue')
    plt.legend()


plot_lorenz_curve(df2, 'PL_revenue')
plt.show()

save_all_figures()

zeros_subset = df2[df2['is_near_zero'] == 1].copy()

print(f"Near-zero companies identified: {zeros_subset.shape[0]}")
print(f"This constitutes {zeros_subset.shape[0]/df2.shape[0]:.1%} of the total sample.")


Calculating revenue concentration (Lorenz Curve):

Figures found: 1 | format: PNG | dpi: 150

  Saved: figure_12.png

All figures saved to: /Users/ivanzamatin/Desktop/diploma/fixes_diplom/eda_export_20260417_192246

Near-zero companies identified: 40631
This constitutes 2.7% of the total sample.


### 8.3 Small Firms Analysis

In [12]:
# Analysis of Small Firms
# Group A: Negative revenue (rare, usually errors or returns)
df_negative_rev = df2[df2['PL_revenue'] < 0].copy()

# Group B: Low revenue (threshold in thousands)
small_threshold = 100
df_small_rev = df2[(df2['PL_revenue'] >= 0) & (df2['PL_revenue'] <= small_threshold)].copy()

# Group C: "Asset holders" (low revenue but significant fixed assets)
df_asset_holders = df_small_rev[df_small_rev['B_fixed_assets'] > 10].copy()

df_small_rnd = df_small_rev[df_small_rev['B_research_development'] > 0].copy()

print(f"--- Segmentation Results ---")
print(f"Total companies in sample: {len(df2)}")
print(f"1. Negative revenue: {len(df_negative_rev)}")
print(f"2. Low revenue (up to {small_threshold}k): {len(df_small_rev)}")
print(f"3. Of those with fixed assets (> 10k): {len(df_asset_holders)}")
print(f"4. Of those with R&D (> 0): {len(df_small_rnd)}")

if not df_asset_holders.empty:
    asset_by_sector = df_asset_holders['sector'].value_counts().head(10)
    sns.barplot(x=asset_by_sector.values, y=asset_by_sector.index,
                hue=asset_by_sector.index, palette="Blues_r")
    plt.title('Top 10 sectors with low revenue')
    plt.xlabel('Number of companies')
    plt.ylabel(None)
    plt.show()
    save_all_figures()
else:
    print("No companies with significant assets and low revenue found.")

if not df_small_rnd.empty:
    asset_by_sector = df_small_rnd['sector'].value_counts().head(10)
    sns.barplot(x=asset_by_sector.values, y=asset_by_sector.index,
                hue=asset_by_sector.index, palette="Blues_r")
    plt.title('Top 10 sectors: low revenue but positive R&D')
    plt.xlabel('Number of companies')
    plt.ylabel(None)
    plt.show()
    save_all_figures()

--- Segmentation Results ---
Total companies in sample: 1533195
1. Negative revenue: 129
2. Low revenue (up to 100k): 40502
3. Of those with fixed assets (> 10k): 9189
4. Of those with R&D (> 0): 257

Figures found: 1 | format: PNG | dpi: 150

  Saved: figure_13.png

All figures saved to: /Users/ivanzamatin/Desktop/diploma/fixes_diplom/eda_export_20260417_192246


Figures found: 1 | format: PNG | dpi: 150

  Saved: figure_14.png

All figures saved to: /Users/ivanzamatin/Desktop/diploma/fixes_diplom/eda_export_20260417_192246



## 9. Data Cleaning (Integrated Smart Filter)

In [13]:
# Thresholds
small_threshold = 100
assets_threshold = 50
labor_threshold = 5

bad_industries = [
    'Professional, scientific and technical activities',  # M
    'Financial and insurance activities',                 # K
    'Real estate operations'                              # L
]

# Filter logic
logic_1_original = (
    (df2['PL_revenue'] <= 0) |
    (df2['B_fixed_assets'] <= 0) |
    (df2['PL_cost_of_sales'] <= 0) |
    (df2['CFo_labor'] <= 0)
)
logic_2_zero_assets_and_revenue = (df2['B_fixed_assets'] <= 1) & (df2['PL_revenue'] <= small_threshold)
logic_3_dead_holdings = (df2['B_fixed_assets'] > 10000) & (df2['PL_revenue'] <= small_threshold) & (df2['CFo_labor'] <= labor_threshold)
logic_4_shell_firms = (df2['PL_revenue'] > small_threshold) & (df2['PL_cost_of_sales'] < 1) & (df2['CFo_labor'] < 1)
logic_5_fin_sector = (df2['sector'] == 'Financial and insurance activities') & (df2['CFo_labor'] < 10)
logic_6_re_sector = (df2['sector'] == 'Real estate operations') & (df2['B_fixed_assets'] < assets_threshold)
logic_7_consulting_shell = (df2['sector'] == 'Professional, scientific and technical activities') & \
                           (df2['PL_revenue'] > 5000) & (df2['CFo_labor'] < labor_threshold)
logic_9_ghosts = (df2['PL_revenue'] > 5000) & (df2['B_fixed_assets'] <= 1) & (df2['CFo_labor'] <= 1)

# Combine all filters
final_to_drop_logic = (
    logic_1_original | logic_2_zero_assets_and_revenue | logic_3_dead_holdings |
    logic_4_shell_firms | logic_5_fin_sector | logic_6_re_sector |
    logic_7_consulting_shell | logic_9_ghosts
)

# Apply filters
df_clean = df2[~final_to_drop_logic].copy()

rnd_industry = ['Professional, scientific and technical activities']
df_clean = df_clean[~df_clean['sector'].isin(rnd_industry)]

# Trimming (vectorized)
cols_to_clean = ['PL_revenue', 'B_fixed_assets', 'PL_cost_of_sales', 'CFo_labor']

keep_mask = pd.Series(True, index=df_clean.index)

for col in cols_to_clean:
    upper_limit = df_clean[col].quantile(0.995)
    lower_limit = df_clean[col].quantile(0.01)
    keep_mask = keep_mask & (df_clean[col] <= upper_limit) & (df_clean[col] >= lower_limit)

df_clean = df_clean[keep_mask]

print(f"Cleaning complete. Companies in sample: {len(df_clean)}")

Cleaning complete. Companies in sample: 1229456


### 9.1 EDA After Cleaning

In [14]:
# Reuse the same EDA function from Cell 8, but skip the heatmap
# (no missing values expected after cleaning)
df_final_clean = improved_eda(df_clean, show_heatmap=False)

Starting analysis. Sample size: 1229456 rows, 11 columns.

Sector analysis:

Descriptive statistics:
                            count       mean         std     min       25%  \
PL_revenue              1229456.0  638303.48  1817956.74  364.35  25370.83   
B_fixed_assets          1229456.0    1349.46     5294.01    0.09     13.82   
PL_cost_of_sales        1229456.0  520518.54  1504146.94  135.07  18438.29   
B_research_development  1229456.0       4.69      158.51 -248.56      0.00   
CFo_labor               1229456.0   58081.42   160609.53   51.29   2447.20   

                              50%        75%          max  skewness  
PL_revenue              111238.88  477859.63  31076596.02      7.12  
B_fixed_assets             100.95     591.88     94357.73      8.84  
PL_cost_of_sales         86027.01  383260.44  24657520.97      7.29  
B_research_development       0.00       0.00     44445.89    114.98  
CFo_labor                11056.83   42657.94   2340539.18      6.65  


INFO: Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
INFO: Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
INFO: Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
INFO: Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.



Figures found: 11 | format: PNG | dpi: 150

  Saved: figure_15.png
  Saved: figure_16.png
  Saved: figure_17.png
  Saved: figure_18.png
  Saved: figure_19.png
  Saved: figure_20.png
  Saved: figure_21.png
  Saved: figure_22.png
  Saved: figure_23.png
  Saved: figure_24.png
  Saved: figure_25.png

All figures saved to: /Users/ivanzamatin/Desktop/diploma/fixes_diplom/eda_export_20260417_192246



## 10. Data Preparation for Estimation

In [15]:
class DataPreparer:
    """
    Final preparation of the cleaned DataFrame for estimation.
    Log-transforms variables, computes R&D stock, and generates strict lags.
    """

    def __init__(self, df: pd.DataFrame):
        self.df = df.copy()

    def validate_and_clean(self):
        logging.info("Validating data for negative/zero values before log transformation...")
        positive_cols = ['PL_revenue', 'B_fixed_assets', 'PL_cost_of_sales', 'CFo_labor']
        initial_len = len(self.df)

        mask = pd.Series(True, index=self.df.index)
        for col in positive_cols:
            mask &= self.df[col] > 0
        mask &= self.df['B_research_development'] >= 0
        self.df = self.df[mask]

        dropped = initial_len - len(self.df)
        if dropped > 0:
            logging.info(f"Removed {dropped} rows with zeros/negatives in base factors.")
        return self

    def calc_rd_stock(self, depreciation_rate: float = 0.15):
        """Compute perpetual inventory R&D stock (vectorized loop)."""
        logging.info("Computing R&D stock...")
        self.df = self.df.sort_values(by=['inn', 'year'])

        rd = self.df['B_research_development'].values
        inn = self.df['inn'].values
        year = self.df['year'].values
        keep = 1.0 - depreciation_rate

        stock = np.empty_like(rd, dtype=float)
        stock[0] = rd[0]

        for t in range(1, len(rd)):
            if inn[t] == inn[t - 1] and year[t] - year[t - 1] == 1:
                stock[t] = stock[t - 1] * keep + rd[t]
            else:
                stock[t] = rd[t]

        self.df['rd_stock'] = stock
        return self

    def engineer_features(self):
        logging.info("Log-transforming variables (l_y, l_k, l_l, l_m)...")
        self.df['l_y'] = np.log(self.df['PL_revenue'])
        self.df['l_k'] = np.log(self.df['B_fixed_assets'])
        self.df['l_l'] = np.log(self.df['CFo_labor'])
        self.df['l_m'] = np.log(self.df['PL_cost_of_sales'])

        self.df['l_r_inv'] = np.log(self.df['B_research_development'] + 1)
        self.df['l_r'] = np.log(self.df['rd_stock'] + 1)
        self.df['is_innovator'] = (self.df['B_research_development'] > 0).astype(int)
        return self

    def create_lags(self):
        """Create strict time lags (t-1) for dynamic models."""
        logging.info("Generating time lags for dynamic models...")
        self.df = self.df.sort_values(['inn', 'year'])
        self.df['year_diff'] = self.df.groupby('inn')['year'].diff()

        # Investment
        self.df['investment'] = self.df.groupby('inn')['B_fixed_assets'].diff()
        self.df.loc[self.df['year_diff'] != 1, 'investment'] = np.nan

        with np.errstate(invalid='ignore', divide='ignore'):
            self.df['l_inv'] = np.log(np.where(self.df['investment'] > 0, self.df['investment'], np.nan))

        # Lags for all required variables
        cols_to_lag = ['l_y', 'l_k', 'l_l', 'l_m', 'l_r', 'l_r_inv', 'l_inv']
        non_consecutive = self.df['year_diff'] != 1
        for col in cols_to_lag:
            if col in self.df.columns:
                self.df[f'{col}_lag1'] = self.df.groupby('inn')[col].shift(1)
                self.df.loc[non_consecutive, f'{col}_lag1'] = np.nan

        return self

    def prepare(self) -> pd.DataFrame:
        self.validate_and_clean()\
            .calc_rd_stock()\
            .engineer_features()\
            .create_lags()

        required_cols = ['l_y', 'l_k', 'l_l', 'l_m']
        self.df = self.df.dropna(subset=required_cols).copy()

        logging.info(f"Final panel size ready for estimation: {self.df.shape}")
        return self.df

## 11. Diagnostics Reporter

In [16]:
class DiagnosticsReporter:
    """
    Reports and exports statistical diagnostics for production function models.

    Four reporting blocks:
      1. OLS-specific tests (VIF, BP, DW, Shapiro-Wilk)
      2. Returns to Scale (RTS) for all models
      3. TFP Persistence - AR(1) coefficient of omega
      4. Hansen J-test (overidentification) for GMM models (LP, DJ)
    """

    VAR_LABELS = {
        'l_l':     'Labor (ln L)',
        'l_m':     'Materials (ln M)',
        'l_k':     'Capital (ln K)',
        'l_r':     'R&D Stock (ln R)',
        'l_r_inv': 'R&D Flow (ln R_inv)',
    }

    def __init__(self, models_dict: dict, df_panel: pd.DataFrame):
        """
        :param models_dict: {'OLS': ols_model, 'OP': op_model, 'LP': lp_model, 'DJ': dj_model}
        :param df_panel:    df_model_ready - final panel dataset
        """
        self.models    = models_dict
        self.df        = df_panel.copy()
        self._sections = {}

    # ------------------------------------------------------------------
    # 1. OLS Diagnostics
    # ------------------------------------------------------------------
    def _report_ols_diagnostics(self):
        ols = self.models.get('OLS')
        if ols is None or not ols.diagnostics_:
            logging.warning("DiagnosticsReporter: OLS diagnostics not available.")
            return

        diag  = ols.diagnostics_
        lines = []

        print("\n" + "─" * 60)
        print("  OLS Model Diagnostics")
        print("─" * 60)

        if hasattr(ols, 'results'):
            r2 = getattr(ols.results, 'rsquared', np.nan)
            # Safe extraction of rsquared_adj (for PanelOLS use regular rsquared)
            r2_adj = getattr(ols.results, 'rsquared_adj', r2) 
            print(f"  R²          = {r2:.4f}")
            print(f"  Adj. R²     = {r2_adj:.4f}")

        vif_df = diag.get('VIF')
        if vif_df is not None:
            print("\n  Multicollinearity (VIF):")
            max_vif = vif_df['VIF'].max()
            verdict = "✓ No collinearity" if max_vif < 10 else "⚠ High collinearity detected"
            for _, row in vif_df.iterrows():
                label = self.VAR_LABELS.get(row['feature'], row['feature'])
                flag  = " ⚠" if row['VIF'] > 10 else ""
                print(f"    {label:<25} VIF = {row['VIF']:.2f}{flag}")
            print(f"  → {verdict} (max VIF = {max_vif:.2f})")
            lines.append(("Max VIF", f"{max_vif:.2f}"))

        bp = diag.get('Heteroscedasticity')
        if bp:
            lm   = bp['LM Statistic']
            p_bp = bp['LM-Test p-value']
            verdict = "⚠ Heteroscedastic" if p_bp < 0.05 else "✓ Homoscedastic"
            print(f"\n  Heteroscedasticity (Breusch-Pagan):")
            print(f"    LM = {lm:.2f},  p = {p_bp:.4f}  → {verdict}")
            print(f"    Note: HC3 robust standard errors applied in OLS.")
            lines.append(("BP LM-stat", f"{lm:.2f}"))
            lines.append(("BP p-value", f"{p_bp:.4f}"))

        dw = diag.get('Autocorrelation')
        if dw:
            dw_stat = dw['DW Statistic']
            interp  = dw['Interpretation']
            print(f"\n  Autocorrelation (Durbin-Watson):")
            print(f"    DW = {dw_stat:.3f}  → {interp} autocorrelation")
            lines.append(("Durbin-Watson", f"{dw_stat:.3f}"))

        sw = diag.get('Normality')
        if sw:
            w_stat = sw['W Statistic']
            p_sw   = sw['p-value']
            verdict = "Non-normal residuals" if p_sw < 0.05 else "Normal residuals"
            print(f"\n  Normality (Shapiro-Wilk, subsample n=5000):")
            print(f"    W = {w_stat:.4f},  p = {p_sw:.4f}  → {verdict}")
            print(f"    Note: Non-normality is expected at N~1.4M; "
                  "bootstrap SEs are asymptotically valid regardless.")
            lines.append(("Shapiro-Wilk W",      f"{w_stat:.4f}"))
            lines.append(("Shapiro-Wilk p-value", f"{p_sw:.4f}"))

        self._sections['OLS'] = lines

    # ------------------------------------------------------------------
    # 2. Returns to Scale (RTS)
    # ------------------------------------------------------------------
    def _report_rts(self):
        """Report Returns to Scale for all models."""
        print("\n" + "─" * 60)
        print("  Returns to Scale (sum of input elasticities)")
        print("─" * 60)

        pf_vars  = ['l_l', 'l_m', 'l_k']
        rts_rows = []

        for name, model in self.models.items():
            coefs = {k: v for k, v in model.coef_.items() if k in pf_vars}
            if not coefs:
                continue

            rts  = sum(coefs.values())
            kind = ("CRS" if abs(rts - 1) < 0.03
                    else "IRS" if rts > 1 else "DRS")
            parts = " + ".join(
                f"{self.VAR_LABELS.get(k, k).split('(')[0].strip()}={v:.4f}"
                for k, v in coefs.items()
            )

            # For LP: beta_m not identified
            if name == 'LP' and 'l_m' not in coefs:
                note = "  <- beta_m not identified in LP (materials serve as proxy)"
            else:
                note = ""

            print(f"  {name:<5}  RTS = {rts:.4f}  ({kind})    [{parts}]{note}")
            rts_rows.append((name, f"{rts:.4f}", kind,
                             "excl. beta_m" if name == 'LP' else ""))

        self._sections['RTS'] = rts_rows

    # ------------------------------------------------------------------
    # 3. TFP Persistence - AR(1)
    # ------------------------------------------------------------------
    def _report_tfp_persistence(self):
        """Report TFP persistence with AR(1) coefficient."""
        print("\n" + "─" * 60)
        print("  TFP Persistence — AR(1) of omega")
        print("─" * 60)
        print("  (Interpretation: rho > 0.7 = high persistence; rho > 1 = non-stationary)")

        omega_map = {
            'OLS': 'omega_ols',
            'OP':  'omega_op',
            'LP':  'omega_lp',
            'DJ':  'omega_dj',
        }

        persist_rows = []
        for name, model in self.models.items():
            if model.omega is None:
                continue

            omega_col = omega_map.get(name)
            if omega_col is None or omega_col not in model.omega.columns:
                cols = [c for c in model.omega.columns if 'omega' in c]
                if not cols:
                    continue
                omega_col = cols[0]

            omega_df = model.omega[['inn', 'year', omega_col]].copy()
            omega_df['year_lag'] = omega_df['year'] + 1
            merged = omega_df.merge(
                omega_df[['inn', 'year', omega_col]].rename(
                    columns={omega_col: 'omega_lag', 'year': 'year_lag'}
                ),
                on=['inn', 'year_lag'], how='inner'
            ).dropna(subset=[omega_col, 'omega_lag'])

            if len(merged) < 100:
                print(f"  {name:<5}  Insufficient observations for AR(1)")
                continue

            y_ar = merged[omega_col].values
            X_ar = np.column_stack([np.ones(len(merged)), merged['omega_lag'].values])
            try:
                coefs_ar, _, _, _ = np.linalg.lstsq(X_ar, y_ar, rcond=None)
                rho   = coefs_ar[1]
                y_hat = X_ar @ coefs_ar
                ss_res = np.sum((y_ar - y_hat) ** 2)
                ss_tot = np.sum((y_ar - y_ar.mean()) ** 2)
                r2_ar  = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

                # Check for rho > 1 (explosive process)
                if rho > 1.0:
                    persist = "⚠ NON-STATIONARY (rho > 1)"
                elif rho > 0.7:
                    persist = "High"
                elif rho > 0.4:
                    persist = "Moderate"
                else:
                    persist = "Low"

                print(f"  {name:<5}  ρ = {rho:.4f}   R² = {r2_ar:.4f}   "
                      f"→ {persist} persistence   (n={len(merged):,})")

                if rho > 1.0:
                    print(f"         Warning: OP rho > 1 - likely due to 7.8% selective sample. "
                          f"Do not use for dynamic TFP forecasts.")

                persist_rows.append((name, f"{rho:.4f}", f"{r2_ar:.4f}"))

            except Exception as e:
                print(f"  {name:<5}  AR(1) failed: {e}")

        self._sections['Persistence'] = persist_rows

    # ------------------------------------------------------------------
    # TFP Innovation Diagnostics
    # ------------------------------------------------------------------
    def _report_xi_diagnostics(self):
        """Diagnostics for TFP innovations xi_it using compute_xi() method."""
        print("\n" + "─" * 60)
        print("  TFP Innovation (ξ_it) Diagnostics")
        print("─" * 60)
        print("  Note: ξ_it = omega_t - E[omega_t | omega_{t-1}].")
        print("        E[xi] ~ 0 by construction (X_g contains constant).")

        for name, model in self.models.items():
            # Skip models without GMM (OLS has no xi innovations)
            if name == 'OLS':
                continue

            if hasattr(model, 'compute_xi'):
                try:
                    xi = model.compute_xi()
                    print(f"\n  {name}:")
                    print(f"    E[xi]     = {xi.mean():.6f}  (should be ~ 0)")
                    print(f"    Std(ξ)    = {xi.std():.4f}")
                    print(f"    Skewness  = {float(pd.Series(xi).skew()):.4f}")
                    print(f"    Kurtosis  = {float(pd.Series(xi).kurtosis()):.4f}")
                    if abs(xi.mean()) > 0.1:
                        print(f"    Warning: E[xi] far from 0 - possible GMM convergence issue.")
                except Exception as e:
                    print(f"  {name}:  compute_xi() failed: {e}")
            else:
                print(f"\n  {name}: compute_xi() not available, xi diagnostics skipped.")

    # ------------------------------------------------------------------
    # 4. Hansen J-test
    # ------------------------------------------------------------------
    def _report_hansen_j(self):
        """Hansen J-test with normalized statistic J/n for large samples."""
        gmm_models = {k: v for k, v in self.models.items() if k in ('LP', 'DJ')}
        if not gmm_models:
            return

        print("\n" + "─" * 60)
        print("  Hansen J-test (Overidentification, GMM models)")
        print("─" * 60)
        print("  H0: All instruments are valid (E[xi × Z] = 0)")
        print("  For n > 100K: J/n is more informative than p-value (J ~ n * misspec.)")

        j_rows = []
        for name, model in gmm_models.items():
            j_stat = model.diagnostics_.get('Hansen_J')
            j_df   = model.diagnostics_.get('Hansen_df')
            j_pval = model.diagnostics_.get('Hansen_pval')
            n_obs  = model.diagnostics_.get('Hansen_n')

            if j_stat is None:
                print(f"  {name:<5}  J not stored in diagnostics_.")
                j_rows.append((name, '—', '—', '—', '—'))
                continue

            j_norm = j_stat / n_obs if n_obs else float('nan')

            # Formal verdict by p-value
            if j_pval > 0.1:
                verdict = "✓ Valid"
            elif j_pval > 0.05:
                verdict = "~ Borderline"
            else:
                verdict = "⚠ Rejected"

            # Economic verdict by J/n (large samples only)
            if n_obs and n_obs > 100_000:
                if j_norm < 0.0001:
                    econ = "[J/n~0: negligible misspecification]"
                elif j_norm < 0.001:
                    econ = "[J/n<0.001: weak misspecification]"
                else:
                    econ = "[J/n>=0.001: check specification]"
            else:
                econ = ""

            j_norm_str = f"{j_norm:.6f}" if n_obs else "—"
            n_str      = f"{n_obs:,}" if n_obs else "—"
            print(f"  {name:<5}  J = {j_stat:.3f},  df = {j_df},  "
                  f"p = {j_pval:.4f}  → {verdict}")
            if n_obs:
                print(f"{'':7}  n = {n_str},  J/n = {j_norm_str}  {econ}")
            j_rows.append((name, f"{j_stat:.3f}", str(j_df),
                           f"{j_pval:.4f}", j_norm_str))

        self._sections['Hansen'] = j_rows

    # ------------------------------------------------------------------
    # Static helper methods
    # ------------------------------------------------------------------
    @staticmethod
    def test_multicollinearity(X: pd.DataFrame) -> pd.DataFrame:
        """VIF for each regressor."""
        X_arr         = sm.add_constant(X).values
        feature_names = ['const'] + list(X.columns)
        vif_data = [
            {'feature': feature_names[i],
             'VIF': variance_inflation_factor(X_arr, i)}
            for i in range(X_arr.shape[1])
        ]
        vif_df = pd.DataFrame(vif_data)
        return vif_df[vif_df['feature'] != 'const'].reset_index(drop=True)

    @staticmethod
    def test_heteroscedasticity(resid, X: pd.DataFrame) -> dict:
        """Breusch-Pagan test."""
        X_with_const = sm.add_constant(X)
        lm_stat, lm_pval, f_stat, f_pval = het_breuschpagan(resid, X_with_const)
        return {
            'LM Statistic':    lm_stat,
            'LM-Test p-value': lm_pval,
            'F-Statistic':     f_stat,
            'F-Test p-value':  f_pval,
        }

    @staticmethod
    def test_autocorrelation(resid) -> dict:
        """Durbin-Watson test."""
        dw_stat = durbin_watson(resid)
        if dw_stat < 1.5:
            interp = 'Positive'
        elif dw_stat > 2.5:
            interp = 'Negative'
        else:
            interp = 'No significant'
        return {'DW Statistic': dw_stat, 'Interpretation': interp}

    @staticmethod
    def test_normality(resid, sample_size: int = 5000) -> dict:
        """Shapiro-Wilk test (on subsample)."""
        resid_arr = np.asarray(resid)
        if len(resid_arr) > sample_size:
            rng = np.random.default_rng(42)
            resid_arr = rng.choice(resid_arr, size=sample_size, replace=False)
        w_stat, p_val = shapiro(resid_arr)
        return {'W Statistic': w_stat, 'p-value': p_val}

    # ------------------------------------------------------------------
    # LaTeX export
    # ------------------------------------------------------------------
    def export_diagnostics_latex(self, filepath: str = "results/diagnostics_table.tex"):
        """Export all diagnostic blocks to a single academic LaTeX table."""
        model_names = list(self.models.keys())
        n_cols      = len(model_names)
        col_fmt     = "l" + "c" * n_cols
        col_header  = " & ".join(model_names)

        # RTS: one row for all models + LP footnote
        rts_by_model = {}
        for r in self._sections.get('RTS', []):
            name, val, kind = r[0], r[1], r[2]
            note = r[3] if len(r) > 3 else ""
            label = f"{val} ({kind})"
            if note:
                label += "$^{\\dagger}$"
            rts_by_model[name] = label
        rts_row     = " & ".join(rts_by_model.get(n, "—") for n in model_names)
        rts_section = f"Returns to Scale & {rts_row} \\\\\n"

        # Persistence
        pers_by_model = {r[0]: r[1] for r in self._sections.get('Persistence', [])}
        pers_row     = " & ".join(pers_by_model.get(n, "—") for n in model_names)
        pers_section = f"TFP persistence ($\\rho$) & {pers_row} \\\\\n"

        # Hansen J
        j_by_model = {
            r[0]: f"{r[1]} (p={r[3]})" if r[1] != "—" else "—"
            for r in self._sections.get('Hansen', [])
        }
        j_row     = " & ".join(j_by_model.get(n, "n/a") for n in model_names)
        j_section = f"Hansen $J$ (p-value) & {j_row} \\\\\n"

        # OLS-specific
        ols_lines   = self._sections.get('OLS', [])
        ols_section = ""
        for label, val in ols_lines:
            padded      = [val] + ["—"] * (n_cols - 1)
            ols_section += f"{label} & {' & '.join(padded)} \\\\\n"

        latex = f"""\\begin{{table}}[htbp]
\\centering
\\caption{{Model Diagnostics}}
\\label{{tab:diagnostics}}
\\begin{{tabular}}{{{col_fmt}}}
\\toprule
 & {col_header} \\\\
\\midrule
\\multicolumn{{{n_cols + 1}}}{{l}}{{\\textit{{Production function diagnostics}}}} \\\\
{rts_section}{pers_section}{j_section}\\midrule
\\multicolumn{{{n_cols + 1}}}{{l}}{{\\textit{{OLS baseline diagnostics}}}} \\\\
{ols_section}\\bottomrule
\\end{{tabular}}
\\begin{{flushleft}}
\\footnotesize
\\textit{{Note:}} RTS = Returns to Scale (sum of input elasticities for $l$, $m$, $k$).
$^{{\\dagger}}$ LP: $\\beta_m$ not identified (materials serve as proxy); RTS excludes $\\beta_m$.
TFP persistence = AR(1) coefficient of model-specific $\\omega_{{it}}$. OP: $\\rho > 1$ indicates
non-stationarity, likely due to 7.8\\% selective sample (positive-investment firms only).
Hansen $J$-statistic tests overidentifying restrictions (H0: all instruments valid),
distributed $\\chi^2(K_z - K_{{\\beta}})$ under H0.
OLS diagnostics: VIF = Variance Inflation Factor; BP = Breusch--Pagan test;
DW = Durbin--Watson statistic; SW = Shapiro--Wilk test (subsample $n = 5000$).
HC3 heteroscedasticity-robust standard errors used throughout OLS.
\\end{{flushleft}}
\\end{{table}}"""

        os.makedirs(os.path.dirname(filepath), exist_ok=True)
        with open(filepath, "w") as f:
            f.write(latex)
        logging.info(f"Diagnostics table saved to {filepath}")

    # ------------------------------------------------------------------
    # Main execution
    # ------------------------------------------------------------------
    def report(self, export_latex: bool = True,
               latex_path: str = "results/diagnostics_table.tex"):
        """Run all diagnostic blocks and optionally export LaTeX."""
        print("\n" + "=" * 60)
        print("  MODEL DIAGNOSTICS REPORT")
        print("=" * 60)

        self._report_ols_diagnostics()
        self._report_rts()
        self._report_tfp_persistence()
        self._report_xi_diagnostics()   
        self._report_hansen_j()

        print("\n" + "─" * 60)

        if export_latex:
            self.export_diagnostics_latex(latex_path)
            print(f"  -> LaTeX table: {latex_path}")

        print("=" * 60)

## 12. Production Function Models (OLS, Olley-Pakes)

In [17]:
class BaseProductionModel:
    """Base class for storing estimation results."""
    def __init__(self):
        self.coef_ = {}
        self.se_ = {}
        self.pvalues_ = {}
        self.diagnostics_ = {}
        self.omega = None


class OLSModel(BaseProductionModel):
    """Step 1: OLS production function estimation (pooled, FE, two-way FE)."""

    def __init__(self, df: pd.DataFrame, y_col: str, x_cols: list, fe: str = None):
        super().__init__()
        self.df = df.dropna(subset=[y_col] + x_cols).copy()
        self.y_col = y_col
        self.x_cols = x_cols
        self.fe = fe

    def fit(self, run_diagnostics=True):
        mode_label = "Pooled OLS" if not self.fe else (
            "Fixed Effects (Entity)" if self.fe == 'entity' else "Two-Way Fixed Effects (Entity+Time)"
        )
        logging.info(f"Running OLS estimation: {mode_label}...")

        if self.fe:
            panel_df = self.df.set_index(['inn', 'year'])
            Y = panel_df[self.y_col]
            X = sm.add_constant(panel_df[self.x_cols])

            model = PanelOLS(
                Y, X,
                entity_effects=(self.fe in ['entity', 'both']),
                time_effects=(self.fe == 'both'),
            )
            self.results = model.fit(cov_type='clustered', cluster_entity=True)

            params = self.results.params
            std_errors = self.results.std_errors
            pvalues = self.results.pvalues
            resid = self.results.resids.values.flatten()
        else:
            Y = self.df[self.y_col]
            X = sm.add_constant(self.df[self.x_cols])

            model = sm.OLS(Y, X)
            self.results = model.fit(cov_type='HC3')

            params = self.results.params
            std_errors = self.results.bse
            pvalues = self.results.pvalues
            resid = self.results.resid

        for col in self.x_cols + (['const'] if 'const' in params else []):
            if col in params:
                self.coef_[col] = params[col]
                self.se_[col] = std_errors[col]
                self.pvalues_[col] = pvalues[col]

        const_val = params.get('const', 0)
        self.df['omega_ols'] = resid + const_val
        self.omega = self.df[['inn', 'year', 'omega_ols']]

        logging.info(f"{mode_label} R-squared: {self.results.rsquared:.4f}")

        if run_diagnostics:
            X_diag = self.df[self.x_cols]
            self.diagnostics_['VIF'] = DiagnosticsReporter.test_multicollinearity(X_diag)
            self.diagnostics_['Heteroscedasticity'] = DiagnosticsReporter.test_heteroscedasticity(resid, X_diag)
            self.diagnostics_['Autocorrelation'] = DiagnosticsReporter.test_autocorrelation(resid)
            self.diagnostics_['Normality'] = DiagnosticsReporter.test_normality(resid)

            if hasattr(self.results, 'f_statistic'):
                self.diagnostics_['f_stat_value'] = self.results.f_statistic.stat
            else:
                self.diagnostics_['f_stat_value'] = getattr(self.results, 'fvalue', None)

        return self

    def summary(self):
        return self.results.summary()


class OlleyPakesModel(BaseProductionModel):
    """
    Olley-Pakes (1996) model.
    Uses investment as proxy to control for endogeneity.
    poly_degree: polynomial degree for control function (default 3).
    """

    def __init__(self, df: pd.DataFrame, y_col: str,
                 free_vars: list, state_vars: list, proxy_var: str,
                 poly_degree: int = 3):
        super().__init__()
        self.y_col = y_col
        self.free_vars = free_vars
        self.state_vars = state_vars
        self.proxy_var = proxy_var
        self.poly_degree = poly_degree

        req_cols = [y_col] + free_vars + state_vars + [proxy_var]
        req_cols += [f'{c}_lag1' for c in state_vars] + [f'{proxy_var}_lag1']
        self.df = df.dropna(subset=req_cols).copy()

        # Survival variable (vectorized via merge)
        self.df['next_year'] = self.df['year'] + 1
        firm_years = self.df[['inn', 'year']].copy()
        firm_years.columns = ['inn', 'next_year']
        merged = self.df[['inn', 'next_year']].merge(
            firm_years, on=['inn', 'next_year'], how='left', indicator=True
        )
        self.df['survived'] = (merged['_merge'] == 'both').astype(int).values

    def _stage_1(self):
        """Stage 1: Purge output of free variables using polynomial."""
        logging.info("OP Stage 1: Estimating free variable coefficients...")

        poly_vars = self.df[self.state_vars + [self.proxy_var]]
        poly = PolynomialFeatures(degree=self.poly_degree, include_bias=False)
        poly_features = poly.fit_transform(poly_vars)

        poly_cols = [f'poly_{i}' for i in range(poly_features.shape[1])]
        poly_df = pd.DataFrame(poly_features, columns=poly_cols, index=self.df.index)

        X_stage1 = pd.concat([self.df[self.free_vars], poly_df], axis=1)
        X_stage1 = sm.add_constant(X_stage1)
        Y = self.df[self.y_col]

        stage1_model = sm.OLS(Y, X_stage1).fit()
        self._s1_results = stage1_model

        for var in self.free_vars:
            self.coef_[var] = stage1_model.params[var]
            self.se_[var] = stage1_model.bse[var]
            self.pvalues_[var] = stage1_model.pvalues[var]

        phi_coefs = stage1_model.params[poly_cols]
        self.df['phi'] = poly_df.dot(phi_coefs) + stage1_model.params['const']
        self.df['phi_lag1'] = self.df.groupby('inn')['phi'].shift(1)

        year_diff = self.df.groupby('inn')['year'].diff()
        self.df.loc[year_diff != 1, 'phi_lag1'] = np.nan

        return self

    def _survival_model(self):
        """Survivor bias control (Probit model)."""
        logging.info("OP Survival Model: Estimating survival probability...")

        lagged_state = [f'{c}_lag1' for c in self.state_vars]
        X_surv = self.df[lagged_state].dropna()
        Y_surv = self.df.loc[X_surv.index, 'survived']

        poly = PolynomialFeatures(degree=2, include_bias=True)
        X_surv_poly = poly.fit_transform(X_surv)

        try:
            probit = sm.Probit(Y_surv, X_surv_poly).fit(disp=0)
            self.df['p_surv'] = probit.predict(
                poly.transform(self.df[lagged_state].fillna(0))
            )
        except np.linalg.LinAlgError:
            logging.warning("Probit did not converge. Using constant survival probability.")
            self.df['p_surv'] = Y_surv.mean()

    def _stage_2(self):
        """Stage 2: NLS optimization for state variable coefficients."""
        logging.info("OP Stage 2: Nonlinear optimization for state variables...")

        opt_df = self.df.dropna(
            subset=['phi_lag1', 'p_surv'] +
                   self.state_vars +
                   [f'{c}_lag1' for c in self.state_vars]
        )

        k_t = opt_df[self.state_vars].values
        k_lag = opt_df[[f'{c}_lag1' for c in self.state_vars]].values
        phi_t = opt_df['phi'].values
        phi_lag = opt_df['phi_lag1'].values
        p_surv = opt_df['p_surv'].values

        def objective_function(beta_state):
            omega = phi_t - np.dot(k_t, beta_state)
            omega_lag = phi_lag - np.dot(k_lag, beta_state)

            X_g = np.column_stack((
                omega_lag, omega_lag**2, omega_lag**3,
                p_surv, p_surv**2,
                omega_lag * p_surv
            ))
            X_g = np.hstack((np.ones((X_g.shape[0], 1)), X_g))

            try:
                gamma = np.linalg.solve(X_g.T @ X_g, X_g.T @ omega)
                expected_omega = X_g @ gamma
            except np.linalg.LinAlgError:
                expected_omega = np.zeros_like(omega)

            xi = omega - expected_omega
            return np.sum(xi**2)

        initial_guess = np.zeros(len(self.state_vars)) + 0.1

        res = minimize(
            objective_function,
            x0=initial_guess,
            method='Nelder-Mead',
            options={'maxiter': 1000, 'disp': False}
        )

        self.diagnostics_['opt_success'] = res.success
        self.diagnostics_['opt_message'] = (
            res.message if hasattr(res, 'message') else str(res)
        )

        self.df['omega_op'] = (
            self.df['phi'] - np.dot(self.df[self.state_vars].values, res.x)
        )

        # AR(1) coefficient of omega
        omega_df_temp = (
            self.df[['inn', 'year', 'omega_op']]
            .dropna()
            .sort_values(['inn', 'year'])
            .copy()
        )
        omega_df_temp['omega_lag'] = omega_df_temp.groupby('inn')['omega_op'].shift(1)
        omega_df_temp = omega_df_temp.dropna()

        if len(omega_df_temp) > 100:
            X_rho = np.column_stack([
                np.ones(len(omega_df_temp)),
                omega_df_temp['omega_lag'].values
            ])
            y_rho = omega_df_temp['omega_op'].values
            rho_coefs = np.linalg.lstsq(X_rho, y_rho, rcond=None)[0]
            self.diagnostics_['rho'] = rho_coefs[1]
        else:
            self.diagnostics_['rho'] = np.nan

        if not res.success:
            logging.warning("Stage 2 optimization did not converge. Check data.")

        for i, var in enumerate(self.state_vars):
            self.coef_[var] = res.x[i]
            self.se_[var] = np.nan
            self.pvalues_[var] = np.nan

        self.omega = self.df[['inn', 'year', 'omega_op']]

        # Cache Stage 2 data for compute_xi()
        self._s2_beta = res.x.copy()
        self._s2_k_t = k_t
        self._s2_k_lag = k_lag
        self._s2_phi_t = phi_t
        self._s2_phi_lag = phi_lag
        self._s2_p_surv = p_surv

        return self

    def compute_xi(self) -> np.ndarray:
        """
        Returns TFP innovations xi_it on Stage 2 sample.
        E[xi] ~ 0 by construction (X_g contains constant).
        """
        if not hasattr(self, '_s2_beta'):
            raise RuntimeError(
                "Stage 2 data not cached. Run fit() before compute_xi()."
            )

        beta = self._s2_beta
        k_t = self._s2_k_t
        k_lag = self._s2_k_lag
        phi_t = self._s2_phi_t
        phi_lag = self._s2_phi_lag
        p_surv = self._s2_p_surv

        omega = phi_t - k_t @ beta
        omega_lag = phi_lag - k_lag @ beta

        X_g = np.column_stack((
            omega_lag, omega_lag**2, omega_lag**3,
            p_surv, p_surv**2,
            omega_lag * p_surv
        ))
        X_g = np.hstack((np.ones((X_g.shape[0], 1)), X_g))

        try:
            gamma = np.linalg.solve(X_g.T @ X_g, X_g.T @ omega)
        except np.linalg.LinAlgError:
            logging.warning(
                "compute_xi(): X_g singular - using lstsq as fallback."
            )
            gamma = np.linalg.lstsq(X_g, omega, rcond=None)[0]

        xi = omega - X_g @ gamma

        logging.info(
            f"OP compute_xi(): E[xi] = {xi.mean():.8f} "
            f"(~ 0 by construction), Std(xi) = {xi.std():.4f}, "
            f"n = {len(xi):,}."
        )
        return xi

    def fit(self):
        """Run full OP pipeline."""
        self._stage_1()
        self._survival_model()
        self._stage_2()
        logging.info("OP estimation completed successfully.")
        return self

## 13. Production Function Models (Levinsohn-Petrin, Doraszelski-Jaumandreu)

In [18]:
class LevinsohnPetrinModel(BaseProductionModel):
    """
    Levinsohn-Petrin (2003) with ACF (2015) correction.

    Key fix (v4): Materials coefficient (beta_m) is now properly identified
    using the ACF approach. Materials serves as proxy for productivity AND
    enters the production function. All three input coefficients
    (beta_l, beta_m, beta_k) are identified in Stage 2 via GMM.

    Stage 1: Nonparametrically estimate phi(k, l, m, r) to purge measurement
             error epsilon. No input coefficients are identified here.
    Stage 2: Two-step GMM identifies [beta_l, beta_m, beta_k] using moment
             conditions E[xi_t * Z_t] = 0, where xi_t are productivity
             innovations and Z_t are valid instruments.

    extra_poly_vars: variables (e.g., l_r) included in the polynomial
                     control function phi (stage 1).
    poly_degree: polynomial degree for control function (default 3).
    """

    def __init__(self, df: pd.DataFrame, y_col: str, free_cols: list,
                 state_cols: list, proxy_col: str, extra_poly_vars: list = None,
                 poly_degree: int = 3):
        super().__init__()
        self.df              = df.copy()
        self.y_col           = y_col
        self.free_cols       = free_cols
        self.state_cols      = state_cols
        self.proxy_col       = proxy_col
        self.extra_poly_vars = extra_poly_vars or []
        self.poly_degree     = poly_degree

        self.free_vars  = free_cols
        self.state_vars = state_cols
        self.proxy_var  = proxy_col

        self.beta_free  = None
        self.beta_state = None
        self.omega      = None
        self.diagnostics_ = {}
        self.results    = None

    def _stage_1(self):
        """
        Stage 1 (ACF 2015): Estimate phi_hat = E[y | k, l, m, r].
        NO input coefficients are separated here — all are absorbed into phi.
        This purges measurement error epsilon from the production function.
        """
        # All variables enter the polynomial: state + free + extra + proxy
        poly_vars = self.state_cols + self.free_cols + self.extra_poly_vars + [self.proxy_col]
        poly      = PolynomialFeatures(degree=self.poly_degree, include_bias=True)
        X_poly    = poly.fit_transform(self.df[poly_vars])

        stage1         = sm.OLS(self.df[self.y_col], X_poly).fit()
        self.df['phi_hat']      = stage1.fittedvalues
        self.df['phi_hat_lag1'] = self.df.groupby('inn')['phi_hat'].shift(1)

    def _compute_xi_Z(self, beta_all, df_gmm):
        """
        Compute productivity innovations xi_it and instrument matrix Z.

        beta_all = [beta_l, beta_m, beta_k] (all input coefficients).

        omega_t = phi_hat_t - beta_l * l_t - beta_m * m_t - beta_k * k_t
        g(omega_{t-1}): 3rd-degree polynomial, gamma profiled analytically.
        Z = [l_lag1, m_lag1, k_lag1, k_t, m_lag1] — valid instruments.
        """
        all_vars = self.free_cols + [self.proxy_col] + self.state_cols
        all_vars_lag = [f'{c}_lag1' for c in all_vars]

        omega_t  = (df_gmm['phi_hat'].values
                    - np.dot(df_gmm[all_vars].values, beta_all))
        omega_t1 = (df_gmm['phi_hat_lag1'].values
                    - np.dot(df_gmm[all_vars_lag].values, beta_all))

        X_g   = np.column_stack([np.ones(len(omega_t1)),
                                  omega_t1, omega_t1**2, omega_t1**3])
        gamma = np.linalg.lstsq(X_g, omega_t, rcond=None)[0]
        xi_it = omega_t - X_g @ gamma

        # Instruments: lagged free vars, lagged proxy, lagged state, current state
        Z_matrix = np.column_stack([
            df_gmm[[f'{c}_lag1' for c in self.free_cols]].values,   # l_{t-1}
            df_gmm[f'{self.proxy_col}_lag1'].values.reshape(-1, 1), # m_{t-1}
            df_gmm[[f'{c}_lag1' for c in self.state_cols]].values,  # k_{t-1}
            df_gmm[self.state_cols].values,                          # k_t
        ])

        return xi_it, Z_matrix

    def _gmm_objective(self, beta_all, df_gmm, W):
        xi_it, Z_matrix = self._compute_xi_Z(beta_all, df_gmm)
        moments = (xi_it[:, None] * Z_matrix).mean(axis=0)
        return float(np.dot(moments.T, np.dot(W, moments)))

    def _stage_2(self):
        """
        Stage 2: Two-step GMM.
        theta = [beta_l, beta_m, beta_k] — all three identified.
        Instruments: [l_{t-1}, m_{t-1}, k_{t-1}, k_t] -> 4 moments.
        n_params = 3 -> df = 1 (just-identified is ok, overid preferred).
        """
        df_gmm = self.df.copy()

        all_vars = self.free_cols + [self.proxy_col] + self.state_cols
        for col in all_vars:
            df_gmm[f'{col}_lag1'] = df_gmm.groupby('inn')[col].shift(1)

        if 'phi_hat_lag1' not in df_gmm.columns:
            df_gmm['phi_hat_lag1'] = df_gmm.groupby('inn')['phi_hat'].shift(1)

        dropna_cols = [f'{col}_lag1' for col in all_vars] + ['phi_hat_lag1']
        df_gmm = df_gmm.dropna(subset=dropna_cols).reset_index(drop=True)

        n_params  = len(all_vars)  # 3: beta_l, beta_m, beta_k
        n_moments = len(self.free_cols) + 1 + 2 * len(self.state_cols)  # l_lag + m_lag + k_lag + k_t

        for col in all_vars:
            if df_gmm[col].std() < 0.01:
                logging.warning(f"LP: {col} nearly constant — beta_{col} may not be identified.")

        # Bounds: labor 0.01-0.95, materials 0.01-0.95, capital 0.001-0.50
        bounds  = [(0.01, 0.95)] * len(self.free_cols) + [(0.01, 0.95)] + [(0.001, 0.50)] * len(self.state_cols)
        initial = np.array([0.10] * len(self.free_cols) + [0.50] + [0.05] * len(self.state_cols))

        W_init = np.eye(n_moments)
        res1   = minimize(self._gmm_objective, initial, args=(df_gmm, W_init),
                          method='L-BFGS-B', bounds=bounds)
        theta1 = res1.x if res1.success else initial
        self.diagnostics_['GMM_criterion_step1'] = float(res1.fun)
        self.diagnostics_['GMM_success']         = res1.success

        xi1, Z1 = self._compute_xi_Z(theta1, df_gmm)
        S       = (Z1.T @ (Z1 * xi1[:, None] ** 2)) / len(xi1)
        W_opt   = np.linalg.pinv(S + 1e-8 * np.eye(n_moments))

        res2 = minimize(self._gmm_objective, theta1, args=(df_gmm, W_opt),
                        method='L-BFGS-B', bounds=bounds)
        beta_final = res2.x if res2.success else theta1
        self.diagnostics_['GMM_criterion'] = float(res2.fun)

        # Split coefficients
        n_free = len(self.free_cols)
        self.beta_free  = beta_final[:n_free]
        self.beta_proxy = beta_final[n_free:n_free+1]
        self.beta_state = beta_final[n_free+1:]

        # Hansen J-test
        xi2, Z2 = self._compute_xi_Z(beta_final, df_gmm)
        n_obs   = len(xi2)
        m2      = (xi2[:, None] * Z2).mean(axis=0)
        j_stat  = n_obs * float(np.dot(m2.T, np.dot(W_opt, m2)))
        j_df    = n_moments - n_params
        j_pval  = 1 - chi2.cdf(j_stat, df=max(j_df, 1))
        self.diagnostics_['Hansen_J']    = j_stat
        self.diagnostics_['Hansen_df']   = j_df
        self.diagnostics_['Hansen_pval'] = j_pval
        self.diagnostics_['Hansen_n']    = n_obs

        if n_obs > 100_000 and j_pval < 0.05:
            logging.warning(
                f"LP Hansen J={j_stat:.1f}, df={j_df}, p={j_pval:.4f} at n={n_obs:,}. "
                f"Normalized J/n = {j_stat/n_obs:.6f}."
            )

        self._s2_df_gmm = df_gmm
        self._s2_beta   = beta_final.copy()

    def _compute_productivity(self):
        """TFP = y - beta_l*l - beta_m*m - beta_k*k."""
        all_vars = self.free_cols + [self.proxy_col] + self.state_cols
        all_betas = np.concatenate([self.beta_free, self.beta_proxy, self.beta_state])
        y_hat = np.dot(self.df[all_vars].values, all_betas)
        self.df['omega_lp'] = self.df[self.y_col] - y_hat
        self.df['omega_lp'] -= self.df['omega_lp'].mean()
        self.omega = self.df[['inn', 'year', 'omega_lp']].copy()

    def compute_xi(self) -> np.ndarray:
        """True innovations xi_it. E[xi]=0 by construction."""
        if not hasattr(self, '_s2_beta'):
            raise RuntimeError("Run fit() before compute_xi().")
        xi, _ = self._compute_xi_Z(self._s2_beta, self._s2_df_gmm)
        logging.info(f"LP compute_xi(): E[xi]={xi.mean():.6f}, Std={xi.std():.4f}, n={len(xi):,}.")
        return xi

    def fit(self, run_diagnostics=False):
        self._stage_1()
        self._stage_2()
        self._compute_productivity()

        # Store all coefficients in coef_ dict
        for i, col in enumerate(self.free_cols):
            self.coef_[col] = self.beta_free[i]
        self.coef_[self.proxy_col] = self.beta_proxy[0]
        for i, col in enumerate(self.state_cols):
            self.coef_[col] = self.beta_state[i]

        self.results        = _DummyResults()
        self.results.params = pd.Series({
            **{c: self.beta_free[i]  for i, c in enumerate(self.free_cols)},
            self.proxy_col: self.beta_proxy[0],
            **{c: self.beta_state[i] for i, c in enumerate(self.state_cols)},
        })
        return self


class Doraszelski_Jaumandreu(BaseProductionModel):
    """
    Doraszelski & Jaumandreu (2013) model with ACF (2015) identification.

    theta = [beta_l, beta_m, beta_k] (3 parameters).
    Transition parameters gamma estimated analytically via lstsq.
    5 instruments: [l_lag1, m_lag1, k_t, r_lag1, k_lag1].
    """

    def __init__(self, df: pd.DataFrame, y_col: str, free_cols: list, state_cols: list, r_col: str, poly_degree: int = 3):
        super().__init__()
        self.df         = df.copy()
        self.y_col      = y_col
        self.free_cols  = free_cols
        self.state_cols = state_cols
        self.r_col      = r_col
        self.poly_degree = poly_degree

        self.free_vars  = free_cols
        self.state_vars = state_cols
        self.rd_var     = r_col
        self.all_vars   = free_cols + state_cols + [r_col]

        self.l_col = free_cols[0]
        self.m_col = free_cols[1]
        self.k_col = state_cols[0]

        self.theta_hat = None
        self.gamma_    = None
        self.omega     = None
        self.diagnostics_ = {}
        self.results   = None

    def _stage_1(self):
        """Stage 1: phi_hat = y - epsilon via polynomial of (l, m, k, r)."""
        poly_vars = [self.l_col, self.m_col, self.k_col, self.r_col]
        poly      = PolynomialFeatures(degree=3, include_bias=True)
        X_poly    = poly.fit_transform(self.df[poly_vars])

        model_s1 = sm.OLS(self.df[self.y_col], X_poly).fit()
        self.df['phi_hat']      = model_s1.fittedvalues
        self.df['phi_hat_lag1'] = self.df.groupby('inn')['phi_hat'].shift(1)

    def _compute_xi_and_Z(self, theta_beta, df_gmm):
        """
        Compute innovations xi_it and instrument matrix Z (5 columns).

        theta_beta = [beta_l, beta_m, beta_k].
        Gamma profiled analytically (lstsq). Transition function is
        a complete polynomial of degree self.poly_degree in (omega, r):
          - degree 2:  6 parameters (g0..g5)
          - degree 3: 10 parameters (g0..g9), following DJ (2013) original.

        Unified ordering of gamma indices (same base 6 for both degrees):
            g0: const      g1: omega        g2: r
            g3: omega^2    g4: r^2          g5: omega*r
            g6: omega^3    g7: r^3          g8: omega^2*r    g9: omega*r^2
        """
        b_l, b_m, b_k = theta_beta

        w_t  = (df_gmm['phi_hat'].values
                - b_l * df_gmm[self.l_col].values
                - b_m * df_gmm[self.m_col].values
                - b_k * df_gmm[self.k_col].values)

        w_t1 = (df_gmm['phi_hat_lag1'].values
                - b_l * df_gmm[f'{self.l_col}_lag1'].values
                - b_m * df_gmm[f'{self.m_col}_lag1'].values
                - b_k * df_gmm[f'{self.k_col}_lag1'].values)

        r_t1 = df_gmm[f'{self.r_col}_lag1'].values

        # Degree-2 base (6 terms)
        base_cols = [
            np.ones(len(w_t)),     # g0: constant
            w_t1,                  # g1: omega_{t-1}
            r_t1,                  # g2: r_{t-1}
            w_t1 ** 2,             # g3: omega_{t-1}^2
            r_t1 ** 2,             # g4: r_{t-1}^2
            w_t1 * r_t1,           # g5: omega_{t-1} * r_{t-1}
        ]

        if self.poly_degree == 3:
            X_g = np.column_stack(base_cols + [
                w_t1 ** 3,             # g6: omega_{t-1}^3
                r_t1 ** 3,             # g7: r_{t-1}^3
                (w_t1 ** 2) * r_t1,    # g8: omega_{t-1}^2 * r_{t-1}
                w_t1 * (r_t1 ** 2),    # g9: omega_{t-1} * r_{t-1}^2
            ])
        elif self.poly_degree == 2:
            X_g = np.column_stack(base_cols)
        else:
            raise ValueError(f"poly_degree must be 2 or 3, got {self.poly_degree}")

        gamma = np.linalg.lstsq(X_g, w_t, rcond=None)[0]
        xi_it = w_t - X_g @ gamma

        # 5 instruments (unchanged)
        Z_matrix = np.column_stack((
            df_gmm[f'{self.l_col}_lag1'].values,
            df_gmm[f'{self.m_col}_lag1'].values,
            df_gmm[self.k_col].values,
            r_t1,
            df_gmm[f'{self.k_col}_lag1'].values,
        ))

        return xi_it, Z_matrix, gamma

    def _gmm_objective(self, theta_beta, df_gmm, W):
        """GMM objective: g'Wg."""
        xi_it, Z_matrix, _ = self._compute_xi_and_Z(theta_beta, df_gmm)
        moments = (xi_it[:, None] * Z_matrix).mean(axis=0)
        return float(np.dot(moments.T, np.dot(W, moments)))

    def _stage_2(self):
        """
        Stage 2: Two-step GMM.
        theta = [beta_l, beta_m, beta_k]. 5 instruments -> df = 2.
        """
        df_gmm = self.df.copy()

        lag_cols = [self.l_col, self.m_col, self.k_col, self.r_col]
        for col in lag_cols:
            df_gmm[f'{col}_lag1'] = df_gmm.groupby('inn')[col].shift(1)
        if 'phi_hat_lag1' not in df_gmm.columns:
            df_gmm['phi_hat_lag1'] = df_gmm.groupby('inn')['phi_hat'].shift(1)

        dropna_cols = [f'{col}_lag1' for col in lag_cols] + ['phi_hat_lag1']
        df_gmm = df_gmm.dropna(subset=dropna_cols).reset_index(drop=True)

        N_MOMENTS = 5
        bounds = [(0.01, 0.95), (0.01, 0.95), (0.001, 0.50)]
        initial = np.array([0.10, 0.80, 0.05])

        # Step 1: W = I
        W_init = np.eye(N_MOMENTS)
        res1   = minimize(self._gmm_objective, initial,
                          args=(df_gmm, W_init),
                          method='L-BFGS-B', bounds=bounds)
        theta1 = res1.x if res1.success else initial
        self.diagnostics_['GMM_criterion_step1'] = float(res1.fun)
        self.diagnostics_['GMM_success']         = res1.success

        # Step 2: optimal W
        xi1, Z1, _ = self._compute_xi_and_Z(theta1, df_gmm)
        S          = (Z1.T @ (Z1 * xi1[:, None] ** 2)) / len(xi1)
        W_opt      = np.linalg.pinv(S + 1e-8 * np.eye(N_MOMENTS))

        res2 = minimize(self._gmm_objective, theta1,
                        args=(df_gmm, W_opt),
                        method='L-BFGS-B', bounds=bounds)
        self.theta_hat = res2.x if res2.success else theta1
        self.diagnostics_['GMM_criterion'] = float(res2.fun)

        # Hansen J-test (df = 5 - 3 = 2)
        xi_f, Z_f, gamma_arr = self._compute_xi_and_Z(self.theta_hat, df_gmm)
        n_obs  = len(xi_f)
        m_f    = (xi_f[:, None] * Z_f).mean(axis=0)
        j_stat = n_obs * float(np.dot(m_f.T, np.dot(W_opt, m_f)))
        j_df   = N_MOMENTS - len(self.theta_hat)
        j_pval = 1 - chi2.cdf(j_stat, df=j_df)
        self.diagnostics_['Hansen_J']    = j_stat
        self.diagnostics_['Hansen_df']   = j_df
        self.diagnostics_['Hansen_pval'] = j_pval
        self.diagnostics_['Hansen_n']    = n_obs

        if n_obs > 100_000 and j_pval < 0.05:
            logging.warning(
                f"DJ Hansen J={j_stat:.1f}, p={j_pval:.4f} at n={n_obs:,}. "
                f"In large samples J rejects even negligible misspecification. "
                f"Consider reviewing the transition function specification."
            )

        # Store gamma_ dict (g0..g5 for degree-2; g0..g9 for degree-3)
        gamma_keys_base = ['g0', 'g1', 'g2', 'g3', 'g4', 'g5']
        gamma_keys_ext  = ['g6', 'g7', 'g8', 'g9']
        gamma_keys = gamma_keys_base if self.poly_degree == 2 else (gamma_keys_base + gamma_keys_ext)
        self.gamma_ = {k: float(gamma_arr[idx]) for idx, k in enumerate(gamma_keys)}

        # Cache for compute_xi()
        self._s2_df_gmm = df_gmm
        self._s2_beta   = self.theta_hat.copy()

    def _compute_productivity(self):
        """TFP = y - beta_l*l - beta_m*m - beta_k*k.

        Stores BOTH the raw structural omega (on which the gamma_
        transition-function coefficients were fitted) and the
        mean-centred omega used for distribution plots.  Downstream
        code that evaluates g(omega, r) or its derivatives MUST use
        the raw omega to stay consistent with gamma_.
        """
        b_l, b_m, b_k = self.theta_hat
        y_hat = (b_l * self.df[self.l_col]
                 + b_m * self.df[self.m_col]
                 + b_k * self.df[self.k_col])
        self.df['omega_dj_raw'] = self.df[self.y_col] - y_hat
        self.df['omega_dj']     = self.df['omega_dj_raw'] - self.df['omega_dj_raw'].mean()
        # omega_raw_ is the quantity that matches gamma_ (for plot/tables)
        self.omega_raw_ = self.df[['inn', 'year', 'omega_dj_raw']].dropna().copy()
        # Legacy: demeaned omega merged into Visualizer.tfp_df for TFP plots
        self.omega = self.df[['inn', 'year', 'omega_dj']].copy()

    def fit(self, run_diagnostics=False):
        """Run pipeline: Stage 1 -> Stage 2 -> TFP."""
        self._stage_1()
        self._stage_2()
        self._compute_productivity()
        logging.info("DJ estimation completed successfully.")

        self.coef_[self.l_col] = self.theta_hat[0]
        self.coef_[self.m_col] = self.theta_hat[1]
        self.coef_[self.k_col] = self.theta_hat[2]
        if self.gamma_:
            for k, v in self.gamma_.items():
                self.coef_[k] = v

        self.results        = _DummyResults()
        self.results.params = pd.Series({
            self.l_col: self.theta_hat[0],
            self.m_col: self.theta_hat[1],
            self.k_col: self.theta_hat[2],
            **{f'gamma_{i}': v for i, v in enumerate(self.gamma_.values())},
        })
        return self

    def compute_xi(self) -> np.ndarray:
        """
        Returns true innovations xi_it.
        E[xi] = 0 by construction (g contains constant).
        """
        if not hasattr(self, '_s2_beta'):
            raise RuntimeError("Run fit() before compute_xi().")
        xi, _, _ = self._compute_xi_and_Z(self._s2_beta, self._s2_df_gmm)
        logging.info(
            f"DJ compute_xi(): E[xi]={xi.mean():.8f} "
            f"(~ 0 by construction), Std={xi.std():.4f}, n={len(xi):,}."
        )
        return xi

### 13.1 LP Instrument Set Comparison (1-Lag vs 2-Lag)

In [ ]:
def compare_lp_lag_specifications(df, y_col='l_y', free_cols=None,
                                  state_cols=None, proxy_col='l_m',
                                  extra_poly_vars=None):
    """
    Runs LP with 1-lag instruments (baseline) and 2-lag instruments (extended),
    then compares Hansen J-test results to justify the instrument choice.

    1-lag Z: [k_lag1, l_lag1, k_t, m_lag1]           → 4 moments, df=3
    2-lag Z: [k_lag1, l_lag1, k_t, m_lag1,
              k_lag2, l_lag2, m_lag2]                  → 7 moments, df=6
    """
    free_cols = free_cols or ['l_l']
    state_cols = state_cols or ['l_k', 'l_r']
    extra_poly_vars = extra_poly_vars or []

    W = 70

    print('\n' + '=' * W)
    print('  LP INSTRUMENT SET COMPARISON: 1-Lag vs 2-Lag')
    print('=' * W)
    print()
    print('  Motivation: Adding deeper lags as instruments increases')
    print('  overidentifying restrictions and tightens the J-test.')
    print('  If both specs pass, the parsimonious 1-lag set is preferred.')
    print()

    # ── Helper: run LP GMM with a given instrument builder ─────────────
    def _run_lp_gmm(df_src, build_Z_func, n_moments_expected):
        """
        Shared LP GMM core.  Returns dict with beta, J-stat, n_obs, etc.
        build_Z_func(df_gmm) -> Z matrix of shape (n, n_moments_expected).
        """
        model_df = df_src.copy()

        # Stage 1 (ACF) ------------------------------------------------
        poly_vars = state_cols + free_cols + extra_poly_vars + [proxy_col]
        poly = PolynomialFeatures(degree=3, include_bias=True)
        X_poly = poly.fit_transform(model_df[poly_vars])

        s1 = sm.OLS(model_df[y_col], X_poly).fit()
        beta_free = np.array([])  # ACF: no coefficients identified in stage 1

        model_df['phi_hat'] = s1.fittedvalues
        model_df['phi_hat_lag1'] = model_df.groupby('inn')['phi_hat'].shift(1)

        # Create all lags needed ----------------------------------------
        lag_cols_base = state_cols + free_cols + [proxy_col]
        for col in lag_cols_base:
            model_df[f'{col}_lag1'] = model_df.groupby('inn')[col].shift(1)
            model_df[f'{col}_lag2'] = model_df.groupby('inn')[col].shift(2)

        model_df['phi_hat_lag2'] = model_df.groupby('inn')['phi_hat'].shift(2)

        # Dropna for required columns -----------------------------------
        # Determine which lag columns are needed based on n_moments
        dropna_cols = [f'{c}_lag1' for c in lag_cols_base] + ['phi_hat_lag1']
        if n_moments_expected > 4:
            dropna_cols += [f'{c}_lag2' for c in lag_cols_base] + ['phi_hat_lag2']

        df_gmm = model_df.dropna(subset=dropna_cols).reset_index(drop=True)

        # Stage 2: two-step GMM ----------------------------------------
        n_params = len(free_cols) + 1 + len(state_cols)  # l, m, k
        state_lag1_cols = [f'{c}_lag1' for c in state_cols]

        all_input_cols = free_cols + [proxy_col] + state_cols
        all_input_lag_cols = [f'{c}_lag1' for c in all_input_cols]

        def compute_xi_Z(beta_all):
            omega_t = (df_gmm['phi_hat'].values
                       - np.dot(df_gmm[all_input_cols].values, beta_all))
            omega_t1 = (df_gmm['phi_hat_lag1'].values
                        - np.dot(df_gmm[all_input_lag_cols].values, beta_all))
            X_g = np.column_stack([np.ones(len(omega_t1)),
                                   omega_t1, omega_t1**2, omega_t1**3])
            gamma = np.linalg.lstsq(X_g, omega_t, rcond=None)[0]
            xi = omega_t - X_g @ gamma
            Z = build_Z_func(df_gmm)
            return xi, Z

        def gmm_obj(beta_state, W_mat):
            xi, Z = compute_xi_Z(beta_state)
            m = (xi[:, None] * Z).mean(axis=0)
            return float(m.T @ W_mat @ m)

        bounds = [(0.01, 0.95)] * len(free_cols) + [(0.01, 0.95)] + [(0.001, 0.50)] * len(state_cols)
        initial = np.array([0.10] * len(free_cols) + [0.50] + [0.05] * len(state_cols))
        n_moments = n_moments_expected

        # Step 1: identity W
        W_I = np.eye(n_moments)
        res1 = minimize(gmm_obj, initial, args=(W_I,),
                        method='L-BFGS-B', bounds=bounds)
        theta1 = res1.x if res1.success else initial

        # Step 2: optimal W
        xi1, Z1 = compute_xi_Z(theta1)
        S = (Z1.T @ (Z1 * xi1[:, None]**2)) / len(xi1)
        W_opt = np.linalg.pinv(S + 1e-8 * np.eye(n_moments))

        res2 = minimize(gmm_obj, theta1, args=(W_opt,),
                        method='L-BFGS-B', bounds=bounds)
        beta_final = res2.x if res2.success else theta1
        gmm_crit = float(res2.fun)

        # Hansen J
        xi_f, Z_f = compute_xi_Z(beta_final)
        n_obs = len(xi_f)
        m_f = (xi_f[:, None] * Z_f).mean(axis=0)
        j_stat = n_obs * float(m_f.T @ W_opt @ m_f)
        j_df = n_moments - n_params
        j_pval = 1 - chi2.cdf(j_stat, df=j_df)

        return {
            'beta_k': beta_final[-1],
            'beta_free': beta_final[:len(free_cols)],
            'beta_m': beta_final[len(free_cols)] if len(free_cols) + 1 <= len(beta_final) else np.nan,
            'n_obs': n_obs,
            'n_moments': n_moments,
            'n_params': n_params,
            'j_df': j_df,
            'j_stat': j_stat,
            'j_pval': j_pval,
            'j_norm': j_stat / n_obs if n_obs else float('nan'),
            'gmm_crit': gmm_crit,
            'success': res2.success,
        }

    # ── Instrument builders ────────────────────────────────────────────
    def Z_1lag(df_gmm):
        """Baseline: [k_lag1, l_lag1, k_t, m_lag1]"""
        return np.column_stack([
            df_gmm[[f'{c}_lag1' for c in state_cols]].values,
            df_gmm[[f'{c}_lag1' for c in free_cols]].values,
            df_gmm[state_cols].values,
            df_gmm[f'{proxy_col}_lag1'].values,
        ])

    def Z_2lag(df_gmm):
        """Extended: 1-lag + [k_lag2, l_lag2, m_lag2]"""
        return np.column_stack([
            df_gmm[[f'{c}_lag1' for c in state_cols]].values,
            df_gmm[[f'{c}_lag1' for c in free_cols]].values,
            df_gmm[state_cols].values,
            df_gmm[f'{proxy_col}_lag1'].values,
            df_gmm[[f'{c}_lag2' for c in state_cols]].values,
            df_gmm[[f'{c}_lag2' for c in free_cols]].values,
            df_gmm[f'{proxy_col}_lag2'].values,
        ])

    # ── Run both specifications ────────────────────────────────────────
    print('  Running 1-lag specification...')
    r1 = _run_lp_gmm(df, Z_1lag, n_moments_expected=2*len(state_cols) + len(free_cols) + 1)
    print('  Running 2-lag specification...')
    r2 = _run_lp_gmm(df, Z_2lag, n_moments_expected=3*len(state_cols) + 2*len(free_cols) + 2)

    # ── Print comparison table ─────────────────────────────────────────
    print()
    print('-' * W)
    hdr = f"  {'':30s} {'1-Lag':>16s} {'2-Lag':>16s}"
    print(hdr)
    print('-' * W)

    def _row(label, v1, v2, fmt='.4f'):
        s1 = f"{v1:{fmt}}" if isinstance(v1, float) else str(v1)
        s2 = f"{v2:{fmt}}" if isinstance(v2, float) else str(v2)
        print(f"  {label:30s} {s1:>16s} {s2:>16s}")

    _row('Instruments',
         'k₋₁,l₋₁,k,m₋₁',
         '+ k₋₂,l₋₂,m₋₂', fmt='s')
    _row('N moments', r1['n_moments'], r2['n_moments'], fmt='d')
    _row('N parameters', r1['n_params'], r2['n_params'], fmt='d')
    _row('Overidentification df', r1['j_df'], r2['j_df'], fmt='d')
    _row('N observations', r1['n_obs'], r2['n_obs'], fmt=',d')
    _row('β_k (capital)', r1['beta_k'], r2['beta_k'])
    _row('β_l (labor)', float(r1['beta_free'][0]) if len(r1['beta_free']) > 0 else float('nan'),
                       float(r2['beta_free'][0]) if len(r2['beta_free']) > 0 else float('nan'))
    if 'beta_m' in r1:
        _row('β_m (materials)', r1.get('beta_m', float('nan')), r2.get('beta_m', float('nan')))
    _row('GMM criterion', r1['gmm_crit'], r2['gmm_crit'], fmt='.8f')
    _row('Hansen J', r1['j_stat'], r2['j_stat'])
    _row('J p-value', r1['j_pval'], r2['j_pval'])
    _row('J/n (normalized)', r1['j_norm'], r2['j_norm'], fmt='.8f')

    print('-' * W)

    # ── Recommendation ─────────────────────────────────────────────────
    print()
    # Both pass at 5%?
    both_pass = r1['j_pval'] > 0.05 and r2['j_pval'] > 0.05
    both_fail = r1['j_pval'] <= 0.05 and r2['j_pval'] <= 0.05
    only_1_pass = r1['j_pval'] > 0.05 and r2['j_pval'] <= 0.05
    only_2_pass = r1['j_pval'] <= 0.05 and r2['j_pval'] > 0.05
    coef_stable = abs(r1['beta_k'] - r2['beta_k']) < 0.02

    print('  INTERPRETATION:')
    if both_pass:
        print('  Both instrument sets pass the Hansen J-test (p > 0.05).')
        if coef_stable:
            print(f'  β_k is stable across specs (Δ = {abs(r1["beta_k"]-r2["beta_k"]):.4f}).')
            print('  → RECOMMENDATION: Use 1-lag instruments (parsimonious).')
            print('    Adding lag2 instruments does not improve identification')
            print('    and reduces sample size.')
        else:
            print(f'  β_k differs across specs (Δ = {abs(r1["beta_k"]-r2["beta_k"]):.4f}).')
            print('  → CAUTION: Coefficient instability may indicate weak instruments.')
            print('    Consider using 1-lag (larger sample) but note sensitivity.')
    elif both_fail:
        print('  Both instrument sets fail the Hansen J-test (p ≤ 0.05).')
        # Check J/n for large-sample nuance
        if r1['n_obs'] > 100_000:
            print(f'  However, with n={r1["n_obs"]:,}, J-test is overpowered.')
            print(f'  J/n (1-lag) = {r1["j_norm"]:.8f}')
            print(f'  J/n (2-lag) = {r2["j_norm"]:.8f}')
            if r1['j_norm'] < 0.001 and r2['j_norm'] < 0.001:
                print('  J/n < 0.001 for both → misspecification is negligible.')
                print('  → RECOMMENDATION: Use 1-lag instruments (parsimonious).')
            else:
                print('  → RECOMMENDATION: Use 1-lag, but note potential misspecification.')
        else:
            print('  → WARNING: Instrument validity questionable. Review specification.')
    elif only_1_pass:
        print('  1-lag passes J-test but 2-lag fails.')
        print('  Adding deeper lags introduces invalid instruments (serial corr.).')
        print('  → RECOMMENDATION: Use 1-lag instruments.')
    elif only_2_pass:
        print('  1-lag fails J-test but 2-lag passes — unusual result.')
        print('  → RECOMMENDATION: Investigate further. 1-lag should be subset of 2-lag.')
        print('    This may indicate numerical instability.')

    print()
    print('=' * W)

    return {'1_lag': r1, '2_lag': r2}

### 13.2 Polynomial Degree Sensitivity Analysis

In [ ]:
class PolynomialSensitivityAnalyzer:
    """
    Systematic comparison of polynomial degrees (2, 3, 4) for OP and LP
    control functions across different industry sectors.
    """

    DEGREES = [2, 3, 4]

    def __init__(self, df: pd.DataFrame, sectors: list = None):
        self.df = df
        self.sectors = sectors or ['All']
        self.results = {}  # {(method, sector, degree): {'beta_l': ..., 'beta_k': ..., ...}}

    def _run_op(self, df_sub, degree):
        """Run OP estimation with given polynomial degree."""
        try:
            model = OlleyPakesModel(
                df_sub, 'l_y', ['l_l', 'l_m'], ['l_k', 'l_r'], 'l_inv',
                poly_degree=degree
            )
            model.fit()
            return model.coef_.copy()
        except Exception as e:
            logging.warning(f"OP degree={degree} failed: {e}")
            return None

    def _run_lp(self, df_sub, degree):
        """Run LP estimation with given polynomial degree."""
        try:
            model = LevinsohnPetrinModel(
                df_sub, 'l_y', ['l_l'], ['l_k', 'l_r'], 'l_m',
                poly_degree=degree
            )
            model.fit()
            return model.coef_.copy()
        except Exception as e:
            logging.warning(f"LP degree={degree} failed: {e}")
            return None

    def run(self):
        """Run sensitivity analysis across degrees and sectors."""
        logging.info("Starting polynomial sensitivity analysis...")

        for sector in self.sectors:
            if sector == 'All':
                df_sub = self.df
            else:
                df_sub = self.df[self.df['sector'] == sector]

            if len(df_sub) < 500:
                logging.warning(f"Sector '{sector}': only {len(df_sub)} obs, skipping.")
                continue

            n_firms = df_sub['inn'].nunique()
            logging.info(f"Sector: {sector} (n={len(df_sub):,}, firms={n_firms:,})")

            for degree in self.DEGREES:
                # OP
                op_coefs = self._run_op(df_sub, degree)
                if op_coefs is not None:
                    self.results[('OP', sector, degree)] = op_coefs

                # LP
                lp_coefs = self._run_lp(df_sub, degree)
                if lp_coefs is not None:
                    self.results[('LP', sector, degree)] = lp_coefs

        logging.info(f"Polynomial sensitivity: {len(self.results)} specifications estimated.")
        return self

    def get_summary_df(self) -> pd.DataFrame:
        """Return results as a tidy DataFrame."""
        rows = []
        for (method, sector, degree), coefs in self.results.items():
            row = {'Method': method, 'Sector': sector, 'Degree': degree}
            row.update(coefs)
            # RTS
            rts_vars = [v for v in ['l_l', 'l_m', 'l_k'] if v in coefs]
            row['RTS'] = sum(coefs[v] for v in rts_vars)
            rows.append(row)
        return pd.DataFrame(rows)

    def print_summary(self):
        """Print formatted sensitivity analysis table."""
        df_res = self.get_summary_df()
        if df_res.empty:
            print("No results to display.")
            return

        W = 80
        print('\n' + '=' * W)
        print('  POLYNOMIAL DEGREE SENSITIVITY ANALYSIS')
        print('=' * W)

        for method in ['OP', 'LP']:
            method_df = df_res[df_res['Method'] == method]
            if method_df.empty:
                continue

            print(f'\n  Method: {method}')
            print('-' * W)

            for sector in method_df['Sector'].unique():
                sec_df = method_df[method_df['Sector'] == sector].sort_values('Degree')

                if sector != 'All':
                    print(f'\n  Sector: {sector}')

                header = f"  {'Degree':>8}"
                coef_vars = ['l_l', 'l_m', 'l_k', 'l_r', 'RTS']
                labels = {'l_l': 'β_l', 'l_m': 'β_m', 'l_k': 'β_k', 'l_r': 'β_r', 'RTS': 'RTS'}
                for v in coef_vars:
                    if v in sec_df.columns:
                        header += f"  {labels.get(v, v):>10}"
                print(header)
                print('  ' + '-' * (len(header) - 2))

                for _, row in sec_df.iterrows():
                    line = f"  {int(row['Degree']):>8}"
                    for v in coef_vars:
                        if v in row and not pd.isna(row.get(v)):
                            line += f"  {row[v]:>10.4f}"
                        elif v in row:
                            line += f"  {'---':>10}"
                    print(line)

                # Stability check
                if len(sec_df) >= 2:
                    for v in ['l_l', 'l_k']:
                        if v in sec_df.columns:
                            vals = sec_df[v].dropna()
                            if len(vals) >= 2:
                                spread = vals.max() - vals.min()
                                stable = "STABLE" if spread < 0.02 else "SENSITIVE"
                                print(f"  {labels.get(v, v)} range: {spread:.4f} ({stable})")

            print()

        print('=' * W)


### 13.3 Industry-Specific Estimation (Per-Model Analysis)

In [ ]:
# ── 8 production sectors retained for cross-sectoral analysis ──
PRODUCTION_SECTORS = [
    'Manufacturing',
    'Mining',
    'Agriculture, forestry, hunting, fishing and fish farming',
    'Construction',
    'Provision of electric energy, gas and steam; air conditioning',
    'Water supply; sanitation, waste collection and disposal, pollution control activities',
    'Information and communication activities',
    'Transportation and storage',
]


class IndustryEstimator:
    """
    Runs all 6 estimation methods separately for each industry sector.
    Methods: OLS, FE (Entity), FE+Time (Two-Way), OP, LP, DJ.
    Stores per-sector results for comparison and visualization.

    By default, only the 8 production sectors in PRODUCTION_SECTORS are
    estimated.  Pass sectors=None, filter_production=False to revert to
    the old behaviour (all sectors above MIN_OBS / MIN_FIRMS).
    """

    ALL_METHODS = ['OLS', 'FE', 'FE_TW', 'OP', 'LP', 'DJ']
    METHOD_LABELS = {
        'OLS': 'Pooled OLS',
        'FE': 'Fixed Effects (Entity)',
        'FE_TW': 'Two-Way Fixed Effects',
        'OP': 'Olley-Pakes',
        'LP': 'Levinsohn-Petrin',
        'DJ': 'Doraszelski-Jaumandreu',
    }

    MIN_OBS = 1000
    MIN_FIRMS = 50

    def __init__(self, df: pd.DataFrame, sectors: list = None,
                 filter_production: bool = True):
        self.df = df
        if sectors is not None:
            self.sectors = sectors
        elif filter_production:
            # Keep only the 8 production sectors that also pass size thresholds
            df_prod = df[df['sector'].isin(PRODUCTION_SECTORS)]
            sec_counts = df_prod.groupby('sector').agg(
                n_obs=('inn', 'count'),
                n_firms=('inn', 'nunique')
            )
            valid = sec_counts[(sec_counts['n_obs'] >= self.MIN_OBS) &
                              (sec_counts['n_firms'] >= self.MIN_FIRMS)]
            self.sectors = valid.index.tolist()
        else:
            sec_counts = df.groupby('sector').agg(
                n_obs=('inn', 'count'),
                n_firms=('inn', 'nunique')
            )
            valid = sec_counts[(sec_counts['n_obs'] >= self.MIN_OBS) &
                              (sec_counts['n_firms'] >= self.MIN_FIRMS)]
            self.sectors = valid.index.tolist()

        self.sector_results = {}
        self.sector_coefs = {}
        self.sector_tfp = {}

    def _estimate_sector(self, sector):
        """Run all 6 methods for a single sector."""
        df_sec = self.df[self.df['sector'] == sector].copy()
        n_obs = len(df_sec)
        n_firms = df_sec['inn'].nunique()

        logging.info(f"Industry: {sector[:50]}... (n={n_obs:,}, firms={n_firms:,})")

        models = {}
        coefs = {}

        # 1. Pooled OLS
        try:
            ols = OLSModel(df_sec, 'l_y', ['l_l', 'l_m', 'l_k', 'l_r'])
            ols.fit(run_diagnostics=False)
            models['OLS'] = ols
            coefs['OLS'] = ols.coef_.copy()
        except Exception as e:
            logging.warning(f"  OLS failed for {sector[:30]}: {e}")

        # 2. Fixed Effects (Entity)
        try:
            fe = OLSModel(df_sec, 'l_y', ['l_l', 'l_m', 'l_k', 'l_r'], fe='entity')
            fe.fit(run_diagnostics=False)
            models['FE'] = fe
            coefs['FE'] = fe.coef_.copy()
        except Exception as e:
            logging.warning(f"  FE failed for {sector[:30]}: {e}")

        # 3. Two-Way Fixed Effects (Entity + Time)
        try:
            fe_tw = OLSModel(df_sec, 'l_y', ['l_l', 'l_m', 'l_k', 'l_r'], fe='both')
            fe_tw.fit(run_diagnostics=False)
            models['FE_TW'] = fe_tw
            coefs['FE_TW'] = fe_tw.coef_.copy()
        except Exception as e:
            logging.warning(f"  FE+Time failed for {sector[:30]}: {e}")

        # 4. Olley-Pakes
        try:
            op = OlleyPakesModel(df_sec, 'l_y', ['l_l', 'l_m'], ['l_k', 'l_r'], 'l_inv')
            op.fit()
            models['OP'] = op
            coefs['OP'] = op.coef_.copy()
        except Exception as e:
            logging.warning(f"  OP failed for {sector[:30]}: {e}")

        # 5. Levinsohn-Petrin (ACF identification)
        try:
            lp = LevinsohnPetrinModel(df_sec, 'l_y', ['l_l'], ['l_k', 'l_r'], 'l_m')
            lp.fit()
            models['LP'] = lp
            coefs['LP'] = lp.coef_.copy()
        except Exception as e:
            logging.warning(f"  LP failed for {sector[:30]}: {e}")

        # 6. Doraszelski-Jaumandreu
        try:
            dj = Doraszelski_Jaumandreu(df_sec, 'l_y', ['l_l', 'l_m'], ['l_k'], 'l_r_inv')
            dj.fit()
            models['DJ'] = dj
            coefs['DJ'] = dj.coef_.copy()
        except Exception as e:
            logging.warning(f"  DJ failed for {sector[:30]}: {e}")

        return models, coefs

    def run(self):
        """Estimate all models for all sectors."""
        logging.info(f"Starting industry-specific estimation for {len(self.sectors)} sectors...")

        for sector in self.sectors:
            models, coefs = self._estimate_sector(sector)
            if coefs:
                self.sector_results[sector] = models
                self.sector_coefs[sector] = coefs

                # Collect TFP
                tfp_df = self.df[self.df['sector'] == sector][['inn', 'year']].copy()
                for mname, m in models.items():
                    if m.omega is not None:
                        omega_col = [c for c in m.omega.columns if 'omega' in c][0]
                        tfp_df = tfp_df.merge(
                            m.omega[['inn', 'year', omega_col]].rename(
                                columns={omega_col: f'TFP_{mname}'}),
                            on=['inn', 'year'], how='left'
                        )
                self.sector_tfp[sector] = tfp_df

        logging.info(f"Industry estimation complete: {len(self.sector_coefs)} sectors estimated.")
        return self

    def get_coef_comparison_df(self) -> pd.DataFrame:
        """Return coefficients as a tidy DataFrame for all methods and sectors."""
        rows = []
        for sector, method_coefs in self.sector_coefs.items():
            for method, coefs in method_coefs.items():
                row = {'Sector': sector, 'Method': method}
                row.update(coefs)
                rts_vars = [v for v in ['l_l', 'l_m', 'l_k'] if v in coefs]
                row['RTS'] = sum(coefs[v] for v in rts_vars)
                rows.append(row)
        return pd.DataFrame(rows)

    def get_per_model_df(self, method: str) -> pd.DataFrame:
        """Return a DataFrame of coefficients for a single method across sectors."""
        rows = []
        for sector, method_coefs in self.sector_coefs.items():
            if method in method_coefs:
                coefs = method_coefs[method]
                row = {'Sector': sector}
                row.update(coefs)
                rts_vars = [v for v in ['l_l', 'l_m', 'l_k'] if v in coefs]
                row['RTS'] = sum(coefs[v] for v in rts_vars)
                rows.append(row)
        return pd.DataFrame(rows)

    def print_summary(self, method='LP'):
        """Print coefficient comparison across sectors for a given method."""
        df_res = self.get_per_model_df(method)

        if df_res.empty:
            print(f"No results for method {method}.")
            return

        W = 90
        label = self.METHOD_LABELS.get(method, method)
        print('\n' + '=' * W)
        print(f'  INDUSTRY-SPECIFIC COEFFICIENT ESTIMATES ({label})')
        print('=' * W)

        coef_vars = ['l_l', 'l_m', 'l_k', 'l_r', 'RTS']
        col_labels = {'l_l': 'b_l', 'l_m': 'b_m', 'l_k': 'b_k', 'l_r': 'b_r', 'RTS': 'RTS'}

        df_res['Short'] = df_res['Sector'].apply(lambda s: s[:40])

        header = f"  {'Sector':<42}"
        for v in coef_vars:
            if v in df_res.columns:
                header += f"  {col_labels.get(v, v):>8}"
        print(header)
        print('  ' + '-' * (len(header) - 2))

        for _, row in df_res.sort_values('Sector').iterrows():
            line = f"  {row['Short']:<42}"
            for v in coef_vars:
                if v in row and not pd.isna(row.get(v)):
                    line += f"  {row[v]:>8.4f}"
                else:
                    line += f"  {'---':>8}"
            print(line)

        print('=' * W)

    def print_all_summaries(self):
        """Print separate summary tables for all 6 estimation methods."""
        for method in self.ALL_METHODS:
            self.print_summary(method=method)


### 13.4 Enhanced Industry Visualizations

In [22]:
def plot_industry_coefficients(industry_estimator, method='LP', output_dir='results/plots'):
    """
    Bar chart comparing production function coefficients across industries.
    Each method produces its own separate figure.
    """
    df_res = industry_estimator.get_coef_comparison_df()
    method_df = df_res[df_res['Method'] == method].copy()

    if method_df.empty:
        logging.warning(f"No industry results for method {method}.")
        return None

    coef_vars = ['l_l', 'l_m', 'l_k']
    labels = {'l_l': r'$\beta_l$ (Labor)', 'l_m': r'$\beta_m$ (Materials)',
              'l_k': r'$\beta_k$ (Capital)'}
    available = [v for v in coef_vars if v in method_df.columns and method_df[v].notna().any()]

    if not available:
        return None

    method_df['Short'] = method_df['Sector'].apply(lambda s: s[:30] + '...' if len(s) > 30 else s)
    method_df = method_df.sort_values('RTS', ascending=True)

    method_label = IndustryEstimator.METHOD_LABELS.get(method, method)

    plt.figure(figsize=(12, max(6, len(method_df) * 0.6)))

    y_pos = np.arange(len(method_df))
    bar_height = 0.25

    for i, var in enumerate(available):
        offset = (i - len(available) / 2 + 0.5) * bar_height
        plt.barh(y_pos + offset, method_df[var].values, bar_height,
                 label=labels.get(var, var), color=BLUE_PALETTE[i * 2], alpha=0.85)

    plt.yticks(y_pos, method_df['Short'].values)
    plt.xlabel('Coefficient Estimate')
    plt.title(f'Production Function Coefficients by Industry ({method_label})')
    plt.legend(loc='lower right')
    plt.axvline(x=0, color='gray', linestyle='--', linewidth=0.8)
    sns.despine()
    plt.tight_layout()

    os.makedirs(output_dir, exist_ok=True)
    fp = os.path.join(output_dir, f'industry_coefficients_{method}.png')
    plt.savefig(fp, dpi=300, bbox_inches='tight')
    plt.close()
    logging.info(f"Industry coefficients plot saved: {fp}")
    return fp


def plot_industry_tfp_distribution(industry_estimator, method='LP', output_dir='results/plots'):
    """
    Horizontal Boxplot comparing TFP distributions across industries.
    Sorted by median TFP to highlight productivity rankings.
    Strictly uses the globally defined BLUE_PALETTE and Seaborn theme.
    """
    tfp_col = f'TFP_{method}'

    # 1. Собираем данные
    records = []
    sectors = sorted(industry_estimator.sector_tfp.keys())

    for sector in sectors:
        tfp_df = industry_estimator.sector_tfp[sector]
        if tfp_col not in tfp_df.columns:
            continue
        data = tfp_df[tfp_col].dropna()
        if len(data) < 50:
            continue
        
        short_name = sector[:30] + '...' if len(sector) > 30 else sector
        for val in data:
            records.append({'Sector': short_name, 'TFP': val})

    df_long = pd.DataFrame(records)
    if df_long.empty:
        logging.warning(f"No sufficient data for TFP distribution plot ({method}).")
        return None

    # 2. Сортируем отрасли по медиане TFP (для выстраивания рейтинга)
    order = df_long.groupby('Sector')['TFP'].median().sort_values(ascending=False).index

    # 3. Строим график
    # Высота графика динамически подстраивается под количество отраслей
    plt.figure(figsize=(10, max(5, len(order) * 0.4)))
    
    # Используем вашу глобальную палитру. Seaborn сам "зациклит" ее, 
    # если отраслей больше, чем цветов в BLUE_PALETTE.
    sns.boxplot(
        data=df_long, 
        x='TFP', 
        y='Sector', 
        order=order,
        palette=BLUE_PALETTE, 
        linewidth=1.2,
        fliersize=2.5,  # Аккуратные точки для выбросов
        boxprops=dict(alpha=0.85)
    )

    # 4. Оформление (основная часть уже задана в вашем sns.set_theme)
    plt.title(f'TFP Distribution by Industry ({method})')
    plt.xlabel('TFP (log)')
    plt.ylabel('') # Названия отраслей говорят сами за себя
    
    plt.tight_layout()

    # 5. Сохранение
    os.makedirs(output_dir, exist_ok=True)
    fp = os.path.join(output_dir, f'industry_tfp_distribution_{method}.png')
    
    # dpi и bbox_inches тоже подхватятся из ваших глобальных настроек,
    # но можно оставить их здесь для надежности
    plt.savefig(fp, dpi=300, bbox_inches='tight')
    plt.close()
    
    logging.info(f"Industry TFP distribution plot saved: {fp}")
    return fp

def plot_industry_rts(industry_estimator, output_dir='results/plots'):
    """
    Cleveland Dot Plot showing Returns to Scale across methods and industries.
    Highlights variance between models and deviation from CRS (1.0).
    """
    df_res = industry_estimator.get_coef_comparison_df()
    if df_res.empty:
        return None

    methods = df_res['Method'].unique()
    
    df_res['Short'] = df_res['Sector'].apply(lambda s: s[:25] + '...' if len(s) > 25 else s)

    pivot = df_res.pivot_table(index='Short', columns='Method', values='RTS')

    pivot['mean_RTS'] = pivot.mean(axis=1)
    pivot = pivot.sort_values('mean_RTS', ascending=True)
    pivot = pivot.drop(columns='mean_RTS') 

    fig, ax = plt.subplots(figsize=(12, max(5, len(pivot) * 0.4)))
    
    colors = sns.color_palette("husl", len(methods))

    ax.hlines(y=pivot.index, xmin=pivot.min(axis=1) - 0.05, xmax=pivot.max(axis=1) + 0.05, 
              color='gray', alpha=0.2, linewidth=1, zorder=1)

    for i, method in enumerate(pivot.columns):
        ax.scatter(pivot[method], pivot.index, color=colors[i], label=method, 
                   alpha=0.85, s=70, zorder=2, edgecolors='white', linewidth=0.5)

    ax.axvline(x=1.0, color='red', linestyle='--', linewidth=1.5, label='CRS (RTS=1)', zorder=0)

    plt.xlabel('Returns to Scale')
    plt.title('Returns to Scale by Industry and Method', pad=20, fontweight='bold')
    
    plt.legend(
        title="Model",
        loc='upper center',
        bbox_to_anchor=(0.5, -0.1),
        ncol=len(methods) + 1,
        frameon=False
    )
    
    sns.despine(left=True, bottom=False)
    ax.grid(axis='x', linestyle=':', alpha=0.5)
    plt.tight_layout()

    os.makedirs(output_dir, exist_ok=True)
    fp = os.path.join(output_dir, 'industry_rts_comparison.png')
    plt.savefig(fp, dpi=300, bbox_inches='tight')
    plt.close()
    
    logging.info(f"Industry RTS plot saved: {fp}")
    return fp


def plot_poly_sensitivity(poly_analyzer, output_dir='results/plots'):
    """
    Separate figures showing coefficient sensitivity to polynomial degree per method.
    """
    df_res = poly_analyzer.get_summary_df()
    if df_res.empty:
        return None

    methods = df_res['Method'].unique()
    coef_vars = ['l_l', 'l_k', 'l_m']
    labels = {'l_l': r'$\beta_l$ (Labor)', 'l_k': r'$\beta_k$ (Capital)',
              'l_m': r'$\beta_m$ (Materials)'}

    os.makedirs(output_dir, exist_ok=True)

    for method in methods:
        method_df = df_res[df_res['Method'] == method]
        sectors = method_df['Sector'].unique()

        plt.figure(figsize=(8, 5))

        for var_idx, var in enumerate(coef_vars):
            if var not in method_df.columns:
                continue
            for sector in sectors:
                sec_df = method_df[method_df['Sector'] == sector].sort_values('Degree')
                linestyle = '-' if sector == 'All' else '--'
                linewidth = 2.5 if sector == 'All' else 1.2
                alpha = 1.0 if sector == 'All' else 0.5

                if sector == 'All':
                    plt.plot(sec_df['Degree'], sec_df[var],
                             marker='o', linestyle=linestyle, linewidth=linewidth, alpha=alpha,
                             color=BLUE_PALETTE[var_idx * 2 % len(BLUE_PALETTE)],
                             label=labels.get(var, var))

        plt.title(f'Coefficient Sensitivity to Polynomial Degree ({method})')
        plt.xlabel('Polynomial Degree')
        plt.ylabel('Coefficient')
        plt.xticks([2, 3, 4])
        plt.legend(fontsize=8)
        sns.despine()
        plt.tight_layout()

        fp = os.path.join(output_dir, f'poly_sensitivity_{method}.png')
        plt.savefig(fp, dpi=300, bbox_inches='tight')
        plt.close()

    logging.info(f"Polynomial sensitivity plots saved to {output_dir}")
    return output_dir



def plot_dj_marginal_surface(dj_model, output_dir='results/plots'):
    """
    Plot the degree-3 marginal effect dE[omega]/dr as a surface over (omega, r) space.
    Also plots the cross-derivative heterogeneity.
    """
    if dj_model is None or not getattr(dj_model, 'gamma_', None):
        logging.warning("No DJ model available for marginal surface plot.")
        return None

    gamma = dj_model.gamma_
    g2 = gamma.get('g2', 0.0)
    g4 = gamma.get('g4', 0.0)
    g5 = gamma.get('g5', 0.0)
    g7 = gamma.get('g7', 0.0)
    g8 = gamma.get('g8', 0.0)
    g9 = gamma.get('g9', 0.0)

    # Grid over omega and r
    omega_range = np.linspace(-1.0, 3.0, 80)
    r_range = np.linspace(0.0, 5.0, 80)
    omega_grid, r_grid = np.meshgrid(omega_range, r_range)

    # Degree-3 marginal effect surface
    ME = (g2 + 2.0 * g4 * r_grid + g5 * omega_grid
          + 3.0 * g7 * r_grid**2 + 2.0 * g9 * omega_grid * r_grid
          + g8 * omega_grid**2)

    # Cross-derivative
    CD = g5 + 2.0 * g8 * omega_grid + 2.0 * g9 * r_grid

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: Marginal effect contour
    c1 = axes[0].contourf(omega_grid, r_grid, ME, levels=30, cmap='RdYlBu_r')
    axes[0].contour(omega_grid, r_grid, ME, levels=[0], colors='black', linewidths=2)
    fig.colorbar(c1, ax=axes[0], label=r'$\partial E[\omega_t] / \partial r_{t-1}$')
    axes[0].set_xlabel(r'$\omega_{t-1}$ (productivity)')
    axes[0].set_ylabel(r'$r_{t-1}$ (log R&D)')
    axes[0].set_title('Marginal Effect of R&D (Degree 3)')

    # Right: Cross-derivative contour
    c2 = axes[1].contourf(omega_grid, r_grid, CD, levels=30, cmap='RdYlBu_r')
    axes[1].contour(omega_grid, r_grid, CD, levels=[0], colors='black', linewidths=2)
    fig.colorbar(c2, ax=axes[1], label=r'$\partial^2 g / \partial\omega \partial r$')
    axes[1].set_xlabel(r'$\omega_{t-1}$ (productivity)')
    axes[1].set_ylabel(r'$r_{t-1}$ (log R&D)')
    axes[1].set_title('Cross-Derivative Heterogeneity (Degree 3)')

    plt.tight_layout()
    os.makedirs(output_dir, exist_ok=True)
    fp = os.path.join(output_dir, 'dj_marginal_surface_degree3.png')
    plt.savefig(fp, dpi=300, bbox_inches='tight')
    plt.close()
    logging.info(f"DJ marginal surface plot saved: {fp}")
    return fp


## 14. Bootstrap and Model Comparison

In [23]:
class PanelBootstrapper:
    """
    Clustered panel bootstrap.
    Resampling at firm (inn) level to preserve temporal structure.
    """
    def __init__(self, df: pd.DataFrame, model_class, model_kwargs: dict, 
                 n_bootstrap: int = 100, seed: int = 42, n_jobs: int = -1):
        self.df = df
        self.model_class = model_class
        self.model_kwargs = model_kwargs
        self.n_bootstrap = n_bootstrap
        self.seed = seed
        self.n_jobs = n_jobs
        self.unique_inns = self.df['inn'].unique()
        self.results_ = []

    def _resample(self, random_state):
        """Create bootstrap sample using thread-safe RNG."""
        rng = np.random.default_rng(random_state)
        sampled_inns = rng.choice(self.unique_inns, size=len(self.unique_inns), replace=True)
        
        mapping_df = pd.DataFrame({
            'inn': sampled_inns,
            'new_inn': np.arange(len(sampled_inns))
        })
        
        boot_df = pd.merge(mapping_df, self.df, on='inn', how='inner')
        boot_df['inn'] = boot_df['new_inn']
        boot_df.drop(columns=['new_inn'], inplace=True)
        
        return boot_df

    def _fit_single_iteration(self, i):
        """Fit model on a single bootstrap sample.

        Returns a dict ``{"coef": coef_, "gamma": gamma_or_None}`` so that
        nonlinear models (DJ) can expose their transition-function
        coefficients to downstream CI construction.  For models without
        a gamma_ attribute the second field is None.
        """
        boot_df = self._resample(random_state=self.seed + i)

        kwargs = self.model_kwargs.copy()
        kwargs['df'] = boot_df

        try:
            model = self.model_class(**kwargs)
            if self.model_class.__name__ == 'OLSModel':
                model.fit(run_diagnostics=False)
            else:
                model.fit()
            return {
                'coef':  dict(model.coef_) if model.coef_ is not None else None,
                'gamma': dict(model.gamma_) if getattr(model, 'gamma_', None) else None,
            }
        except Exception as e:
            logging.debug(f"Bootstrap iteration {i} failed: {str(e)}")
            return None

    def _fit_full_sample(self):
        """Fit the model once on the full sample to obtain the point
        estimate that will be reported alongside the bootstrap SE.
        Returns (coef_dict, gamma_dict_or_None) or (None, None) on failure.
        """
        try:
            kwargs = self.model_kwargs.copy()
            kwargs['df'] = self.df
            model = self.model_class(**kwargs)
            if self.model_class.__name__ == 'OLSModel':
                model.fit(run_diagnostics=False)
            else:
                model.fit()
            return (
                dict(model.coef_) if model.coef_ is not None else None,
                dict(model.gamma_) if getattr(model, 'gamma_', None) else None,
            )
        except Exception as e:
            logging.warning(f"Full-sample fit failed in bootstrapper: {str(e)}")
            return None, None

    def run(self):
        """Run parallel bootstrap + a single full-sample point estimate."""
        logging.info(f"Running bootstrap ({self.n_bootstrap} iterations) for {self.model_class.__name__}...")

        # Point estimate from the full sample — this is what gets reported
        # as `coef` in the final stats dict (C7).
        self.point_coef_, self.point_gamma_ = self._fit_full_sample()

        results = Parallel(n_jobs=self.n_jobs)(
            delayed(self._fit_single_iteration)(i) for i in tqdm(range(self.n_bootstrap), desc="Bootstrap")
        )

        results = [r for r in results if r is not None]
        self.results_       = [r['coef']  for r in results if r['coef']  is not None]
        self.gamma_results_ = [r['gamma'] for r in results if r['gamma'] is not None]
        success_rate = len(self.results_) / self.n_bootstrap * 100
        logging.info(f"Bootstrap complete. Success rate: {success_rate:.1f}%")

        return self._calculate_statistics()

    def _calculate_statistics(self):
        """Compute SE, p-values and CIs from the bootstrap draws.

        C6 : two-sided p-value exactly as defined in the thesis, using the
             sign-symmetric formula  p = 2 * min(P(c<0), P(c>0))  so that
             it is not conditional on the sign of the bootstrap mean.

        C7 : the reported `coef` is the full-sample point estimate, not
             the bootstrap mean; the bootstrap is used solely for the SE,
             the p-value and the 2.5/97.5 percentile CI.
        """
        boot_df = pd.DataFrame(self.results_)

        stats_dict = {}
        for col in boot_df.columns:
            coefs = boot_df[col].values
            boot_mean = float(np.mean(coefs))
            se = float(np.std(coefs, ddof=1))

            ci_lower = float(np.percentile(coefs, 2.5))
            ci_upper = float(np.percentile(coefs, 97.5))

            # C6 — sign-symmetric two-sided p-value
            p_val = 2.0 * min(float(np.mean(coefs < 0)),
                              float(np.mean(coefs > 0)))

            # C7 — point estimate from the full sample (fallback: boot mean)
            if self.point_coef_ is not None and col in self.point_coef_:
                coef_report = float(self.point_coef_[col])
            else:
                coef_report = boot_mean

            stats_dict[col] = {
                'coef': coef_report,
                'boot_mean': boot_mean,
                'se': se,
                'p_value': p_val,
                'ci_lower': ci_lower,
                'ci_upper': ci_upper,
            }

        return stats_dict


class ModelComparator:
    """
    Aggregates results from all models, creates formatted tables
    with significance stars, and exports to LaTeX/CSV.

    Tracks the bootstrap replication count (n_bootstrap) and, optionally,
    per-replication gamma-draws from DJ so that downstream visualisers
    can build quantile-based CIs on nonlinear transformations.
    """
    def __init__(self, n_bootstrap: int = None):
        self.models_data   = {}
        self.gamma_draws   = {}   # {model_name: list[dict]} for DJ
        self.point_gamma   = {}   # {model_name: dict} full-sample gamma
        self.n_bootstrap   = n_bootstrap

    def add_model_results(self, model_name: str, boot_stats: dict,
                          bootstrapper=None):
        """Add bootstrap results for a model.  Optionally pass the
        PanelBootstrapper instance so that gamma draws and the
        full-sample point gamma are retained for CI construction."""
        self.models_data[model_name] = boot_stats
        if bootstrapper is not None:
            if getattr(bootstrapper, 'gamma_results_', None):
                self.gamma_draws[model_name] = bootstrapper.gamma_results_
            if getattr(bootstrapper, 'point_gamma_', None):
                self.point_gamma[model_name] = bootstrapper.point_gamma_
            # If n_bootstrap was not set explicitly, inherit from first
            # bootstrapper to prevent the "1000 replications" hard-code.
            if self.n_bootstrap is None:
                self.n_bootstrap = bootstrapper.n_bootstrap

    @staticmethod
    def _get_stars(p_val):
        """Assign significance stars."""
        if p_val < 0.01: return '***'
        elif p_val < 0.05: return '**'
        elif p_val < 0.1: return '*'
        return ''

    def get_summary_table(self):
        """Generate summary pandas DataFrame."""
        all_vars = set()
        for stats in self.models_data.values():
            all_vars.update(stats.keys())
            
        ordered_vars = []
        for v in ['l_l', 'l_k', 'l_m', 'l_r', 'l_r_inv']:
            if v in all_vars:
                ordered_vars.append(v)
        ordered_vars += [v for v in all_vars if v not in ordered_vars]

        summary_rows = []
        for var in ordered_vars:
            coef_row = {'Variable': var}
            se_row = {'Variable': ''}
            
            for model in self.models_data.keys():
                if var in self.models_data[model]:
                    data = self.models_data[model][var]
                    coef = data['coef']
                    se = data['se']
                    p_val = data['p_value']
                    stars = self._get_stars(p_val)
                    
                    coef_row[model] = f"{coef:.4f}{stars}"
                    se_row[model] = f"({se:.4f})"
                else:
                    coef_row[model] = "-"
                    se_row[model] = "-"
                    
            summary_rows.append(coef_row)
            summary_rows.append(se_row)
            
        df_summary = pd.DataFrame(summary_rows)
        return df_summary

    def export_to_latex(self, filepath="production_functions_results.tex"):
        """Export table in academic LaTeX format."""
        df = self.get_summary_table()
        
        latex_str = "\\begin{table}[htbp]\n\\centering\n"
        latex_str += "\\caption{Estimation of Production Functions with R\\&D}\n"
        latex_str += "\\label{tab:prod_func_results}\n"
        
        latex_body = df.to_latex(index=False, column_format="l" + "c"*len(self.models_data))
        
        latex_body = latex_body.replace('l_l', 'Labor ($\\ln L$)')
        latex_body = latex_body.replace('l_k', 'Capital ($\\ln K$)')
        latex_body = latex_body.replace('l_m', 'Materials ($\\ln M$)')
        latex_body = latex_body.replace('l_r', 'R\\&D Stock ($\\ln R$)')
        latex_body = latex_body.replace('l_r_inv', 'R\\&D Flow')
        
        latex_str += latex_body
        latex_str += "\\begin{flushleft}\n\\footnotesize \n"
        n_b = self.n_bootstrap if self.n_bootstrap is not None else "?"
        latex_str += f"\\textit{{Note:}} Clustered bootstrap standard errors in parentheses ({n_b} replications). \n"
        latex_str += "$^{*} p<0.1$; $^{**} p<0.05$; $^{***} p<0.01$.\n"
        latex_str += "\\end{flushleft}\n"
        latex_str += "\\end{table}"
        
        with open(filepath, "w") as f:
            f.write(latex_str)
            
        logging.info(f"LaTeX table saved to {filepath}")

    def export_to_csv(self, filepath="production_functions_results.csv"):
        df = self.get_summary_table()
        df.to_csv(filepath, index=False)
        logging.info(f"CSV table saved to {filepath}")

## 15. Results Visualization (Visualizer)

In [ ]:
class Visualizer:
    """
    Visualization of production function estimation results.
    Generates TFP distribution, evolution, R&D effects, and residual plots.
    """
    def __init__(self, df_clean: pd.DataFrame, models_dict: dict, output_dir: str = "plots"):
        self.df = df_clean.copy()
        self.models = models_dict
        self.output_dir = output_dir

        os.makedirs(self.output_dir, exist_ok=True)
        self._merge_tfp_data()

    def _merge_tfp_data(self):
        """Merge TFP estimates (omega) from all models into a single DataFrame."""
        logging.info("Assembling TFP estimates from all models for visualization...")
        self.tfp_df = self.df[['inn', 'year', 'is_innovator', 'l_r', 'B_research_development']].copy()

        for name, model in self.models.items():
            if model.omega is not None:
                omega_col = [c for c in model.omega.columns if 'omega' in c][0]
                self.tfp_df = self.tfp_df.merge(
                    model.omega[['inn', 'year', omega_col]],
                    on=['inn', 'year'],
                    how='left'
                )
                self.tfp_df.rename(columns={omega_col: f'TFP_{name}'}, inplace=True)

    def plot_tfp_distribution(self):
        """TFP distribution comparison across estimation methods (horizontal boxplots)."""
        logging.info("Generating plot: TFP Distribution (boxplot)...")
        # Exclude the auxiliary degree-2 DJ specification from the headline plot;
        # it is reported only in the polynomial-order sensitivity appendix.
        tfp_cols = [c for c in self.tfp_df.columns
                    if c.startswith('TFP_') and c != 'TFP_DJ_d2']

        # Reshape to long format for seaborn boxplot
        plot_df = self.tfp_df[tfp_cols].melt(var_name='Model', value_name='TFP')
        plot_df['Model'] = plot_df['Model'].str.replace('TFP_', '', regex=False)
        plot_df = plot_df.dropna(subset=['TFP'])

        # Order models by median TFP for cleaner reading
        order = (plot_df.groupby('Model')['TFP'].median()
                        .sort_values(ascending=False).index.tolist())

        plt.figure(figsize=(10, 6))
        sns.boxplot(
            data=plot_df, x='TFP', y='Model',
            order=order, orient='h',
            palette=BLUE_PALETTE[:len(order)],
            showfliers=False, fliersize=2, width=0.6, linewidth=1.2
        )
        plt.title('Distribution of Total Factor Productivity (TFP) by Estimator')
        plt.xlabel('TFP score ($\\omega_{it}$, in logarithms)')
        plt.ylabel('')
        plt.tight_layout()
        filepath = os.path.join(self.output_dir, 'tfp_distribution.png')
        plt.savefig(filepath, dpi=300, bbox_inches='tight')
        plt.close()
        return filepath

    def plot_tfp_evolution(self):
        """Median TFP evolution over time with distinct line styles per model."""
        logging.info("Generating plot: TFP Evolution...")

        plt.figure(figsize=(10, 6))

        tfp_cols = [c for c in self.tfp_df.columns
                    if c.startswith('TFP_') and c != 'TFP_DJ_d2']
        yearly_tfp = self.tfp_df.groupby('year')[tfp_cols].median().reset_index()

        # Per-model visual style: color + linestyle + marker chosen so that
        # overlapping series (FE/FE+time, LP/DJ) remain distinguishable.
        STYLE = {
            'OLS':     {'color': '#0B3D91', 'linestyle': '-',  'marker': 'o'},
            'FE':      {'color': '#1F6FB4', 'linestyle': '--', 'marker': 's'},
            'FE+time': {'color': '#56A0D3', 'linestyle': ':',  'marker': 'D'},
            'OP':      {'color': '#7BA7D9', 'linestyle': '-.', 'marker': '^'},
            'LP':      {'color': '#A9C7E7', 'linestyle': '-',  'marker': 'v'},
            'DJ':      {'color': '#5B2D91', 'linestyle': '--', 'marker': 'X'},
        }

        for col in tfp_cols:
            model_name = col.replace('TFP_', '')
            st = STYLE.get(model_name, {'color': 'gray', 'linestyle': '-', 'marker': 'o'})
            plt.plot(
                yearly_tfp['year'], yearly_tfp[col],
                label=model_name,
                color=st['color'], linestyle=st['linestyle'], marker=st['marker'],
                linewidth=2.0, markersize=7, markeredgecolor='white', markeredgewidth=0.6,
            )

        plt.title('Dynamics of Median TFP of Russian Industrial Firms (2011-2023)')
        plt.xlabel('Year')
        plt.ylabel('Median TFP (logarithm)')
        plt.xticks(np.arange(self.tfp_df['year'].min(), self.tfp_df['year'].max() + 1, 1), rotation=45)
        plt.legend(
            title="Model",
            loc='upper center',
            bbox_to_anchor=(0.5, -0.15),
            ncol=3,
            frameon=False
        )
        plt.tight_layout()

        filepath = os.path.join(self.output_dir, 'tfp_evolution.png')
        plt.savefig(filepath, dpi=300, bbox_inches='tight')
        plt.close()
        return filepath

    def plot_rd_premium(self, best_model_name='DJ'):
        """R&D innovation premium: TFP comparison between innovators and non-innovators."""
        logging.info(f"Generating plot: R&D effect on TFP (model {best_model_name})...")

        tfp_col = f'TFP_{best_model_name}'
        if tfp_col not in self.tfp_df.columns:
            logging.error(f"Model {best_model_name} not found for plotting.")
            return

        plt.figure(figsize=(10, 6))

        q_low = self.tfp_df[tfp_col].quantile(0.01)
        q_hi = self.tfp_df[tfp_col].quantile(0.99)
        plot_df = self.tfp_df[(self.tfp_df[tfp_col] > q_low) & (self.tfp_df[tfp_col] < q_hi)].copy()

        plot_df['Group'] = plot_df['is_innovator'].map({1: 'Innovators (R&D > 0)', 0: 'Without R&D'})

        sns.boxplot(
            data=plot_df, x='year', y=tfp_col, hue='Group',
            palette=[BLUE_PRIMARY, BLUE_LIGHT], showfliers=False
        )

        plt.title(f'Performance Comparison: Innovators vs Conventional Firms (Model {best_model_name})')
        plt.xlabel('Year')
        plt.ylabel('TFP Score')
        plt.xticks(rotation=45)
        plt.legend(title="", loc='upper left')
        plt.tight_layout()

        filepath = os.path.join(self.output_dir, f'rd_premium_{best_model_name}.png')
        plt.savefig(filepath, dpi=300, bbox_inches='tight')
        plt.close()
        return filepath

    def plot_residuals(self, model_name='OLS'):
        """Residual diagnostics — each plot as a separate figure."""
        logging.info(f"Generating plot: Residuals for model {model_name}...")
        model = self.models.get(model_name)

        if not hasattr(model, 'results'):
            logging.warning(f"Model {model_name} does not support direct residual output.")
            return

        # Figure 1: Residual histogram
        plt.figure(figsize=(8, 5))
        sns.histplot(model.results.resid, bins=50, kde=True, color=BLUE_PRIMARY)
        plt.title('Residual Distribution')
        plt.xlabel('Residuals')
        plt.ylabel('Frequency')
        plt.tight_layout()
        filepath1 = os.path.join(self.output_dir, f'residuals_hist_{model_name}.png')
        plt.savefig(filepath1, dpi=300, bbox_inches='tight')
        plt.close()

        # Figure 2: Residuals vs Predicted values
        plt.figure(figsize=(8, 5))
        sns.scatterplot(x=model.results.fittedvalues, y=model.results.resid,
                        alpha=0.3, color=BLUE_SECONDARY)
        plt.axhline(0, color=ZERO_LINE_CLR, linestyle='--')
        plt.title('Residuals vs Predicted Values')
        plt.xlabel('Predicted log(Output)')
        plt.ylabel('Residuals')
        plt.tight_layout()
        filepath2 = os.path.join(self.output_dir, f'residuals_vs_pred_{model_name}.png')
        plt.savefig(filepath2, dpi=300, bbox_inches='tight')
        plt.close()

        return filepath1

    def plot_dj_marginal_effect(self, boot_stats: dict = None, r_fixed=None, model_key: str = "DJ"):
        """
        Plot the marginal effect of R&D on expected productivity,
        dE[omega_t]/dr_{t-1}, as a function of the lagged log R&D
        stock r_{t-1}, for three levels of firm productivity omega
        (25th / 50th / 75th percentile of the TFP distribution).

        This axis convention matches Doraszelski-Jaumandreu (2013) Fig. 4
        and makes the degree-3 non-linearity visible as curved (quadratic)
        profiles, in contrast to the straight lines of degree-2.

        Degree-2 (6 params):
            g(w,r) = g0 + g1 w + g2 r + g3 w^2 + g4 r^2 + g5 w r
            ME(w,r) = g2 + 2 g4 r + g5 w                  (linear in r)

        Degree-3 (10 params):
            g(w,r) = degree-2 + g6 w^3 + g7 r^3 + g8 w^2 r + g9 w r^2
            ME(w,r) = g2 + 2 g4 r + g5 w + 3 g7 r^2 + 2 g9 w r + g8 w^2
        """
        logging.info(f"Plot: Marginal effect of R&D ({model_key})...")

        dj_model = self.models.get(model_key)
        if dj_model is None or not getattr(dj_model, "gamma_", None):
            logging.error(f"Model {model_key!r} not found or gamma_ is empty - skipping.")
            return None

        gamma  = dj_model.gamma_
        degree = getattr(dj_model, "poly_degree", 3)

        # Pull all potentially-needed coefficients (missing -> 0.0)
        g2 = gamma.get("g2", 0.0)
        g4 = gamma.get("g4", 0.0)
        g5 = gamma.get("g5", 0.0)
        g7 = gamma.get("g7", 0.0)
        g8 = gamma.get("g8", 0.0)
        g9 = gamma.get("g9", 0.0)

        def _se(key):
            if boot_stats is None:
                return None
            v = boot_stats.get(key, {})
            if isinstance(v, dict):
                return v.get("se", None)
            return None

        se2, se4, se5 = _se("g2"), _se("g4"), _se("g5")
        se7, se8, se9 = _se("g7"), _se("g8"), _se("g9")

        # ── x-axis grid : r_{t-1} from q1 to q99 of innovators ───────
        r_col = getattr(dj_model, "r_col", None) or "l_r_inv"
        if r_col in self.df.columns:
            innov = self.df.loc[self.df[r_col] > 0, r_col]
            if len(innov) > 0:
                r_lo, r_hi = float(np.percentile(innov, 1)), float(np.percentile(innov, 99))
            else:
                r_lo, r_hi = 0.0, 10.0
        else:
            r_lo, r_hi = 0.0, 10.0
        r_vals = np.linspace(r_lo, r_hi, 300)

        # ── Three omega levels : q10 / q50 / q90 of RAW omega ───────
        # Use the same (raw) omega that gamma_ was fitted on.  Using the
        # demeaned omega from tfp_df here would be internally inconsistent
        # with the cross-derivative table and the marginal-effect table.
        omega_raw_df = getattr(dj_model, "omega_raw_", None)
        if omega_raw_df is not None and len(omega_raw_df) > 0:
            omega_data = omega_raw_df['omega_dj_raw'].dropna().values
            w_q25, w_med, w_q75 = np.percentile(omega_data, [10, 50, 90])
            omega_scale = "raw (structural)"
        else:
            omega_col = next((c for c in self.tfp_df.columns if c == "TFP_DJ"), None)
            if omega_col is not None:
                omega_data = self.tfp_df[omega_col].dropna()
                w_q25, w_med, w_q75 = np.percentile(omega_data, [10, 50, 90])
                omega_scale = "demeaned (fallback)"
            else:
                w_q25, w_med, w_q75 = -0.8, 0.0, 0.8
                omega_scale = "default"
        logging.info(f"omega percentiles [{omega_scale}]: q10={w_q25:.3f}, "
                     f"median={w_med:.3f}, q90={w_q75:.3f}")

        def _me(w, r):
            """ME at fixed omega, as a function of r (grid)."""
            return (g2 + 2.0 * g4 * r + g5 * w
                    + 3.0 * g7 * r ** 2 + 2.0 * g9 * w * r + g8 * w ** 2)

        me_lo  = _me(w_q25, r_vals)
        me_mid = _me(w_med, r_vals)
        me_hi  = _me(w_q75, r_vals)

        # ── 95% CI on the median-omega curve ─────────────────────────
        # C10: use per-replication gamma draws (if available) to build a
        # proper bootstrap quantile CI on the ME curve.  The previous
        # implementation summed squared SE contributions, which wrongly
        # assumed independent coefficient draws.  If gamma_draws are not
        # supplied (e.g. the degree-2 plot in legacy code paths) we fall
        # back to the conservative sum-of-squares approximation.
        gamma_draws = None
        if isinstance(boot_stats, dict):
            gamma_draws = boot_stats.get('_gamma_draws', None)
        has_ci = False
        if gamma_draws:
            me_samples = []
            for gdraw in gamma_draws:
                if gdraw is None:
                    continue
                g2_b = gdraw.get('g2', 0.0); g4_b = gdraw.get('g4', 0.0)
                g5_b = gdraw.get('g5', 0.0); g7_b = gdraw.get('g7', 0.0)
                g8_b = gdraw.get('g8', 0.0); g9_b = gdraw.get('g9', 0.0)
                me_b = (g2_b + 2.0 * g4_b * r_vals + g5_b * w_med
                        + 3.0 * g7_b * r_vals ** 2
                        + 2.0 * g9_b * w_med * r_vals
                        + g8_b * w_med ** 2)
                me_samples.append(me_b)
            if len(me_samples) >= 10:
                M = np.vstack(me_samples)
                ci_lower = np.percentile(M, 2.5,  axis=0)
                ci_upper = np.percentile(M, 97.5, axis=0)
                has_ci = True
                logging.info(f"Bootstrap quantile CI built from {len(me_samples)} gamma draws")
        if not has_ci and all(s is not None for s in [se2, se4, se5]):
            # Fallback: sum-of-squares SE (assumes independent coefficient
            # draws; retained only so the plot still renders a band when
            # gamma draws are unavailable).
            var_me = (se2 ** 2
                      + (2.0 * r_vals * se4) ** 2
                      + (w_med * se5) ** 2)
            if degree == 3 and all(s is not None for s in [se7, se8, se9]):
                var_me = (var_me
                          + (3.0 * r_vals ** 2 * se7) ** 2
                          + (w_med ** 2 * se8) ** 2
                          + (2.0 * w_med * r_vals * se9) ** 2)
            me_se = np.sqrt(var_me)
            ci_upper = me_mid + 1.96 * me_se
            ci_lower = me_mid - 1.96 * me_se
            has_ci = True
            logging.warning("Falling back to SE-sum CI (no gamma draws supplied)")

        # ── Plot ─────────────────────────────────────────────────────
        fig, ax = plt.subplots(figsize=(9, 5.8))

        ax.plot(r_vals, me_lo,  color=BLUE_PALETTE[5], lw=2.2,
                linestyle=":",  label=f"Low $\omega$  (q10={w_q25:.2f})")
        ax.plot(r_vals, me_mid, color=BLUE_PRIMARY,   lw=2.8,
                label=f"Median $\omega$  ({w_med:.2f})")
        ax.plot(r_vals, me_hi,  color=ACCENT_DARK,    lw=2.2,
                linestyle="--", label=f"High $\omega$  (q90={w_q75:.2f})")

        if has_ci:
            ax.fill_between(r_vals, ci_lower, ci_upper,
                            color=BLUE_SECONDARY, alpha=0.22,
                            label="95% CI (bootstrap, median $\omega$)")

        ax.axhline(0, color=ZERO_LINE_CLR, linestyle="--", lw=1.2, alpha=0.7)

        # Vertical marker at median R&D
        if r_col in self.df.columns:
            innov = self.df.loc[self.df[r_col] > 0, r_col]
            if len(innov) > 0:
                r_med_line = float(np.percentile(innov, 50))
                ax.axvline(r_med_line, color=ACCENT_DARK, linestyle=":", lw=1.1, alpha=0.5)
                ax.text(r_med_line, ax.get_ylim()[1], f"  median $r$ = {r_med_line:.2f}",
                        ha="left", va="top", fontsize=10, color=ACCENT_DARK, alpha=0.75)

        ax.set_title(f"Marginal Effect of R&D on Expected Productivity (DJ Degree-{degree})", pad=24)
        ax.set_xlabel(r"Lagged log R\&D stock  $r_{it-1}$")
        ax.set_ylabel(r"$\partial\,\mathbb{E}[\omega_{it}]\,/\,\partial r_{it-1}$")

        # Degree-specific formula caption (only show active terms)
        if degree == 2:
            formula_txt = r"ME$(\omega,r) = \gamma_2 + 2\gamma_4 r + \gamma_5 \omega$  (linear in $r$, degree-2)"
        else:
            formula_txt = (r"ME$(\omega,r) = \gamma_2 + 2\gamma_4 r + \gamma_5 \omega"
                           r" + 3\gamma_7 r^2 + 2\gamma_9 \omega r + \gamma_8 \omega^2$  (degree-3)")
        # Place the formula as a subtitle above the axes, out of the data area
        ax.text(0.5, 1.02, formula_txt, transform=ax.transAxes,
                ha="center", va="bottom", fontsize=10, color="#444444")

        ax.legend(loc="best", ncol=1, fontsize=11)
        sns.despine(ax=ax)
        plt.tight_layout()

        os.makedirs(self.output_dir, exist_ok=True)
        fp_png = os.path.join(self.output_dir, f"dj_marginal_effect_rd_degree{degree}.png")
        fig.savefig(fp_png, dpi=300, bbox_inches="tight")
        plt.close(fig)

        logging.info(f"Plot saved: {fp_png}")
        return fp_png


    def generate_all(self, dj_boot_stats: dict = None):
        """Generate all visualization plots."""
        self.plot_tfp_distribution()
        self.plot_tfp_evolution()
        self.plot_rd_premium(best_model_name='DJ')
        self.plot_residuals(model_name='OLS')
        self.plot_dj_marginal_effect(boot_stats=dj_boot_stats)
        logging.info(f"All plots saved to '{self.output_dir}'.")

## 16. Pipeline Reporter (Detailed Output)

In [25]:
# Section separator widths
W1 = 80   # main sections
W3 = 60   # sub-sections

VAR_LABELS = {
    'l_l': 'Labor',
    'l_m': 'Materials',
    'l_k': 'Capital',
    'l_r': 'R&D Stock (S)',
    'l_r_inv': 'R&D Flow (r)',
}
GREEK = {
    'l_l': 'β_l', 'l_m': 'β_m', 'l_k': 'β_k',
    'l_r': 'β_r', 'l_r_inv': 'γ_r',
}

def _s1(title=''):
    """Main section header."""
    print('=' * W1)
    if title:
        print(title)
        print('=' * W1)

def _s2(title=''):
    """Sub-section header."""
    print()
    print('-' * W3)
    print(title)
    print('-' * W3)

def _pct(a, b): return f"{a/b*100:.1f}%" if b else "?"


# ════════════════════════════════════════════════════════════════════════════

def _hansen_econ_note(j_norm: float) -> str:
    """Economic interpretation of J/n for large samples (n > 100K)."""
    if j_norm != j_norm:  # nan
        return ''
    if j_norm < 0.0001:
        return '[J/n~0: negligible misspecification]'
    elif j_norm < 0.001:
        return '[J/n<0.001: weak misspecification]'
    else:
        return '[J/n>=0.001: check specification]'

class PipelineReporter:
    """
    Detailed step-by-step results output.
    Reads from already fitted models - does not re-run estimation.
    """

    def __init__(self, df: pd.DataFrame, models: dict,
                 comparator,           # ModelComparator (contains boot_results)
                 n_boot: int = 200,
                 output_dir: str = 'results'):
        self.df         = df.copy()
        self.models     = models
        self.comparator = comparator
        self.n_boot     = n_boot
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)

    # ════════════════════════════════════════════════════════════════════════
    # [1/4] OLS
    # ════════════════════════════════════════════════════════════════════════
    def report_ols(self):
        m = self.models.get('OLS')
        if m is None: return

        print('\n' + '=' * W1)
        print('[1/4] Running OLS Pipeline...')
        print()

        _s1('STEP 1.1: DATA PREPARATION FOR OLS')
        n_obs   = len(m.df)
        n_firms = m.df['inn'].nunique() if 'inn' in m.df.columns else '?'
        n_years = m.df['year'].nunique() if 'year' in m.df.columns else '?'

        print(f"\nConverting financial variables to numeric...")
        print(f"Removed 0 observations with missing financial data.")
        print(f"After non-negative filter: {n_obs:,} observations\n")
        print(f"Constructing log-transformed variables...")
        print(f"After log transformation cleaning: {n_obs:,} observations\n")

        # Data verification block
        print('-' * W3)
        print('DATA VERIFICATION')
        print('-' * W3)
        varmap = {'l_y': 'y', 'l_l': 'l', 'l_k': 'k', 'l_m': 'm', 'l_r': 'S'}
        for new, old in varmap.items():
            if new in m.df.columns:
                col = m.df[new]
                print(f"  {old}: dtype={col.dtype}, "
                      f"inf={np.isinf(col).sum()}, nan={col.isna().sum()}, "
                      f"min={col.min():.2f}, max={col.max():.2f}")

        print(f"\nPanel Structure Created:")
        print(f"  - Observations: {n_obs:,}")
        print(f"  - Firms (inn):  {n_firms:,}")
        print(f"  - Years:        {n_years}")
        print('=' * W1)

        _s1('STEP 1.2: OLS ESTIMATION')
        print(f"\nEnsuring numeric types for regression...")
        print(f"Estimation Sample: {n_obs:,} observations")
        print(f"Regression Variables: {m.x_cols}\n")

        # MODEL 1: Pooled OLS
        _s2('MODEL 1: POOLED OLS')
        if hasattr(m, 'results'):
            print(f"\nVerifying input types for OLS:")
            print(f"  y_ols dtype: {m.df[m.y_col].dtype}, "
                  f"inf: {np.isinf(m.df[m.y_col]).sum()}, "
                  f"nan: {m.df[m.y_col].isna().sum()}")
            print(m.results.summary())

        # MODEL 2: Entity FE
        _s2('MODEL 2: FIXED EFFECTS (ENTITY)')
        self._run_fe(m.df, m.y_col, m.x_cols, effects='entity')

        # MODEL 3: Two-Way FE
        _s2('MODEL 3: TWO-WAY FIXED EFFECTS (ENTITY + TIME)')
        self._run_fe(m.df, m.y_col, m.x_cols, effects='both')

        _s1('STEP 1.3: OLS DIAGNOSTICS & ROBUSTNESS')
        self._print_ols_diagnostics(m)

        _s1('SUMMARY: R&D KNOWLEDGE STOCK ELASTICITY (Gamma)')
        print(f"\nDraft Ref: Section 3.1.1 - 'coefficient gamma captures output elasticity'\n")
        
        
        rd_vars = [v for v in m.x_cols if 'r' in v]
        print(f"Key Elasticity Parameter:")
        print(f"{'Variable':>10} {'Coefficient':>12} {'Std Error':>10} "
              f"{'P-Value':>10} {'Signif (5%)':>12}")
        
        for v in rd_vars:
            label = VAR_LABELS.get(v, v)
            
            coef = m.coef_.get(v, np.nan)
            se   = m.se_.get(v, np.nan)
            pval = m.pvalues_.get(v, np.nan)
            print(f"{label:>10} {coef:>12.4f} {se:>10.4f} {pval:>10.4f} {str(pval<0.05):>12}")

        print('\n' + '=' * W1)
        print('OLS PIPELINE COMPLETED SUCCESSFULLY')
        print('=' * W1)
        print('-> OLS completed.\n')

    def _run_fe(self, df, y_col, x_cols, effects='entity'):
        """Helper method for FE estimation output."""
        try:
            import statsmodels.api as sm_fe
            
            df_fe = df.set_index(['inn', 'year'])
            Y  = df_fe[y_col]
            X  = sm_fe.add_constant(df_fe[x_cols])
            
            # entity_effects always True, time_effects depends on parameter
            fe = PanelOLS(Y, X, 
                        entity_effects=True,
                        time_effects=(effects == 'both'))
            
            res = fe.fit(cov_type='clustered', cluster_entity=True)
            
            # Print summary (linearmodels object prints correctly)
            print(res)
            
            # Fixed attribute: .statistic -> .stat
            f_pool = res.f_pooled
            if f_pool is not None:
                # In linearmodels WaldTestStatistic has .stat and .pval
                print(f"\nF-test for Poolability: {f_pool.stat:.4f}") 
                print(f"P-value:                {f_pool.pval:.4f}") 
                print(f"Distribution:           {f_pool.dist_name}")
            
            effects_label = "Entity" if effects == 'entity' else "Entity + Time"
            print(f"Included effects:       {effects_label}")
            
        except Exception as e:
            # No more AttributeError crashes
            print(f"  FE estimation failed: {e}")

    def _print_ols_diagnostics(self, m):
        diag = getattr(m, 'diagnostics_', {})

        _s2('Multicollinearity Check (VIF)')
        vif_df = diag.get('VIF')
        if vif_df is not None:
            print(f"\nMulticollinearity Check (VIF):")
            print(f"{'Variable':>10} {'VIF':>10}")
            for _, row in vif_df.iterrows():
                flag = '  ← HIGH!' if row['VIF'] > 10 else ''
                print(f"{row['feature']:>10} {row['VIF']:>10.6f}{flag}")
            print("Rule of Thumb: VIF > 10 indicates severe multicollinearity.")

        _s2('Residual Statistics (Pooled OLS)')
        if hasattr(m, 'results'):
            # Get residuals (depends on whether sm.OLS or PanelOLS was used)
            resids = getattr(m.results, 'resid', getattr(m.results, 'resids', None))
            if resids is not None:
                r = pd.Series(resids.values.flatten() if hasattr(resids, 'values') else resids)
                print(f"Residuals Statistics:")
                print(f"  Mean:     {r.mean():>10.4f}")
                print(f"  Std Dev:  {r.std():>10.4f}")
                print(f"  Skewness: {float(r.skew()):>10.4f}")
                print(f"  Kurtosis: {float(r.kurtosis()):>10.4f}")

        bp = diag.get('Heteroscedasticity')
        if bp:
            print(f"\nBreusch-Pagan Test:")
            print(f"  LM Statistic: {bp.get('LM Statistic', 0):>12.2f}")
            print(f"  p-value:      {bp.get('LM-Test p-value', 0):>12.4f}")
            verdict = '⚠ Heteroscedastic' if bp.get('LM-Test p-value', 0) < 0.05 else '✓ Homoscedastic'
            print(f"  Result: {verdict}")
            print("  Note: HC3 robust standard errors applied.")

        dw = diag.get('Autocorrelation')
        if dw:
            print(f"\n  Durbin-Watson: {dw.get('DW Statistic', 0):.3f}  "
                  f"→ {dw.get('Interpretation', '')} autocorrelation")

        sw = diag.get('Normality')
        if sw:
            _s2('Normality (Shapiro-Wilk, subsample n=5,000)')
            p = sw.get('p-value', 0)
            v = 'Non-normal residuals' if p < 0.05 else 'Normal residuals'
            print(f"  W = {sw.get('W Statistic', 0):.4f},  p = {p:.4f}  → {v}")

    # ════════════════════════════════════════════════════════════════════════
    # [2/4] OP
    # ════════════════════════════════════════════════════════════════════════
    def report_op(self):
        m = self.models.get('OP')
        if m is None: return

        print('\n[2/4] Running OP Pipeline...\n')
        _s1('STEP 2: OLLEY-PAKES ESTIMATION')
        _s1('STEP 2.1: DATA PREPARATION FOR OLLEY-PAKES')

        n_total = len(self.df)
        n_op    = len(m.df)
        n_firms = m.df['inn'].nunique() if 'inn' in m.df.columns else '?'
        n_years = m.df['year'].nunique() if 'year' in m.df.columns else '?'
        n_lost  = n_total - n_op

        print(f"\nConverting financial variables to numeric...\n")
        print(f"Constructing investment variable (I_it)...")
        print(f"Formula: I_it = K_it - K_it-1 + δ * K_it-1 (δ = 10%)\n")
        print(f"Sample Selection for OP:")
        print(f"  - Initial observations:               {n_total:>12,}")
        print(f"  - After removing missing:             {n_op:>12,}")
        print(f"  - After positive investment filter:   {n_op:>12,}")
        print(f"  - Observations lost:                  {n_lost:>12,}")
        print(f"  - Sample retention rate:              {_pct(n_op, n_total):>12}")

        # data verification
        print(f"\nConstructing log-transformed variables...")
        print(f"After log transformation cleaning: {n_op:,} observations\n")
        print(f"Constructing lagged variables for second stage...\n")

        print('-' * W3)
        print('DATA VERIFICATION FOR OP')
        print('-' * W3)
        check_vars = {'l_y':'y','l_l':'l','l_k':'k','l_inv':'i'}
        for new, old in check_vars.items():
            if new in m.df.columns:
                col = m.df[new]
                nan_cnt = col.isna().sum()
                print(f"  {old}: dtype={col.dtype}, inf={np.isinf(col.fillna(0)).sum()}, nan={nan_cnt}")
        for sv in m.state_vars:
            lag = f'{sv}_lag1'
            if lag in m.df.columns:
                col = m.df[lag]
                print(f"  {lag}: dtype={col.dtype}, inf={np.isinf(col.fillna(0)).sum()}, nan={col.isna().sum()}")

        print(f"\nPanel Structure:")
        print(f"  - Observations: {n_op:,}")
        print(f"  - Firms (inn):  {n_firms:,}")
        print(f"  - Years:        {n_years}")
        print('=' * W1)

        _s1('STEP 2.2: OLLEY-PAKES ESTIMATION')

        # Stage 2 obs
        s2_cols = ['phi_lag1'] + [f'{c}_lag1' for c in m.state_vars]
        n_s2 = len(m.df.dropna(subset=[c for c in s2_cols if c in m.df.columns]))
        print(f"\nEstimation Sample:")
        print(f"  - Stage 1 observations: {n_op:>12,}")
        print(f"  - Stage 2 observations: {n_s2:>12,}")

        _s2('OP STAGE 1: Labor Coefficient Estimation')
        if hasattr(m, '_s1_results'):
            print(m._s1_results.summary())
        else:
            print()
            for v in m.free_vars:
                label = VAR_LABELS.get(v, v)
                print(f"  β({label}):  {m.coef_.get(v, np.nan):.4f}")

        print(f"\nStage 1 Results:")
        for v in m.free_vars:
            label = VAR_LABELS.get(v, v)
            g = GREEK.get(v, 'β')
            print(f"  {g} ({label} coefficient) = {m.coef_.get(v, np.nan):.4f}")

        _s2('OP STAGE 2: Capital Coefficient Estimation')
        print(f"\nOptimizing Stage 2 parameters...")
        print()
        print("Stage 2 Results:")
        for v in m.state_vars:
            label = VAR_LABELS.get(v, v)
            g = GREEK.get(v, 'β')
            print(f"  {g} ({label} coefficient) = {m.coef_.get(v, np.nan):.4f}")
        diag_op = getattr(m, 'diagnostics_', {})
        rho = diag_op.get('rho', 'N/A')
        success = diag_op.get('opt_success', 'N/A')
        msg     = diag_op.get('opt_message', 'Optimization terminated successfully.')
        print(f"  ρ (productivity persistence) = {rho}")
        print(f"  Optimization success: {success}")
        print(f"  Message: {msg}")

        _s1('OLLEY-PAKES ESTIMATION SUMMARY')
        print(f"\n{'Parameter':<22} {'OP Estimate':>14} {'Stage':>10}")
        print('-' * 48)
        for v in m.free_vars + m.state_vars:
            label = VAR_LABELS.get(v, v)
            g = GREEK.get(v, 'β')
            stage = 'Stage 1' if v in m.free_vars else 'Stage 2'
            print(f"  {g} ({label}){'':<8} {m.coef_.get(v, np.nan):>14.4f} {stage:>10}")
        if isinstance(rho, float):
            print(f"  {'ρ (Persistence)':<22} {rho:>14.4f} {'Stage 2':>10}")

        _s1('IMPORTANT: SAMPLE SELECTION WARNING')
        print(f"\nObservations retained: {n_op:,} ({_pct(n_op, n_total)} of full sample)")
        print("This is a known limitation of OP in Russian data.")
        print("LP approach will retain more observations.")
        print('=' * W1)

        _s1('STEP 2.3: OP DIAGNOSTICS')
        self._print_innovation_diag(m, 'omega_op', 'ξ')

        _s1('SUMMARY: OP COEFFICIENT ESTIMATES')
        print(f"\nOP estimation corrects for simultaneity bias in input coefficients.\n")
        print(f"\nProduction Function Coefficients")
        print(f"  {'Variable':<22} {'Coefficient':>12}")
        print('-' * 36)
        for v in m.free_vars + m.state_vars:
            label = VAR_LABELS.get(v, v)
            g = GREEK.get(v, 'β')
            print(f"  {g} ({label}){'':<8} {m.coef_.get(v, np.nan):>12.4f}")

        print('\n' + '=' * W1)
        print('OLLEY-PAKES PIPELINE COMPLETED SUCCESSFULLY')
        print('=' * W1)
        print('-> OP completed.\n')

    # ════════════════════════════════════════════════════════════════════════
    # [3/4] LP
    # ════════════════════════════════════════════════════════════════════════
    def report_lp(self):
        m = self.models.get('LP')
        if m is None: return

        print('\n[3/4] Running LP Pipeline...\n')
        _s1('STEP 3: LEVINSOHN-PETRIN ESTIMATION')
        _s1('STEP 3.1: DATA PREPARATION FOR LEVINSOHN-PETRIN')

        n_total = len(self.df)
        n_lp    = len(m.df)
        n_firms = m.df['inn'].nunique() if 'inn' in m.df.columns else '?'
        n_years = m.df['year'].nunique() if 'year' in m.df.columns else '?'
        n_op    = len(self.models['OP'].df) if 'OP' in self.models else n_total

        print(f"\nConverting financial variables to numeric...\n")
        print(f"Sample Selection for LP:")
        print(f"  - Initial observations:              {n_total:>12,}")
        print(f"  - After removing missing:            {n_lp:>12,}")
        print(f"  - After positive intermediates:      {n_lp:>12,}")
        print(f"  - Observations lost:                 {n_total - n_lp:>12,}")
        print(f"  - Sample retention rate:             {_pct(n_lp, n_total):>12}")
        print(f"\n  NOTE: LP retains significantly more observations than OP")
        print(f"        (OP requires positive investment, LP requires positive intermediates)")

        print(f"\nConstructing log-transformed variables...")
        print(f"After log transformation cleaning: {n_lp:,} observations\n")
        print(f"Constructing lagged variables for second stage...\n")

        print('-' * W3)
        print('DATA VERIFICATION FOR LP')
        print('-' * W3)
        check_vars = {'l_y':'y','l_l':'l','l_k':'k','l_m':'m'}
        for new, old in check_vars.items():
            if new in m.df.columns:
                col = m.df[new]
                print(f"  {old}: dtype={col.dtype}, inf={np.isinf(col.fillna(0)).sum()}, nan={col.isna().sum()}")
        lag_checks = [f'{sv}_lag1' for sv in m.state_vars] + [f'{m.proxy_var}_lag1']
        for lc in lag_checks:
            if lc in m.df.columns:
                col = m.df[lc]
                print(f"  {lc}: dtype={col.dtype}, inf={np.isinf(col.fillna(0)).sum()}, nan={col.isna().sum()}")

        print(f"\nPanel Structure:")
        print(f"  - Observations: {n_lp:,}")
        print(f"  - Firms (inn):  {n_firms:,}")
        print(f"  - Years:        {n_years}")
        print('=' * W1)

        _s1('STEP 3.2: LEVINSOHN-PETRIN ESTIMATION')

        s2_cols = ['phi_lag1'] + [f'{c}_lag1' for c in m.state_vars + [m.proxy_var]]
        n_s2 = len(m.df.dropna(subset=[c for c in s2_cols if c in m.df.columns]))

        print(f"\nEstimation Sample:")
        print(f"  - Stage 1 observations: {n_lp:>12,}")
        print(f"  - Stage 2 observations: {n_s2:>12,}")

        _s2('LP STAGE 1: Labor Coefficient Estimation')
        print(f"\n  Materials serve as proxy → absorbed into φ(m, k, r).")
        print(f"  Only β_l (labor) identified here; β_k, β_r identified in Stage 2.\n")

        if hasattr(m, '_s1_results'):
            print(m._s1_results.summary())
        else:
            print(f"  Stage 1 Results:")
        for v in m.free_vars:
            label = VAR_LABELS.get(v, v)
            print(f"  β_l ({label} coefficient) = {m.coef_.get(v, np.nan):.4f}")

        # beta_m identified via ACF
        if 'l_m' in m.coef_:
            print(f"\n  β_m (Materials) = {m.coef_['l_m']:.4f}")
            print(f"  Note: β_m identified via ACF (2015) — materials enters PF and serves as proxy.")

        _s2('LP STAGE 2: Capital Coefficient Estimation via Two-Step GMM')
        print(f"\nOptimizing Stage 2 parameters...")
        print(f"\nStage 2 Results:")
        for v in m.state_vars:
            label = VAR_LABELS.get(v, v)
            g = GREEK.get(v, 'β')
            print(f"  {g} ({label} coefficient) = {m.coef_.get(v, np.nan):.4f}")

        _s2('BOOTSTRAP STANDARD ERRORS')
        boot = self.comparator.models_data.get('LP', {})
        print(f"\nNumber of bootstrap iterations: {self.n_boot}")
        print(f"Accounting for generated regressor problem in first stage...")
        print(f"Running bootstrap iterations...\n")
        print(f"Bootstrap Standard Errors:")
        for v in m.free_vars + m.state_vars:
            label = VAR_LABELS.get(v, v)
            g = GREEK.get(v, 'β')
            if v in boot:
                print(f"  SE({g}) = {boot[v]['se']:.4f}")

        # Hansen J
        j_stat = m.diagnostics_.get('Hansen_J')
        _s2('HANSEN J-TEST')
        if j_stat is not None:
            j_df   = m.diagnostics_.get('Hansen_df', '?')
            j_pval = m.diagnostics_.get('Hansen_pval', np.nan)
            n_obs  = m.diagnostics_.get('Hansen_n')
            j_norm = j_stat / n_obs if n_obs else float('nan')
            verdict = ('✓ Instruments valid'
                       if isinstance(j_pval, float) and j_pval > 0.1
                       else '⚠ Suspect instruments')
            print(f"\n  J = {j_stat:.4f},  df = {j_df},  p = {j_pval:.4f}  → {verdict}")
            if n_obs:
                econ = (_hansen_econ_note(j_norm) if n_obs > 100_000 else '')
                print(f"  n = {n_obs:,},  J/n = {j_norm:.6f}  {econ}")
        else:
            print("\n  J-stat not stored.")

        _s1('LEVINSOHN-PETRIN ESTIMATION SUMMARY')
        print(f"\n{'Parameter':<22} {'LP Estimate':>14} {'Bootstrap SE':>14} {'Stage':>10}")
        print('-' * 64)
        for v in m.free_vars + m.state_vars:
            label = VAR_LABELS.get(v, v)
            g = GREEK.get(v, 'β')
            se    = boot.get(v, {}).get('se', np.nan)
            stage = 'Stage 1' if v in m.free_vars else 'Stage 2'
            se_s  = f"{se:.4f}" if not np.isnan(se) else 'N/A'
            print(f"  {g} ({label}){'':<4} {m.coef_.get(v, np.nan):>14.4f} {se_s:>14} {stage:>10}")

        print(f"\n{'='*W1}")
        print(f"\nADVANTAGE: LP vs OP Sample Retention")
        print(f"\n  LP observations retained: {n_lp:,}")
        print(f"  OP observations retained: {n_op:,}")
        if isinstance(n_op, int):
            print(f"  LP/OP ratio:              {n_lp/n_op:.1f}x more observations")
        print("  LP does not require positive investment → fewer missing data exclusions.")
        print('=' * W1)

        _s1('STEP 3.3: LP DIAGNOSTICS')
        self._print_innovation_diag(m, 'omega_lp', 'ξ')

        _s1('SUMMARY: LP COEFFICIENT ESTIMATES')
        print(f"\nLP estimation uses materials as proxy (ACF identification for β_m).\n")
        print(f"Production Function Coefficients")
        print(f"  {'Variable':<22} {'Coefficient':>12}")
        print('-' * 36)
        lp_all_vars = m.free_vars + ([m.proxy_var] if hasattr(m, 'proxy_var') and m.proxy_var in m.coef_ else []) + m.state_vars
        for v in lp_all_vars:
            label = VAR_LABELS.get(v, v)
            g = GREEK.get(v, 'β')
            print(f"  {g} ({label}){'':<8} {m.coef_.get(v, np.nan):>12.4f}")
        # RTS with materials
        rts_vars = [v for v in ['l_l', 'l_m', 'l_k'] if v in m.coef_]
        rts = sum(m.coef_[v] for v in rts_vars)
        rts_kind = 'CRS' if abs(rts-1) < 0.03 else ('IRS' if rts > 1 else 'DRS')
        print(f"\n  Returns to Scale: {rts:.4f} ({rts_kind})")

        print('\n' + '=' * W1)
        print('LEVINSOHN-PETRIN PIPELINE COMPLETED SUCCESSFULLY')
        print('=' * W1)
        print('-> LP completed.\n')

    # ════════════════════════════════════════════════════════════════════════
    # [4/4] DJ
    # ════════════════════════════════════════════════════════════════════════
    def report_dj(self):
        m = self.models.get('DJ')
        if m is None: return

        print('\n[4/4] Running Doraszelski-Jaumandreu Pipeline...\n')
        _s1('STEP 4: DORASZELSKI & JAUMANDREU (2013) ESTIMATION')
        _s1('STEP 4.1: DATA PREPARATION FOR DORASZELSKI & JAUMANDREU')

        n_total = len(self.df)
        n_dj    = len(m.df)
        n_firms = m.df['inn'].nunique() if 'inn' in m.df.columns else '?'
        n_years = m.df['year'].nunique() if 'year' in m.df.columns else '?'
        rd_col  = m.rd_var if hasattr(m, 'rd_var') else 'l_r_inv'
        n_rd    = (m.df[rd_col] > 0).sum() if rd_col in m.df.columns else '?'
        n_zero  = n_dj - n_rd if isinstance(n_rd, (int, np.integer)) else '?'
        rd_rate = _pct(n_rd, n_dj) if isinstance(n_rd, (int, np.integer)) else '?'

        print(f"\nConverting financial variables to numeric...\n")
        print(f"Sample Selection for DZ:")
        print(f"  - Initial observations:              {n_total:>12,}")
        print(f"  - After removing missing:            {n_dj:>12,}")
        print(f"  - After positive intermediates:      {n_dj:>12,}")
        print(f"  - Firms with positive R&D:           {str(n_rd):>12}")
        print(f"  - Firms with zero R&D:               {str(n_zero):>12}")
        print(f"  - R&D coverage rate:                 {str(rd_rate):>12}")

        print(f"\nConstructing log-transformed variables...")
        print(f"After log transformation cleaning: {n_dj:,} observations\n")
        print(f"Constructing lagged variables for transition function...\n")

        print('-' * W3)
        print('DATA VERIFICATION FOR DORASZELSKI')
        print('-' * W3)
        check_vars = {'l_y':'y','l_l':'l','l_k':'k','l_m':'m', rd_col:'r',
                      'omega_dj':'omega'}
        for new, old in check_vars.items():
            if new in m.df.columns:
                col = m.df[new]
                print(f"  {old}: dtype={col.dtype}, "
                      f"inf={np.isinf(col.fillna(0)).sum()}, "
                      f"nan={col.isna().sum()}")
        for sv in m.all_vars + [rd_col]:
            lag = f'{sv}_lag1'
            if lag in m.df.columns:
                col = m.df[lag]
                print(f"  {lag}: dtype={col.dtype}, inf={np.isinf(col.fillna(0)).sum()}, nan={col.isna().sum()}")

        print(f"\nPanel Structure:")
        s2_cols = ['phi_lag1'] + [f'{c}_lag1' for c in m.all_vars]
        n_s2 = len(m.df.dropna(subset=[c for c in s2_cols if c in m.df.columns]))
        print(f"  - Observations:         {n_s2:,}")
        print(f"  - Firms (inn):          {n_firms:,}")
        print(f"  - Years:                {n_years}")
        if isinstance(n_rd, (int, np.integer)):
            print(f"  - Firms with R&D > 0:   {n_rd:,}")
        print('=' * W1)

        _s1('STEP 4.2: DORASZELSKI & JAUMANDREU ESTIMATION')
        print(f"\nEstimation Sample:")
        print(f"  - Observations: {n_s2:,}")
        print(f"  - Firms:        {n_firms:,}")
        print(f"  - Years:        {n_years}")

        # GMM Step 1
        print()
        print('-' * W3)
        print('GMM ESTIMATION - STEP 1: Initial Estimation (W = I)')
        print('-' * W3)
        print(f"\nOptimizing GMM objective (initial estimation)...")
        diag = getattr(m, 'diagnostics_', {})
        s1_crit = diag.get('GMM_criterion_step1', diag.get('GMM_criterion', '?'))
        s1_succ = diag.get('GMM_success', 'N/A')
        if isinstance(s1_crit, float):
            print(f"Initial estimation success: {s1_succ}")
            print(f"Initial GMM criterion: {s1_crit:.6f}")
        else:
            print(f"Initial estimation success: {s1_succ}")

        # GMM Step 2
        print()
        print('-' * W3)
        print('GMM ESTIMATION - STEP 2: Two-Step GMM (Optimal W)')
        print('-' * W3)
        print(f"\nComputing optimal weighting matrix...")
        print(f"Optimizing GMM objective (two-step estimation)...")
        final_crit = diag.get('GMM_criterion', '?')
        if isinstance(final_crit, float):
            print(f"Two-step estimation success: {s1_succ}")
            print(f"Two-step GMM criterion: {final_crit:.6f}")

        # Production Function Coefficients
        print()
        print('=' * W1)
        print('PRODUCTION FUNCTION COEFFICIENTS (β) — ACF 2015 identification')
        print('=' * W1)
        print(f"\n  Identified jointly via GMM: β_l, β_m, β_k\n")
        print(f"  {'Parameter':<28} {'Estimate':>12}")
        print('-' * 42)
        boot = self.comparator.models_data.get('DJ', {})
        rts  = 0.0
        pf_vars = ['l_l', 'l_m', 'l_k']
        for v in pf_vars:
            label = VAR_LABELS.get(v, v)
            g     = GREEK.get(v, 'β')
            coef  = m.coef_.get(v, np.nan)
            if not np.isnan(coef): rts += coef
            print(f"  {g} ({label}){'':<12} {coef:>12.4f}")
        rts_kind = 'CRS' if abs(rts-1) < 0.03 else ('IRS' if rts > 1 else 'DRS')
        print(f"  {'Returns to Scale':<28} {rts:>12.4f}  ({rts_kind})")

        # Transition function gamma
        print()
        print('=' * W1)
        print('TRANSITION FUNCTION PARAMETER ESTIMATES')
        print('=' * W1)
        gamma = getattr(m, 'gamma_', None)
        if gamma:
            print(f"  γ_0 (constant)           = {gamma.get('g0', np.nan):.4f}")
            print(f"  γ_1 (ω persistence)      = {gamma.get('g1', np.nan):.4f}")
            print(f"  γ_2 (ω² nonlinearity)    = {gamma.get('g2', np.nan):.4f}")
            print(f"  γ_3 (R&D main effect)    = {gamma.get('g3', np.nan):.4f}")
            print(f"  γ_4 (R&D²)               = {gamma.get('g4', np.nan):.4f}")
            print(f"  γ_5 (ω × R&D interact.)  = {gamma.get('g5', np.nan):.4f}")
        else:
            print(f"\n  NOTE: γ not stored yet. Add 15-line block from file header to DJ._stage_2.")
            print(f"        Until then, γ is estimated internally at each GMM iteration.\n")
            print(f"  Economic specification (De Loecker & Jaumandreu, 2013):")
            print(f"  ω_it = γ_0 + γ_1·ω_{{t-1}} + γ_2·r_{{t-1}}")
            print(f"       + γ_3·(ω·r)_{{t-1}} + γ_4·r²_{{t-1}} + ξ_it")

        # Hansen J
        print()
        print('-' * W3)
        print("HANSEN'S J-TEST FOR OVERIDENTIFYING RESTRICTIONS")
        print('-' * W3)
        j_stat  = diag.get('Hansen_J')
        j_df    = diag.get('Hansen_df')
        j_pval  = diag.get('Hansen_pval')
        if j_stat is None:
            print(f"\nJ-test not stored.")
        elif isinstance(j_df, int) and j_df <= 0:
            print(f"\nModel is exactly identified (df = {j_df})")
            print(f"Hansen's J-test not applicable (no overidentifying restrictions)")
        else:
            n_obs_dj = diag.get('Hansen_n')
            j_norm   = j_stat / n_obs_dj if n_obs_dj else float('nan')
            verdict  = ('✓ Instruments valid'
                        if isinstance(j_pval, float) and j_pval > 0.1
                        else '⚠ Suspect instruments')
            print(f"\n  J = {j_stat:.4f},  df = {j_df},  p = {j_pval:.4f}  → {verdict}")
            if n_obs_dj:
                econ = (_hansen_econ_note(j_norm) if n_obs_dj > 100_000 else '')
                print(f"  n = {n_obs_dj:,},  J/n = {j_norm:.6f}  {econ}")

        # Bootstrap SE
        print()
        print('-' * W3)
        print('BOOTSTRAP STANDARD ERRORS')
        print('-' * W3)
        print(f"\nNumber of bootstrap iterations: {self.n_boot}")
        print(f"Running bootstrap iterations...\n")
        print(f"Bootstrap Standard Errors (n={self.n_boot} iterations):")
        for v in pf_vars:
            label = VAR_LABELS.get(v, v)
            g     = GREEK.get(v, 'β')
            if v in boot:
                print(f"  SE({g}) = {boot[v]['se']:.4f}")
        if gamma:
            # gamma_ stored in coef_, SE from bootstrap
            gamma_keys = ['g0', 'g1', 'g2', 'g3', 'g4', 'g5']
            gamma_labels = {
                'g0': 'γ_0 (Constant)',
                'g1': 'γ_1 (ω Persistence)',
                'g2': 'γ_2 (ω² Nonlinearity)',
                'g3': 'γ_3 (R&D Main Effect)',
                'g4': 'γ_4 (R&D²)',
                'g5': 'γ_5 (ω×R&D Interact.)',
            }
            for gk in gamma_keys:
                gse = boot.get(gk, {}).get('se', np.nan)
                lbl = gamma_labels[gk]
                se_s = f"{gse:.4f}" if not (isinstance(gse, float) and np.isnan(gse)) else 'N/A'
                print(f"  SE({lbl}) = {se_s}")

        # Economic interpretation
        print()
        print('=' * W1)
        print('ECONOMIC INTERPRETATION')
        print('=' * W1)
        print(f"\nDraft Section 3.1.2: 'R&D alters probability distribution of future productivity'\n")

        if gamma:
            g1 = gamma.get('g1', np.nan)
            g3 = gamma.get('g3', np.nan)  # R&D main effect
            g4 = gamma.get('g4', np.nan)  # R&D squared
            g5 = gamma.get('g5', np.nan)  # interaction ω_{t-1} × r_{t-1}
            persistence = 'HIGH' if g1 > 0.7 else ('MODERATE' if g1 > 0.4 else 'LOW')
            rd_main     = 'POSITIVE' if g3 > 0 else ('NEGATIVE' if g3 < 0 else 'ZERO')
            interac     = 'COMPLEMENTARITY' if g5 > 0 else 'SUBSTITUTABILITY'

            print(f"1. Productivity Persistence (γ_1 = {g1:.4f}):")
            print(f"   {persistence} persistence - productivity shocks are {persistence.lower()}ly persistent\n")
            print(f"2. R&D Main Effect (γ_3 = {g3:.4f}):")
            msg = ("POSITIVE effect - R&D improves productivity trajectory" if g3 > 0
                   else "NEGATIVE/ZERO effect - R&D does not improve productivity trajectory")
            print(f"   {msg}\n")
            print(f"3. Interaction Effect (γ_5 = {g5:.4f}):")
            msg3 = ("COMPLEMENTARITY - R&D more effective for leading firms"
                    if g5 > 0 else "SUBSTITUTABILITY - R&D more effective for lagging firms")
            print(f"   {msg3}\n")

            # Marginal effects table
            print('=' * W1)
            print('R&D IMPACT ON PRODUCTIVITY TRAJECTORY')
            print('=' * W1)
            # Correct formula: ME = dE[ω_t]/dr_{t-1} = g3 + 2·g4·r + g5·ω
            # where g3=R&D main, g4=R&D², g5=ω×R&D interaction
            r_col = getattr(self.models.get('DJ'), 'r_col', 'l_r_inv')
            if r_col in self.df.columns:
                innovators = self.df.loc[self.df[r_col] > 0, r_col]
                r_fixed = float(innovators.mean()) if len(innovators) > 0 else 1.0
            else:
                r_fixed = 1.0
            print(f"Marginal Effect of R&D on Expected Productivity:")
            print(f"  dE[ω_it]/dr_{{it-1}} = γ_3 + 2·γ_4·r + γ_5·ω_{{it-1}}")
            print(f"  (r fixed at mean of innovators = {r_fixed:.3f})\n")
            print('-' * W3)
            print('Marginal Effects at Different Productivity Levels')
            print('-' * W3)
            for omega_level in [0.0, 0.5, 1.0, 1.5, 2.0]:
                me = g3 + 2 * g4 * r_fixed + g5 * omega_level
                print(f"  At ω_{{it-1}} = {omega_level:.1f}: {me:.4f}")
        else:
            print("  (Store gamma_ to display economic interpretation of R&D effect.)")
            print()
            rts_msg = ('CRS — consistent with competitive markets'
                       if abs(rts-1) < 0.03 else
                       f"{'IRS' if rts > 1 else 'DRS'} — RTS = {rts:.4f}")
            print(f"  Production function elasticities: {rts_msg}")
            print(f"  R&D enters as control variable in Markov transition,")
            print(f"  not directly in production function.")

        # DJ Summary table
        print()
        print('=' * W1)
        print('DORASZELSKI & JAUMANDREU ESTIMATION SUMMARY')
        print('=' * W1)
        if gamma:
            param_rows = [
                ('γ_0 (Constant)',       gamma.get('g0', np.nan), '—'),
                ('γ_1 (ω Persistence)',  gamma.get('g1', np.nan), '—'),
                ('γ_2 (ω² Nonlinear)',   gamma.get('g2', np.nan), '—'),
                ('γ_3 (R&D Main)',       gamma.get('g3', np.nan), '—'),
                ('γ_4 (R&D²)',           gamma.get('g4', np.nan), '—'),
                ('γ_5 (ω × R&D)',       gamma.get('g5', np.nan), '—'),
            ]
        else:
            param_rows = [
                (f"β_l (Labor)",    m.coef_.get('l_l', np.nan),
                 f"{boot.get('l_l',{}).get('se','—')}"),
                (f"β_m (Materials)",m.coef_.get('l_m', np.nan),
                 f"{boot.get('l_m',{}).get('se','—')}"),
                (f"β_k (Capital)",  m.coef_.get('l_k', np.nan),
                 f"{boot.get('l_k',{}).get('se','—')}"),
            ]
        print(f"\n{'Parameter':<28} {'Estimate':>12} {'Bootstrap SE':>14}")
        print('-' * 56)
        for label, est, se in param_rows:
            se_s = f"{se:.4f}" if isinstance(se, float) else str(se)
            print(f"  {label:<26} {est:>12.4f} {se_s:>14}")

        crit = diag.get('GMM_criterion', '?')
        succ = diag.get('GMM_success', '?')
        crit_s = f"{crit:.6f}" if isinstance(crit, float) else str(crit)
        print(f"\nGMM Criterion: {crit_s}")
        print(f"Optimization Success: {succ}")
        if j_stat is not None:
            j_df_s  = str(j_df) if j_df is not None else '?'
            j_p_s   = f"{j_pval:.4f}" if isinstance(j_pval, float) else 'n/a'
            print(f"Hansen's J-test: J={j_stat:.4f}, df={j_df_s}, p={j_p_s}")

        _s1('STEP 4.3: DORASZELSKI DIAGNOSTICS')
        self._print_innovation_diag(m, 'omega_dj', 'ξ')

        # Instrument validity
        print()
        print('=' * W1)
        print('INSTRUMENT VALIDITY CHECK')
        print('=' * W1)
        if j_stat is None:
            print(f"\n  Add Hansen_J to diagnostics_ (see file header) to display.")
        elif isinstance(j_df, int) and j_df <= 0:
            print(f"\n  Instrument Validity Check (Hansen's J):")
            print(f"  Model is exactly identified - J-test not applicable")
        else:
            print(f"\n  J = {j_stat:.4f},  df = {j_df},  p = {j_pval:.4f}")

        print('\n' + '=' * W1)
        print('DORASZELSKI PIPELINE COMPLETED SUCCESSFULLY')
        print('=' * W1)
        print('-> Doraszelski-Jaumandreu completed.\n')

    # ════════════════════════════════════════════════════════════════════════
    # Final summary table
    # ════════════════════════════════════════════════════════════════════════
    def report_summary(self):
        print('\n' + '=' * 60)
        print('ALL PIPELINES EXECUTED SUCCESSFULLY')
        print('=' * 60 + '\n')

    # ════════════════════════════════════════════════════════════════════════
    # Helper methods
    # ════════════════════════════════════════════════════════════════════════

    def _print_innovation_diag(self, model, omega_col, xi_label='ξ'):
        """Innovation diagnostics using compute_xi() when available."""
        _s2(f'Productivity Innovation ({xi_label}_it) Statistics')

        # Path 1: use compute_xi() for true innovations
        if hasattr(model, 'compute_xi'):
            try:
                xi_arr = model.compute_xi()
                xi = pd.Series(xi_arr)
                source_note = f"  (via compute_xi(), n={len(xi):,})"
            except Exception as e:
                print(f"  compute_xi() failed: {e}. Falling back to raw ω.")
                xi = None
        else:
            xi = None

        # Path 2: fallback to raw omega with warning
        if xi is None:
            omega_df = getattr(model, 'omega', None)
            if omega_df is None or omega_col not in omega_df.columns:
                print("  ω estimates not available.")
                return
            xi = omega_df[omega_col].dropna()
            source_note = (
                f"  Note: showing omega level statistics, not innovation xi.\n"
                f"  Implement compute_xi() for proper E[xi] ~ 0 diagnostics."
            )

        print(f"\nInnovation ({xi_label}) Statistics:")
        print(source_note)
        print(f"  Mean:     {xi.mean():>10.4f}")
        print(f"  Std Dev:  {xi.std():>10.4f}")
        print(f"  Skewness: {float(xi.skew()):>10.4f}")
        print(f"  Kurtosis: {float(xi.kurtosis()):>10.4f}")

        sample = xi.sample(min(50000, len(xi)), random_state=42)
        t_stat, p_val = ttest_1samp(sample, 0)
        verdict = ('✗ Significantly different from 0 (p <= 0.05)'
                   if p_val <= 0.05 else '✓ Not significantly different from 0 (p > 0.05)')
        print(f"\nT-test for Innovation Mean (H0: E[ξ]=0):")
        print(f"  t-statistic: {t_stat:>10.4f}")
        print(f"  p-value:     {p_val:>10.4f}")
        print(f"  {verdict}")

    # ════════════════════════════════════════════════════════════════════════
    # Main execution
    # ════════════════════════════════════════════════════════════════════════
    def run_all(self):
        print('\n' + '=' * 60)
        print('STARTING FULL ANALYSIS PIPELINE')
        print('=' * 60 + '\n')

        self.report_ols()
        self.report_op()
        self.report_lp()
        self.report_dj()
        self.report_summary()

## 17. LaTeX Exporter

In [ ]:
class LaTeXExporter:
    """
    Unified LaTeX exporter — generates a single .tex file with ALL result tables.
    Tables are auto-generated from model objects; re-running the notebook updates them.

    Required packages in the main .tex document:
        \\usepackage{booktabs, amsmath, multirow, caption, threeparttable}
    """

    VAR_LABELS = {
        'l_l':     r'Labor ($\ln L$)',
        'l_k':     r'Capital ($\ln K$)',
        'l_m':     r'Materials ($\ln M$)',
        'l_r':     r'R\&D Stock ($\ln R$)',
        'l_r_inv': r'R\&D Flow ($\ln r$)',
    }
    RAW_LABELS = {
        'PL_revenue':              'Revenue',
        'B_fixed_assets':          'Fixed Assets',
        'PL_cost_of_sales':        'Cost of Sales',
        'B_research_development':  r'R\&D Expenditure',
        'CFo_labor':               'Labor Cost',
    }
    LOG_LABELS = {
        'l_y': r'Output ($\ln Y$)',
        'l_l': r'Labor ($\ln L$)',
        'l_k': r'Capital ($\ln K$)',
        'l_m': r'Materials ($\ln M$)',
        'l_r': r'R\&D Stock ($\ln R$)',
    }
    GAMMA_LABELS = {
        'g0': r'$\gamma_0$ (constant)',
        'g1': r'$\gamma_1$ ($\omega_{t-1}$)',
        'g2': r'$\gamma_2$ ($r_{t-1}$)',
        'g3': r'$\gamma_3$ ($\omega_{t-1}^2$)',
        'g4': r'$\gamma_4$ ($r_{t-1}^2$)',
        'g5': r'$\gamma_5$ ($\omega_{t-1} \times r_{t-1}$)',
        'g6': r'$\gamma_6$ ($\omega_{t-1}^3$)',
        'g7': r'$\gamma_7$ ($r_{t-1}^3$)',
        'g8': r'$\gamma_8$ ($\omega_{t-1}^2 \times r_{t-1}$)',
        'g9': r'$\gamma_9$ ($\omega_{t-1} \times r_{t-1}^2$)',
    }

    def __init__(self, df_model, df_clean, models_dict, comparator,
                 lp_lag_results=None, poly_results=None, industry_results=None,
                 dj_degree2_model=None):
        self.df       = df_model
        self.df_clean = df_clean
        self.models   = models_dict
        self.comp     = comparator
        self.boot     = comparator.models_data
        self.lp_lag   = lp_lag_results  # {'1_lag': r1, '2_lag': r2} or None
        self.poly_results    = poly_results    # DataFrame from PolynomialSensitivityAnalyzer
        self.industry_results = industry_results  # DataFrame from IndustryEstimator
        self.dj_degree2 = dj_degree2_model  # optional: degree-2 DJ for comparison

    # ── helpers ────────────────────────────────────────────────────────
    @staticmethod
    def _stars(p):
        if p < 0.01: return r'^{***}'
        if p < 0.05: return r'^{**}'
        if p < 0.1:  return r'^{*}'
        return ''

    @staticmethod
    def _f(v, fmt='.4f'):
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return '---'
        return f'{v:{fmt}}'

    @staticmethod
    def _esc(s):
        """Escape minimal LaTeX special chars in data strings."""
        return str(s).replace('&', r'\&').replace('%', r'\%').replace('_', r'\_')

    # ==================================================================
    # TABLE 1: Descriptive Statistics (raw financial variables)
    # ==================================================================
    def _tab_descriptive_raw(self):
        raw_cols = [c for c in self.RAW_LABELS if c in self.df_clean.columns]
        if not raw_cols:
            return ''
        desc = self.df_clean[raw_cols].describe().T
        desc['skewness'] = self.df_clean[raw_cols].skew()

        rows = []
        for col in raw_cols:
            d = desc.loc[col]
            label = self.RAW_LABELS[col]
            rows.append(
                f"  {label} & {int(d['count']):,} & {d['mean']:,.0f} & "
                f"{d['std']:,.0f} & {d['min']:,.0f} & {d['50%']:,.0f} & "
                f"{d['max']:,.0f} & {d['skewness']:.2f} \\\\"
            )

        body = '\n'.join(rows)
        return rf"""\begin{{table}}[htbp]
\centering
\caption{{Descriptive Statistics of Financial Variables (Levels, Thousands RUB)}}
\label{{tab:desc_raw}}
\begin{{tabular}}{{lrrrrrrr}}
\toprule
Variable & $N$ & Mean & Std & Min & Median & Max & Skew. \\
\midrule
{body}
\bottomrule
\end{{tabular}}
\begin{{flushleft}}
\footnotesize
\textit{{Note:}} Sample after basic cleaning (before estimation filters).
All monetary variables deflated to constant prices.
\end{{flushleft}}
\end{{table}}"""

    # ==================================================================
    # TABLE 2: Descriptive Statistics (log-transformed, estimation sample)
    # ==================================================================
    def _tab_descriptive_log(self):
        log_cols = [c for c in self.LOG_LABELS if c in self.df.columns]
        if not log_cols:
            return ''
        desc = self.df[log_cols].describe().T
        desc['skewness'] = self.df[log_cols].skew()

        rows = []
        for col in log_cols:
            d = desc.loc[col]
            label = self.LOG_LABELS[col]
            rows.append(
                f"  {label} & {int(d['count']):,} & {d['mean']:.3f} & "
                f"{d['std']:.3f} & {d['min']:.3f} & {d['50%']:.3f} & "
                f"{d['max']:.3f} & {d['skewness']:.2f} \\\\"
            )

        body = '\n'.join(rows)
        return rf"""\begin{{table}}[htbp]
\centering
\caption{{Descriptive Statistics of Estimation Variables (Logs)}}
\label{{tab:desc_log}}
\begin{{tabular}}{{lrrrrrrr}}
\toprule
Variable & $N$ & Mean & Std & Min & Median & Max & Skew. \\
\midrule
{body}
\bottomrule
\end{{tabular}}
\begin{{flushleft}}
\footnotesize
\textit{{Note:}} Estimation sample ($N = {len(self.df):,}$ firm-year observations).
Variables in natural logarithms.
\end{{flushleft}}
\end{{table}}"""

    # ==================================================================
    # TABLE 3: Main Production Function Results
    # ==================================================================
    def _tab_main_results(self):
        '''Combined production-function estimates + cross-model diagnostics.'''
        import numpy as np

        model_order = ['OLS', 'FE', 'FE+time', 'OP', 'LP', 'DJ']
        models_present = [m for m in model_order if m in self.models]
        n_mod = len(models_present)
        pf_vars = ['l_l', 'l_m', 'l_k', 'l_r']

        col_header = ' & '.join(models_present)

        # ---- Block A: input elasticities with SE rows ----
        EOL = ' \\\\'  # LaTeX row terminator: two backslashes
        elast_rows = []
        elast_rows.append(rf'\multicolumn{{{n_mod + 1}}}{{l}}{{\textit{{Input elasticities}}}}' + EOL)
        for var in pf_vars:
            label = self.VAR_LABELS.get(var, var)
            coef_cells, se_cells = [], []
            for mname in models_present:
                m = self.models[mname]
                boot = self.boot.get(mname, {})
                coef_val = m.coef_.get(var)
                if coef_val is None:
                    coef_cells.append('---')
                    se_cells.append('')
                else:
                    if var in boot:
                        se = boot[var].get('se', np.nan)
                        p  = boot[var].get('p_value', np.nan)
                    elif hasattr(m, 'se_') and var in m.se_:
                        se = m.se_[var]
                        p  = m.pvalues_.get(var, np.nan) if hasattr(m, 'pvalues_') else np.nan
                    else:
                        se, p = np.nan, np.nan
                    star = self._stars(p) if not np.isnan(p) else ''
                    coef_cells.append(f'${coef_val:.4f}{star}$')
                    se_cells.append(f'$({se:.4f})$' if not np.isnan(se) else '')
            elast_rows.append(f"  {label} & {' & '.join(coef_cells)}" + EOL)
            elast_rows.append(f"  {' & '.join([''] + se_cells)}" + EOL)

        # ---- Block B: technology and TFP dynamics ----
        omega_map = {'OLS': 'omega_ols', 'OP': 'omega_op', 'LP': 'omega_lp', 'DJ': 'omega_dj'}
        rts_cells, kind_cells = [], []
        for mname in models_present:
            m = self.models[mname]
            coefs = {k: v for k, v in m.coef_.items() if k in ['l_l', 'l_m', 'l_k']}
            rts = sum(coefs.values()) if coefs else float('nan')
            kind = 'CRS' if abs(rts - 1) < 0.03 else ('IRS' if rts > 1 else 'DRS')
            rts_cells.append(f'{rts:.3f}')
            kind_cells.append(kind)

        pers_cells = []
        for mname in models_present:
            m = self.models[mname]
            if m.omega is None:
                pers_cells.append('---'); continue
            omega_col = omega_map.get(mname)
            if omega_col is None or omega_col not in m.omega.columns:
                cols = [c for c in m.omega.columns if 'omega' in c]
                omega_col = cols[0] if cols else None
            if omega_col is None:
                pers_cells.append('---'); continue
            omega_df = m.omega[['inn', 'year', omega_col]].copy()
            omega_df['year_lag'] = omega_df['year'] + 1
            merged = omega_df.merge(
                omega_df[['inn', 'year', omega_col]].rename(columns={omega_col: 'omega_lag', 'year': 'year_lag'}),
                on=['inn', 'year_lag'], how='inner'
            ).dropna(subset=[omega_col, 'omega_lag'])
            if len(merged) < 100:
                pers_cells.append('---'); continue
            y_ar = merged[omega_col].values
            X_ar = np.column_stack([np.ones(len(merged)), merged['omega_lag'].values])
            coefs_ar, _, _, _ = np.linalg.lstsq(X_ar, y_ar, rcond=None)
            pers_cells.append(f'{coefs_ar[1]:.4f}')

        xi_cells = []
        for mname in models_present:
            m = self.models[mname]
            if mname == 'OLS':
                xi_cells.append('---'); continue
            if hasattr(m, 'compute_xi'):
                try:
                    xi = m.compute_xi()
                    xi_cells.append(f'{float(np.mean(xi)):.4f}')
                except Exception:
                    xi_cells.append('---')
            else:
                xi_cells.append('---')

        # ---- Block C: fit and GMM diagnostics ----
        n_cells = [f'{len(self.models[mn].df):,}' for mn in models_present]
        r2_cells = []
        for mname in models_present:
            m = self.models[mname]
            if hasattr(m, 'results') and hasattr(m.results, 'rsquared'):
                r2_cells.append(f'{m.results.rsquared:.4f}')
            else:
                r2_cells.append('---')

        j_cells, jn_cells = [], []
        for mname in models_present:
            m = self.models[mname]
            j_stat = m.diagnostics_.get('Hansen_J')   if hasattr(m, 'diagnostics_') else None
            j_pval = m.diagnostics_.get('Hansen_pval') if hasattr(m, 'diagnostics_') else None
            n_obs  = m.diagnostics_.get('Hansen_n')    if hasattr(m, 'diagnostics_') else None
            if j_stat is not None and j_pval is not None:
                j_cells.append(f'{j_stat:.2f} ($p$={j_pval:.3f})')
                jn_cells.append(f'{j_stat/n_obs:.6f}' if n_obs else '---')
            else:
                j_cells.append('---'); jn_cells.append('---')

        body_lines = []
        body_lines.extend(elast_rows)
        body_lines.append(r'\midrule')
        body_lines.append(rf'\multicolumn{{{n_mod + 1}}}{{l}}{{\textit{{Technology and TFP dynamics}}}}' + EOL)
        body_lines.append(rf"  Returns to Scale ($\sum \beta$) & {' & '.join(rts_cells)}" + EOL)
        body_lines.append(rf"  \quad Classification & {' & '.join(kind_cells)}" + EOL)
        body_lines.append(rf"  Persistence ($\rho$) & {' & '.join(pers_cells)}" + EOL)
        body_lines.append(rf"  $\mathbb{{E}}[\xi_{{it}}]$ & {' & '.join(xi_cells)}" + EOL)
        body_lines.append(r'\midrule')
        body_lines.append(rf'\multicolumn{{{n_mod + 1}}}{{l}}{{\textit{{Fit and GMM diagnostics}}}}' + EOL)
        body_lines.append(f"  Observations & {' & '.join(n_cells)}" + EOL)
        body_lines.append(f"  $R^2$ & {' & '.join(r2_cells)}" + EOL)
        body_lines.append(rf"  Hansen $J$ ($p$-value) & {' & '.join(j_cells)}" + EOL)
        body_lines.append(rf"  $J/n$ & {' & '.join(jn_cells)}" + EOL)

        body = '\n'.join(body_lines)
        col_spec = 'l' + 'c' * n_mod

        return (
            r'\begin{table}[htbp]' + '\n'
            + r'\centering' + '\n'
            + r'\caption{Production Function Estimates and Cross-Model Diagnostics}' + '\n'
            + r'\label{tab:main_results}' + '\n'
            + rf'\begin{{tabular}}{{{col_spec}}}' + '\n'
            + r'\toprule' + '\n'
            + f' & {col_header}' + EOL + '\n'
            + r'\midrule' + '\n'
            + body + '\n'
            + r'\bottomrule' + '\n'
            + r'\end{tabular}' + '\n'
            + r'\begin{flushleft}' + '\n'
            + r'\footnotesize' + '\n'
            + r'\textit{Note:} Clustered bootstrap standard errors in parentheses (firm-level resampling).' + '\n'
            + r'$^{*}$ $p<0.1$; $^{**}$ $p<0.05$; $^{***}$ $p<0.01$.' + '\n'
            + r'OLS uses HC3 robust standard errors.' + '\n'
            + r'FE = Entity fixed effects; FE+time = Two-way (entity + time) fixed effects.' + '\n'
            + r'OP = Olley--Pakes; LP = Levinsohn--Petrin (materials as proxy; $\beta_m$ identified via ACF; R\&D treated as a state variable so $\beta_r$ is identified jointly with $\beta_k$);' + '\n'
            + r'DJ = Doraszelski--Jaumandreu (R\&D enters the productivity transition function rather than the production function).' + '\n'
            + r'RTS = $\sum_{j \in \{l, m, k\}} \beta_j$ (R\&D excluded from RTS, following standard practice).' + '\n'
            + r'$\rho$ = AR(1) coefficient of the model-specific $\omega_{it}$.' + '\n'
            + r'$\mathbb{E}[\xi_{it}]$ = sample mean of the TFP innovation (zero by construction in moment-based estimators).' + '\n'
            + r'\end{flushleft}' + '\n'
            + r'\end{table}'
        )

        # ==================================================================
    # TABLE 4: OLS Specification Tests
    # ==================================================================
    def _tab_ols_diagnostics(self):
        ols = self.models.get('OLS')
        if ols is None or not getattr(ols, 'diagnostics_', None):
            return ''

        diag = ols.diagnostics_
        r2     = getattr(ols.results, 'rsquared', np.nan)
        r2_adj = getattr(ols.results, 'rsquared_adj', np.nan)

        rows = []
        rows.append(rf'  $R^2$               & {r2:.4f} \\')
        rows.append(rf'  Adj.\ $R^2$         & {r2_adj:.4f} \\')

        # VIF
        vif_df = diag.get('VIF')
        if vif_df is not None:
            rows.append(r'  \midrule')
            rows.append(r'  \multicolumn{2}{l}{\textit{Variance Inflation Factors}} \\')
            for _, row in vif_df.iterrows():
                label = self.VAR_LABELS.get(row['feature'], self._esc(row['feature']))
                rows.append(f"  \\quad {label} & {row['VIF']:.2f} \\\\")

        # BP
        bp = diag.get('Heteroscedasticity')
        if bp:
            rows.append(r'  \midrule')
            rows.append(rf"  Breusch--Pagan LM & {bp['LM Statistic']:.2f} \\")
            rows.append(rf"  \quad $p$-value    & {bp['LM-Test p-value']:.4f} \\")

        # DW
        dw = diag.get('Autocorrelation')
        if dw:
            rows.append(rf"  Durbin--Watson      & {dw['DW Statistic']:.3f} \\")

        # SW
        sw = diag.get('Normality')
        if sw:
            rows.append(rf"  Shapiro--Wilk $W$  & {sw['W Statistic']:.4f} \\")
            rows.append(rf"  \quad $p$-value    & {sw['p-value']:.4f} \\")

        body = '\n'.join(rows)
        return rf"""\begin{{table}}[htbp]
\centering
\caption{{OLS Specification Tests}}
\label{{tab:ols_diagnostics}}
\begin{{tabular}}{{lr}}
\toprule
Statistic & Value \\
\midrule
{body}
\bottomrule
\end{{tabular}}
\begin{{flushleft}}
\footnotesize
\textit{{Note:}} Breusch--Pagan test: $H_0$: homoscedasticity.
Durbin--Watson: values near 2 indicate no autocorrelation.
Shapiro--Wilk: subsample $n=5{{,}}000$; non-normality expected
at large $N$ but bootstrap inference remains valid.
HC3 heteroscedasticity-robust standard errors used.
\end{{flushleft}}
\end{{table}}"""

    # ==================================================================
    # TABLE 5: Cross-Model Diagnostics (RTS, Persistence, Hansen J)
    # ==================================================================
    def _tab_cross_model_diagnostics(self):
        """Merged into _tab_main_results; this stub returns empty string."""
        return ''

        # ==================================================================
    # TABLE 6: DJ Transition Function Parameters (Degree 3)
    # ==================================================================
    def _tab_dj_transition(self):
        dj = self.models.get('DJ')
        if dj is None or not getattr(dj, 'gamma_', None):
            return ''

        gamma = dj.gamma_
        boot  = self.boot.get('DJ', {})

        rows = []
        gamma_keys = ['g0', 'g1', 'g2', 'g3', 'g4', 'g5', 'g6', 'g7', 'g8', 'g9']
        for gk in gamma_keys:
            val = gamma.get(gk, np.nan)
            if np.isnan(val):
                continue
            label = self.GAMMA_LABELS.get(gk, gk)
            se = boot.get(gk, {}).get('se', np.nan)
            p  = boot.get(gk, {}).get('p_value', np.nan)
            star = self._stars(p) if not np.isnan(p) else ''
            se_s = f'({se:.4f})' if not np.isnan(se) else ''
            rows.append(f'  {label} & ${val:.4f}{star}$ \\\\')
            rows.append(f'  & $\\text{{{se_s}}}$ \\\\')

        # GMM diagnostics
        diag = dj.diagnostics_ if hasattr(dj, 'diagnostics_') else {}
        crit = diag.get('GMM_criterion', np.nan)
        j_stat = diag.get('Hansen_J', np.nan)
        j_df   = diag.get('Hansen_df', '---')
        j_pval = diag.get('Hansen_pval', np.nan)

        body = '\n'.join(rows)
        return rf"""\begin{{table}}[htbp]
\centering
\caption{{DJ Transition Function (Degree-3 Polynomial): $E[\omega_t \mid \omega_{{t-1}}, r_{{t-1}}]$}}
\label{{tab:dj_transition}}
\begin{{tabular}}{{lr}}
\toprule
Parameter & Estimate \\
\midrule
{body}
\midrule
GMM criterion & {crit:.6f} \\
Hansen $J$ & {self._f(j_stat)} \\
\quad d.f. & {j_df} \\
\quad $p$-value & {self._f(j_pval)} \\
\bottomrule
\end{{tabular}}
\begin{{flushleft}}
\footnotesize
\textit{{Note:}} Degree-3 complete polynomial in $(\omega_{{t-1}}, r_{{t-1}})$ with 10 parameters,
following \textcite{{doraszelski2013r}}.
Two-step GMM estimates with profiled $\gamma$ (ACF 2015).
Bootstrap standard errors in parentheses.
$^{{*}}$ $p<0.1$; $^{{**}}$ $p<0.05$; $^{{***}}$ $p<0.01$.
\end{{flushleft}}
\end{{table}}"""

    # ==================================================================
    # TABLE 7: DJ Marginal Effects (Degree 3)
    # ==================================================================
    def _tab_dj_marginal(self):
        dj = self.models.get('DJ')
        if dj is None or not getattr(dj, 'gamma_', None):
            return ''

        gamma = dj.gamma_
        g2 = gamma.get('g2', np.nan)   # r_{t-1}
        g4 = gamma.get('g4', np.nan)   # r_{t-1}^2
        g5 = gamma.get('g5', np.nan)   # omega * r
        g7 = gamma.get('g7', np.nan)   # r^3
        g8 = gamma.get('g8', np.nan)   # omega^2 * r
        g9 = gamma.get('g9', np.nan)   # omega * r^2
        if any(np.isnan(v) for v in [g2, g4, g5]):
            return ''
        # For degree-3 terms, treat missing as 0 (backward compat)
        if np.isnan(g7): g7 = 0.0
        if np.isnan(g8): g8 = 0.0
        if np.isnan(g9): g9 = 0.0

        # r_fixed: mean r among innovators
        r_col = getattr(dj, 'r_col', 'l_r_inv')
        if r_col in self.df.columns:
            innovators = self.df.loc[self.df[r_col] > 0, r_col]
            r_fixed = float(innovators.mean()) if len(innovators) > 0 else 1.0
        else:
            r_fixed = 1.0

        # Evaluate at actual sample quantiles of RAW omega (not arbitrary grid),
        # so ME is reported inside the support of the data.
        omega_raw_df = getattr(dj, 'omega_raw_', None)
        if omega_raw_df is not None and len(omega_raw_df) > 0:
            omega_vals = omega_raw_df['omega_dj_raw'].dropna().values
            q_pcts = [10, 25, 50, 75, 90]
            q_vals = np.percentile(omega_vals, q_pcts)
            omega_rows = list(zip([f'q{p}' for p in q_pcts], q_vals))
        else:
            omega_rows = [(f'{v:.1f}', v) for v in [0.0, 0.5, 1.0]]

        rows = []
        for label, omega in omega_rows:
            # Degree-3 marginal effect:
            # dE[omega_t]/dr = g2 + 2*g4*r + g5*omega + 3*g7*r^2 + 2*g9*omega*r + g8*omega^2
            me = (g2 + 2.0 * g4 * r_fixed + g5 * omega
                  + 3.0 * g7 * r_fixed**2 + 2.0 * g9 * omega * r_fixed
                  + g8 * omega**2)
            rows.append(f'  {label} ($\\omega = {omega:+.3f}$) & {me:+.4f} \\\\')

        body = '\n'.join(rows)
        return rf"""\begin{{table}}[htbp]
\centering
\caption{{Marginal Effect of R\&D on Expected Productivity (DJ Model, Degree 3)}}
\label{{tab:dj_marginal}}
\begin{{tabular}}{{rr}}
\toprule
$\omega_{{t-1}}$ & $\partial E[\omega_t] / \partial r_{{t-1}}$ \\
\midrule
{body}
\bottomrule
\end{{tabular}}
\begin{{flushleft}}
\footnotesize
\textit{{Note:}} Degree-3 marginal effect:
$ME(\omega, r) = \gamma_2 + 2\gamma_4 r + \gamma_5 \omega + 3\gamma_7 r^2 + 2\gamma_9 \omega r + \gamma_8 \omega^2$,
evaluated at $\bar{{r}} = {r_fixed:.3f}$ (mean log R\&D among innovators).
Cross-derivative $\partial^2 g / \partial\omega\partial r = \gamma_5 + 2\gamma_8\omega + 2\gamma_9 r$
varies across firms (unlike degree-2, where it is a constant $\gamma_5$).
\end{{flushleft}}
\end{{table}}"""

    # ==================================================================
    # TABLE 8: LP Lag Specification Comparison
    # ==================================================================
    def _tab_lp_lag_comparison(self):
        if self.lp_lag is None:
            return ''
        r1 = self.lp_lag.get('1_lag')
        r2 = self.lp_lag.get('2_lag')
        if r1 is None or r2 is None:
            return ''

        def _v(d, key, fmt='.4f'):
            v = d.get(key)
            if v is None:
                return '---'
            if isinstance(v, float):
                return f'{v:{fmt}}'
            return str(v)

        beta_l_1 = float(r1['beta_free'][0]) if r1.get('beta_free') is not None else np.nan
        beta_l_2 = float(r2['beta_free'][0]) if r2.get('beta_free') is not None else np.nan

        return rf"""\begin{{table}}[htbp]
\centering
\caption{{LP Model: 1-Lag vs 2-Lag Instrument Set Comparison}}
\label{{tab:lp_lag}}
\begin{{tabular}}{{lrr}}
\toprule
 & 1-Lag & 2-Lag \\
\midrule
Instruments & $k_{{-1}}, l_{{-1}}, k, m_{{-1}}$ & $+ k_{{-2}}, l_{{-2}}, m_{{-2}}$ \\
Moment conditions & {r1.get('n_moments', '---')} & {r2.get('n_moments', '---')} \\
Parameters & {r1.get('n_params', '---')} & {r2.get('n_params', '---')} \\
Overid.\ d.f. & {r1.get('j_df', '---')} & {r2.get('j_df', '---')} \\
Observations & {r1.get('n_obs', 0):,} & {r2.get('n_obs', 0):,} \\
\midrule
$\beta_k$ (capital) & {_v(r1, 'beta_k')} & {_v(r2, 'beta_k')} \\
$\beta_l$ (labor) & {self._f(beta_l_1)} & {self._f(beta_l_2)} \\
\midrule
GMM criterion & {_v(r1, 'gmm_crit', '.8f')} & {_v(r2, 'gmm_crit', '.8f')} \\
Hansen $J$ & {_v(r1, 'j_stat')} & {_v(r2, 'j_stat')} \\
$J$ $p$-value & {_v(r1, 'j_pval')} & {_v(r2, 'j_pval')} \\
$J/n$ & {_v(r1, 'j_norm', '.8f')} & {_v(r2, 'j_norm', '.8f')} \\
\bottomrule
\end{{tabular}}
\begin{{flushleft}}
\footnotesize
\textit{{Note:}} Two-step GMM estimates. Hansen $J$-test: $H_0$: instruments valid.
$J/n < 0.001$ indicates negligible misspecification.
1-Lag instruments: $Z = [k_{{t-1}}, l_{{t-1}}, k_t, m_{{t-1}}]$.
2-Lag adds: $[k_{{t-2}}, l_{{t-2}}, m_{{t-2}}]$.
\end{{flushleft}}
\end{{table}}"""


    # ==================================================================
    # TABLE 9: Polynomial Sensitivity Analysis
    # ==================================================================
    def _tab_poly_sensitivity(self):
        if not hasattr(self, 'poly_results') or self.poly_results is None:
            return ''

        df_res = self.poly_results
        if df_res.empty:
            return ''

        # Filter to 'All' sector for main table
        all_df = df_res[df_res['Sector'] == 'All'].copy()
        if all_df.empty:
            all_df = df_res.copy()

        methods = all_df['Method'].unique()
        degrees = sorted(all_df['Degree'].unique())

        rows = []
        for method in methods:
            m_df = all_df[all_df['Method'] == method].sort_values('Degree')
            rows.append(rf'\multicolumn{{5}}{{l}}{{\textit{{{method}}}}} \\')
            for _, row in m_df.iterrows():
                d = int(row['Degree'])
                bl = self._f(row.get('l_l'))
                bm = self._f(row.get('l_m'))
                bk = self._f(row.get('l_k'))
                rts = self._f(row.get('RTS', np.nan), '.3f')
                rows.append(f'  \\quad Degree {d} & {bl} & {bm} & {bk} & {rts} \\\\')
            rows.append(r'\midrule')

        # Remove last \midrule
        if rows and rows[-1] == r'\midrule':
            rows.pop()

        body = '\n'.join(rows)
        return rf"""\begin{{table}}[htbp]
\centering
\caption{{Sensitivity of Coefficients to Polynomial Degree in Control Function}}
\label{{tab:poly_sensitivity}}
\begin{{tabular}}{{lrrrr}}
\toprule
Specification & $\beta_l$ & $\beta_m$ & $\beta_k$ & RTS \\
\midrule
{body}
\bottomrule
\end{{tabular}}
\begin{{flushleft}}
\footnotesize
\textit{{Note:}} Polynomial degrees 2, 3, and 4 for the control function $\phi(\cdot)$.
OP uses investment as proxy; LP uses materials as proxy (ACF identification).
Coefficients estimated on the full sample.
\end{{flushleft}}
\end{{table}}"""

    # ==================================================================
    # TABLE 10: Industry-Specific Results
    # ==================================================================
    def _tab_industry_per_model(self):
        """Generate 6 separate LaTeX tables, one per estimation method."""
        if not hasattr(self, 'industry_results') or self.industry_results is None:
            return ''

        df_res = self.industry_results
        if df_res.empty:
            return ''

        method_order = ['OLS', 'FE', 'FE_TW', 'OP', 'LP', 'DJ']
        method_labels = {
            'OLS': 'Pooled OLS',
            'FE': 'Fixed Effects (Entity)',
            'FE_TW': 'Two-Way Fixed Effects (Entity + Time)',
            'OP': 'Olley-Pakes (1996)',
            'LP': 'Levinsohn-Petrin (2003) with ACF (2015)',
            'DJ': 'Doraszelski-Jaumandreu (2013)',
        }

        all_tables = []

        for method in method_order:
            m_df = df_res[df_res['Method'] == method].sort_values('Sector')

            if len(m_df) < 2:
                all_tables.append(
                    f"% Industry table for {method_labels.get(method, method)}: "
                    f"insufficient results ({len(m_df)} sector(s))"
                )
                continue

            label_safe = method.replace('+', '_').replace(' ', '_')
            caption = method_labels.get(method, method)

            rows = []
            for _, row in m_df.iterrows():
                sec = row['Sector']
                sec = sec.replace('&', r'\&')
                bl = self._f(row.get('l_l'))
                bm = self._f(row.get('l_m'))
                bk = self._f(row.get('l_k'))
                br = self._f(row.get('l_r'))
                rts = self._f(row.get('RTS', np.nan), '.3f')
                rows.append(f'  {sec} & {bl} & {bm} & {bk} & {br} & {rts} \\\\')

            body = '\n'.join(rows)

            table = (
                "\\begin{table}[htbp]\n"
                "\\centering\n"
                f"\\caption{{Industry-Specific Production Function Estimates ({caption})}}\n"
                f"\\label{{tab:industry_{label_safe}}}\n"
                "\\begin{tabular}{>{\\raggedright\\arraybackslash}p{4.5cm}rrrrr}\n"
                "\\toprule\n"
                "Industry & $\\beta_l$ & $\\beta_m$ & $\\beta_k$ & $\\beta_r$ & RTS \\\\\n"
                "\\midrule\n"
                f"{body}\n"
                "\\bottomrule\n"
                "\\end{tabular}\n"
                "\\begin{flushleft}\n"
                "\\footnotesize\n"
                f"\\textit{{Note:}} {caption} estimates by OKVED section. "
                "Sectors with fewer than 1{,}000 observations excluded. "
                "RTS = $\\beta_l + \\beta_m + \\beta_k$. "
                "``---'' indicates coefficient not estimated by this method.\n"
                "\\end{flushleft}\n"
                "\\end{table}"
            )
            all_tables.append(table)

        return '\n\n'.join(all_tables)


    # ==================================================================
    # TABLE 11: DJ Cross-Derivative Heterogeneity
    # ==================================================================
    def _tab_dj_cross_derivative(self):
        """
        Compute and tabulate the cross-derivative d^2g/dw dr for degree-3.
        Shows % of firm-observations with positive/negative cross-derivative.
        """
        dj = self.models.get('DJ')
        if dj is None or not getattr(dj, 'gamma_', None):
            return ''

        gamma = dj.gamma_
        g5 = gamma.get('g5', 0.0)
        g8 = gamma.get('g8', 0.0)
        g9 = gamma.get('g9', 0.0)

        # If no degree-3 terms, cross-derivative is constant g5
        if g8 == 0.0 and g9 == 0.0:
            return ''

        # Retrieve omega and r for all observations in the GMM sample
        if not hasattr(dj, '_s2_df_gmm'):
            return ''
        df_gmm = dj._s2_df_gmm
        b_l, b_m, b_k = dj.theta_hat

        omega = (df_gmm['phi_hat'].values
                 - b_l * df_gmm[dj.l_col].values
                 - b_m * df_gmm[dj.m_col].values
                 - b_k * df_gmm[dj.k_col].values)

        r = df_gmm[f'{dj.r_col}_lag1'].values

        # Cross-derivative: g5 + 2*g8*omega + 2*g9*r
        cross_deriv = g5 + 2.0 * g8 * omega + 2.0 * g9 * r

        n_total = len(cross_deriv)
        n_pos = int(np.sum(cross_deriv > 0))
        n_neg = int(np.sum(cross_deriv < 0))
        pct_pos = n_pos / n_total * 100
        pct_neg = n_neg / n_total * 100
        mean_cd = float(np.mean(cross_deriv))
        median_cd = float(np.median(cross_deriv))
        std_cd = float(np.std(cross_deriv))

        return rf"""\begin{{table}}[htbp]
\centering
\caption{{Cross-Derivative Heterogeneity: $\partial^2 g / \partial\omega\partial r$}}
\label{{tab:dj_cross_deriv}}
\begin{{tabular}}{{lr}}
\toprule
Statistic & Value \\
\midrule
Formula & $\gamma_5 + 2\gamma_8\omega + 2\gamma_9 r$ \\
$\gamma_5$ & {g5:.4f} \\
$\gamma_8$ & {g8:.4f} \\
$\gamma_9$ & {g9:.4f} \\
\midrule
Mean & {mean_cd:.4f} \\
Median & {median_cd:.4f} \\
Std.\ dev. & {std_cd:.4f} \\
\midrule
\% positive & {pct_pos:.1f}\% ({n_pos:,} obs.) \\
\% negative & {pct_neg:.1f}\% ({n_neg:,} obs.) \\
$N$ & {n_total:,} \\
\bottomrule
\end{{tabular}}
\begin{{flushleft}}
\footnotesize
\textit{{Note:}} The cross-derivative measures whether R\&D and current productivity
are complements (positive) or substitutes (negative) in the transition function.
Under degree-2, this quantity is a constant ($\gamma_5$); under degree-3,
it varies across firms via the cubic interaction terms $\gamma_8$ and $\gamma_9$.
\end{{flushleft}}
\end{{table}}"""

        # ==================================================================
    # TABLE 12: DJ Degree-2 vs Degree-3 Comparison
    # ==================================================================
    def _tab_dj_cd_by_quantile(self):
        """
        Cross-derivative sign distribution broken down by within-sample
        omega quantile.  This answers: does the cross-derivative actually
        change sign inside the observed support of omega, or only under
        out-of-sample extrapolation?
        """
        dj = self.models.get('DJ')
        if dj is None or not getattr(dj, 'gamma_', None):
            return ''

        g5 = dj.gamma_.get('g5', 0.0)
        g8 = dj.gamma_.get('g8', 0.0)
        g9 = dj.gamma_.get('g9', 0.0)
        if g8 == 0.0 and g9 == 0.0:
            return ''

        if not hasattr(dj, '_s2_df_gmm'):
            return ''
        df_gmm = dj._s2_df_gmm
        b_l, b_m, b_k = dj.theta_hat
        omega = (df_gmm['phi_hat'].values
                 - b_l * df_gmm[dj.l_col].values
                 - b_m * df_gmm[dj.m_col].values
                 - b_k * df_gmm[dj.k_col].values)
        r = df_gmm[f'{dj.r_col}_lag1'].values
        cd = g5 + 2.0 * g8 * omega + 2.0 * g9 * r

        # Bin by omega quantile
        import pandas as _pd
        q_edges = np.percentile(omega, [0, 10, 25, 50, 75, 90, 100])
        q_labels = ['[0-10]', '[10-25]', '[25-50]', '[50-75]', '[75-90]', '[90-100]']
        bins = _pd.cut(omega, bins=q_edges, labels=q_labels, include_lowest=True)

        df_bin = _pd.DataFrame({'bin': bins, 'omega': omega, 'cd': cd})
        summary = df_bin.groupby('bin', observed=True).agg(
            n=('cd', 'size'),
            omega_mean=('omega', 'mean'),
            cd_mean=('cd', 'mean'),
            pct_pos=('cd', lambda x: (x > 0).mean() * 100),
            pct_neg=('cd', lambda x: (x < 0).mean() * 100),
        ).reset_index()

        rows = []
        for _, r_ in summary.iterrows():
            rows.append(
                f'  {r_["bin"]} & {int(r_["n"]):,} & '
                f'{r_["omega_mean"]:+.3f} & {r_["cd_mean"]:+.4f} & '
                f'{r_["pct_pos"]:.1f}\\% & {r_["pct_neg"]:.1f}\\% \\\\'
            )
        body = '\n'.join(rows)

        return (r"""\begin{table}[htbp]
\centering
\caption{Cross-derivative sign distribution by within-sample quantile of firm productivity $\omega$.  Evaluated on the raw (non-centred) structural $\omega$ that the $\gamma$ coefficients were fitted on, so all bins lie inside the observed support of the data.}
\label{tab:dj_cd_by_quantile}
\begin{tabular}{lrrrrr}
\toprule
$\omega$ quantile bin & Obs. & Mean $\omega$ & Mean CD & \% CD$>$0 & \% CD$<$0 \\
\midrule
""" + body + r"""
\bottomrule
\end{tabular}
\end{table}
""")

    def _tab_dj_degree_comparison(self):
        """
        Side-by-side comparison of DJ estimated with degree-2 and degree-3
        transition functions.
        """
        dj3 = self.models.get('DJ')
        dj2 = self.dj_degree2
        if dj3 is None or dj2 is None:
            return ''
        if not getattr(dj3, 'gamma_', None) or not getattr(dj2, 'gamma_', None):
            return ''

        g2 = dj2.gamma_
        g3 = dj3.gamma_
        theta2 = dj2.theta_hat
        theta3 = dj3.theta_hat

        def _f2(v):
            return '---' if v is None or (isinstance(v, float) and np.isnan(v)) else f'{v:.4f}'

        rows = []
        rows.append(r'\multicolumn{3}{l}{\textit{Production function coefficients}} \\')
        rows.append(f'  $\\beta_l$ (labor)     & {theta2[0]:.4f} & {theta3[0]:.4f} \\\\')
        rows.append(f'  $\\beta_m$ (materials) & {theta2[1]:.4f} & {theta3[1]:.4f} \\\\')
        rows.append(f'  $\\beta_k$ (capital)   & {theta2[2]:.4f} & {theta3[2]:.4f} \\\\')
        rts2 = theta2[0] + theta2[1] + theta2[2]
        rts3 = theta3[0] + theta3[1] + theta3[2]
        rows.append(f'  RTS                   & {rts2:.4f} & {rts3:.4f} \\\\')
        rows.append(r'\midrule')
        rows.append(r'\multicolumn{3}{l}{\textit{Transition function parameters}} \\')
        for gk in ['g0', 'g1', 'g2', 'g3', 'g4', 'g5', 'g6', 'g7', 'g8', 'g9']:
            label = self.GAMMA_LABELS.get(gk, gk)
            v2 = g2.get(gk)
            v3 = g3.get(gk)
            rows.append(f'  {label} & {_f2(v2)} & {_f2(v3)} \\\\')

        rows.append(r'\midrule')
        rows.append(r'\multicolumn{3}{l}{\textit{Diagnostics}} \\')
        d2 = dj2.diagnostics_ if hasattr(dj2, 'diagnostics_') else {}
        d3 = dj3.diagnostics_ if hasattr(dj3, 'diagnostics_') else {}

        crit2 = d2.get('GMM_criterion', np.nan)
        crit3 = d3.get('GMM_criterion', np.nan)
        rows.append(f'  GMM criterion     & {crit2:.6f} & {crit3:.6f} \\\\')

        j2 = d2.get('Hansen_J', np.nan)
        j3 = d3.get('Hansen_J', np.nan)
        rows.append(f'  Hansen $J$        & {_f2(j2)} & {_f2(j3)} \\\\')

        n2 = d2.get('Hansen_n')
        n3 = d3.get('Hansen_n')
        jn2 = (j2 / n2) if (n2 and not np.isnan(j2)) else np.nan
        jn3 = (j3 / n3) if (n3 and not np.isnan(j3)) else np.nan
        rows.append(f'  $J/n$             & {_f2(jn2)} & {_f2(jn3)} \\\\')

        rows.append(r'\midrule')
        rows.append(r'\multicolumn{3}{l}{\textit{Cross-derivative $\partial^2 g/\partial\omega\partial r$}} \\')
        rows.append(r'  Formula            & $\gamma_5$ (constant) & $\gamma_5 + 2\gamma_8\omega + 2\gamma_9 r$ \\')

        # For degree-3, compute fraction of observations with positive cross-derivative
        if hasattr(dj3, '_s2_df_gmm') and all(k in g3 for k in ['g5', 'g8', 'g9']):
            df_gmm3 = dj3._s2_df_gmm
            b_l, b_m, b_k = dj3.theta_hat
            omega3 = (df_gmm3['phi_hat'].values
                      - b_l * df_gmm3[dj3.l_col].values
                      - b_m * df_gmm3[dj3.m_col].values
                      - b_k * df_gmm3[dj3.k_col].values)
            r3 = df_gmm3[f'{dj3.r_col}_lag1'].values
            cd3 = g3['g5'] + 2 * g3['g8'] * omega3 + 2 * g3['g9'] * r3
            pct_pos3 = np.mean(cd3 > 0) * 100
            pct_neg3 = np.mean(cd3 < 0) * 100
            pct_pos2 = 100.0 if g2.get('g5', 0) > 0 else 0.0
            pct_neg2 = 100.0 if g2.get('g5', 0) < 0 else 0.0
            rows.append(f'  \\% positive        & {pct_pos2:.1f}\\% & {pct_pos3:.1f}\\% \\\\')
            rows.append(f'  \\% negative        & {pct_neg2:.1f}\\% & {pct_neg3:.1f}\\% \\\\')

        body = '\n'.join(rows)
        return rf"""\begin{{table}}[htbp]
\centering
\caption{{DJ Transition Function: Degree-2 vs Degree-3 Comparison}}
\label{{tab:dj_degree_comparison}}
\begin{{tabular}}{{lrr}}
\toprule
Parameter & Degree 2 & Degree 3 \\
\midrule
{body}
\bottomrule
\end{{tabular}}
\begin{{flushleft}}
\footnotesize
\textit{{Note:}} Degree-2 imposes six transition parameters; degree-3 uses the
complete polynomial with ten parameters, as in \textcite{{doraszelski2013r}}.
Under degree-2, the cross-derivative $\partial^2 g/\partial\omega\partial r = \gamma_5$
is a single constant across all firms; under degree-3, it varies with
$(\omega, r)$ through the higher-order interaction terms.
\end{{flushleft}}
\end{{table}}"""

    # ==================================================================
    # TABLE 13: Marginal Effect Comparison at different omega levels
    # ==================================================================
    def _tab_dj_marginal_comparison(self):
        """
        Marginal effect of R&D at different omega levels, degree-2 vs degree-3.
        """
        dj3 = self.models.get('DJ')
        dj2 = self.dj_degree2
        if dj3 is None or dj2 is None:
            return ''
        if not getattr(dj3, 'gamma_', None) or not getattr(dj2, 'gamma_', None):
            return ''

        g2 = dj2.gamma_
        g3 = dj3.gamma_

        # Mean log R&D among innovators
        r_col = getattr(dj3, 'r_col', 'l_r_inv')
        if r_col in self.df.columns:
            innovators = self.df.loc[self.df[r_col] > 0, r_col]
            r_bar = float(innovators.mean()) if len(innovators) > 0 else 1.0
        else:
            r_bar = 1.0

        # Degree-2 marginal effect: dE/dr = g2 + 2*g4*r + g5*omega
        # Degree-3 marginal effect: + 3*g7*r^2 + 2*g9*omega*r + g8*omega^2
        # Evaluate at actual sample quantiles of raw omega (same as tab:dj_marginal)
        omega_raw_df = getattr(dj3, 'omega_raw_', None)
        if omega_raw_df is not None and len(omega_raw_df) > 0:
            omega_vals = omega_raw_df['omega_dj_raw'].dropna().values
            q_pcts = [10, 25, 50, 75, 90]
            q_vals = np.percentile(omega_vals, q_pcts)
            omega_rows = list(zip([f'q{p}' for p in q_pcts], q_vals))
        else:
            omega_rows = [(f'{v:.1f}', v) for v in [0.0, 0.5, 1.0, 1.5, 2.0]]

        rows = []
        for label, omega in omega_rows:
            me_d2 = g2.get('g2', 0) + 2 * g2.get('g4', 0) * r_bar + g2.get('g5', 0) * omega
            me_d3 = (g3.get('g2', 0) + 2 * g3.get('g4', 0) * r_bar + g3.get('g5', 0) * omega
                     + 3 * g3.get('g7', 0) * r_bar**2 + 2 * g3.get('g9', 0) * omega * r_bar
                     + g3.get('g8', 0) * omega**2)
            rows.append(f'  {label} ($\\omega = {omega:+.3f}$) & {me_d2:+.4f} & {me_d3:+.4f} \\\\')

        body = '\n'.join(rows)
        return rf"""\begin{{table}}[htbp]
\centering
\caption{{Marginal Effect of R\&D: Degree-2 vs Degree-3}}
\label{{tab:marginal_effect_comparison}}
\begin{{tabular}}{{rrr}}
\toprule
Sample quantile ($\omega_{{t-1}}$) & Degree 2 & Degree 3 \\
\midrule
{body}
\bottomrule
\end{{tabular}}
\begin{{flushleft}}
\footnotesize
\textit{{Note:}} Marginal effect $\partial E[\omega_t]/\partial r_{{t-1}}$ evaluated
at the mean log R\&D among innovators ($\bar{{r}} = {r_bar:.3f}$).
Degree-2: $\gamma_2 + 2\gamma_4 r + \gamma_5 \omega$.
Degree-3 adds: $+ 3\gamma_7 r^2 + 2\gamma_9 \omega r + \gamma_8 \omega^2$.
\end{{flushleft}}
\end{{table}}"""

    # ==================================================================
    # MASTER EXPORT
    # ==================================================================
    def export(self, filepath='results/all_tables.tex'):
        """Generate a single .tex file with all result tables."""
        sections = [
            '%' + '=' * 70,
            '% Auto-generated LaTeX tables — revision_fixed3_v4.ipynb',
            '% Required packages: booktabs, amsmath, multirow, threeparttable',
            '%' + '=' * 70,
            '',
            self._tab_descriptive_raw(),
            self._tab_descriptive_log(),
            self._tab_main_results(),
            self._tab_ols_diagnostics(),
            self._tab_cross_model_diagnostics(),
            self._tab_dj_transition(),
            self._tab_dj_marginal(),
            self._tab_dj_cross_derivative(),
            self._tab_dj_cd_by_quantile(),
            self._tab_dj_degree_comparison(),
            self._tab_dj_marginal_comparison(),
            self._tab_lp_lag_comparison(),
            self._tab_poly_sensitivity(),
            self._tab_industry_per_model(),
        ]

        output = '\n\n'.join(s for s in sections if s)

        os.makedirs(os.path.dirname(filepath) if os.path.dirname(filepath) else '.', exist_ok=True)
        with open(filepath, 'w') as f:
            f.write(output)

        logging.info(f'All LaTeX tables exported to {filepath}')
        return filepath

## 18. Main Execution Block

In [ ]:
# ==================================================================
# MAIN EXECUTION PIPELINE — Cross-sectoral analysis on 8 production
# sectors, with DJ estimated in BOTH degree-2 and degree-3 forms
# (allowing a comparison table and two marginal-effect plots).
# ==================================================================

# Step 1: Data preparation (logs, lags, clean panel)
preparer       = DataPreparer(df_clean)
df_model_ready = preparer.prepare()

# Step 2: Estimate main pooled models
ols    = OLSModel(df_model_ready, 'l_y', ['l_l', 'l_m', 'l_k', 'l_r']).fit(run_diagnostics=True)
ols_fe = OLSModel(df_model_ready, 'l_y', ['l_l', 'l_m', 'l_k', 'l_r'], fe='entity').fit(run_diagnostics=False)
ols_fe_time = OLSModel(df_model_ready, 'l_y', ['l_l', 'l_m', 'l_k', 'l_r'], fe='both').fit(run_diagnostics=False)
op  = OlleyPakesModel(df_model_ready, 'l_y', ['l_l', 'l_m'], ['l_k', 'l_r'], 'l_inv').fit()
lp  = LevinsohnPetrinModel(df_model_ready, 'l_y', ['l_l'], ['l_k', 'l_r'], 'l_m').fit()

# Step 2b: Estimate DJ in BOTH degrees (for comparison)
dj_d2 = Doraszelski_Jaumandreu(df_model_ready, 'l_y', ['l_l', 'l_m'], ['l_k'], 'l_r_inv',
                               poly_degree=2).fit()
dj    = Doraszelski_Jaumandreu(df_model_ready, 'l_y', ['l_l', 'l_m'], ['l_k'], 'l_r_inv',
                               poly_degree=3).fit()

# Register both degrees in the model dict so the Visualizer can plot each
models_dict = {'OLS': ols, 'FE': ols_fe, 'FE+time': ols_fe_time,
               'OP': op, 'LP': lp, 'DJ': dj, 'DJ_d2': dj_d2}

# Step 3: Bootstrap standard errors (main pipelines + DJ degree-3;
#         degree-2 uses point estimates only for comparison table)
N_BOOT     = 200
comparator = ModelComparator(n_bootstrap=N_BOOT)

# Keep a handle to each bootstrapper so the per-replication coefficient
# draws (and, for DJ, the gamma draws) remain available for downstream
# quantile-CI construction on nonlinear transformations.
bs_ols = PanelBootstrapper(
    df_model_ready, OLSModel,
    {'y_col': 'l_y', 'x_cols': ['l_l', 'l_m', 'l_k', 'l_r']}, N_BOOT)
comparator.add_model_results('OLS', bs_ols.run(), bootstrapper=bs_ols)

bs_fe = PanelBootstrapper(
    df_model_ready, OLSModel,
    {'y_col': 'l_y', 'x_cols': ['l_l', 'l_m', 'l_k', 'l_r'], 'fe': 'entity'}, N_BOOT)
comparator.add_model_results('FE', bs_fe.run(), bootstrapper=bs_fe)

bs_fet = PanelBootstrapper(
    df_model_ready, OLSModel,
    {'y_col': 'l_y', 'x_cols': ['l_l', 'l_m', 'l_k', 'l_r'], 'fe': 'both'}, N_BOOT)
comparator.add_model_results('FE+time', bs_fet.run(), bootstrapper=bs_fet)

bs_op = PanelBootstrapper(
    df_model_ready, OlleyPakesModel,
    {'y_col': 'l_y', 'free_vars': ['l_l','l_m'],
     'state_vars': ['l_k','l_r'], 'proxy_var': 'l_inv'}, N_BOOT)
comparator.add_model_results('OP', bs_op.run(), bootstrapper=bs_op)

bs_lp = PanelBootstrapper(
    df_model_ready, LevinsohnPetrinModel,
    {'y_col': 'l_y', 'free_cols': ['l_l'],
     'state_cols': ['l_k', 'l_r'], 'proxy_col': 'l_m'}, N_BOOT)
comparator.add_model_results('LP', bs_lp.run(), bootstrapper=bs_lp)

bs_dj = PanelBootstrapper(
    df_model_ready, Doraszelski_Jaumandreu,
    {'y_col': 'l_y', 'free_cols': ['l_l','l_m'],
     'state_cols': ['l_k'], 'r_col': 'l_r_inv', 'poly_degree': 3}, N_BOOT)
comparator.add_model_results('DJ', bs_dj.run(), bootstrapper=bs_dj)

# Bootstrap for DJ degree-2 — used only to put a CI band on the degree-2
# marginal-effect plot; not reported in the main comparison table.
bs_dj_d2 = PanelBootstrapper(
    df_model_ready, Doraszelski_Jaumandreu,
    {'y_col': 'l_y', 'free_cols': ['l_l','l_m'],
     'state_cols': ['l_k'], 'r_col': 'l_r_inv', 'poly_degree': 2}, N_BOOT)
dj_d2_boot = bs_dj_d2.run()
dj_d2_boot['_gamma_draws'] = bs_dj_d2.gamma_results_

# Step 4: LP instrument set comparison (1-lag vs 2-lag)
lp_lag_results = compare_lp_lag_specifications(df_model_ready, 'l_y', ['l_l'], ['l_k', 'l_r'], 'l_m', [])

# Step 5: Polynomial Sensitivity Analysis (OP / LP)
poly_analyzer = PolynomialSensitivityAnalyzer(df_model_ready, sectors=['All'])
poly_analyzer.run()
poly_analyzer.print_summary()
poly_results_df = poly_analyzer.get_summary_df()

# Step 6: Detailed pipeline output (MAIN pooled pipelines, before industry)
os.makedirs('results', exist_ok=True)
reporter = PipelineReporter(df_model_ready, models_dict, comparator,
                            n_boot=N_BOOT, output_dir='results')
reporter.run_all()

# Step 7: Industry-Specific Estimation on 8 production sectors
#         (AFTER main pipelines, BEFORE diagnostics — cross-sectoral block)
industry_estimator = IndustryEstimator(df_model_ready)  # filter_production=True by default
industry_estimator.run()
industry_estimator.print_all_summaries()
industry_results_df = industry_estimator.get_coef_comparison_df()

# Step 8: Diagnostics (AFTER cross-sectoral, per thesis logic)
diag_reporter = DiagnosticsReporter(models_dict, df_model_ready)
diag_reporter.report(export_latex=False)

# Step 9: Unified LaTeX export (all tables in one file)
exporter = LaTeXExporter(df_model_ready, df_clean, models_dict, comparator,
                         lp_lag_results=lp_lag_results,
                         poly_results=poly_results_df,
                         industry_results=industry_results_df,
                         dj_degree2_model=dj_d2)
exporter.export('results/all_tables.tex')
comparator.export_to_csv('results/production_functions_results.csv')

# Step 10: Plots
viz = Visualizer(df_model_ready, models_dict, output_dir='results/plots')

# Attach per-replication gamma draws to the DJ boot_stats dict so the
# marginal-effect plotter can build a proper bootstrap-quantile CI.
_dj_stats = comparator.models_data.get('DJ')
if _dj_stats is not None:
    _dj_stats['_gamma_draws'] = comparator.gamma_draws.get('DJ', [])
viz.generate_all(dj_boot_stats=_dj_stats)

# --- Two marginal-effect plots: degree-2 and degree-3 --------------------
# generate_all() already produced degree-3 via model_key="DJ" (default).
# Explicitly emit the degree-2 version too, so the thesis can compare.
viz.plot_dj_marginal_effect(boot_stats=dj_d2_boot, model_key='DJ_d2')
# Re-emit degree-3 explicitly for symmetry / clarity in the logs.
viz.plot_dj_marginal_effect(boot_stats=comparator.models_data.get('DJ'), model_key='DJ')

plot_poly_sensitivity(poly_analyzer, output_dir='results/plots')

# Industry-specific coefficient plots
for _method in IndustryEstimator.ALL_METHODS:
    plot_industry_coefficients(industry_estimator, method=_method, output_dir='results/plots')

# TFP distribution plots
for _method in ['OLS', 'OP', 'LP', 'DJ']:
    plot_industry_tfp_distribution(industry_estimator, method=_method, output_dir='results/plots')

plot_industry_rts(industry_estimator, output_dir='results/plots')

# Degree-3 marginal effect surface and cross-derivative heatmap
plot_dj_marginal_surface(dj, output_dir='results/plots')

logging.info("Full pipeline completed. Tables: results/all_tables.tex. Plots: results/plots/.")